# Notebook 4 — Robust Cross-Dataset SHAP and Clinician Comparison

This notebook performs post-model-development interpretation of the frozen MIMIC-IV internal temporal-test and MC-MED external-validation results.

The primary analysis representation is the shared set of 2,495 grouped clinically interpretable raw features. The 2,498-column one-hot/prepared XGBoost representation is retained only for technical provenance and is not used for cross-dataset feature comparison or clinician comparison.

The notebook evaluates patient-level SHAP magnitude, distance from zero, direction, support and stability; compares HOME and ADMITTED encounters; assesses SHAP transportability between MIMIC-IV and MC-MED; and subsequently compares robust model evidence with clinician-defined feature importance.

All model development is complete. No model fitting, preprocessing fitting, recalibration, threshold modification or predictive feature reselection is permitted in this notebook.

In [ ]:



from collections import defaultdict
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path

import gc
import hashlib
import json
import os
import platform
import subprocess
import sys
import time
import warnings
import re



import numpy as np
import pandas as pd

from scipy import stats



import matplotlib
import matplotlib.pyplot as plt

import plotly
import plotly.graph_objects as go



import shap



from IPython.display import display

from tqdm.auto import tqdm



RANDOM_SEED = 42


np.random.seed(
    RANDOM_SEED
)


pd.set_option(
    "display.max_columns",
    200,
)


pd.set_option(
    "display.max_rows",
    200,
)


pd.set_option(
    "display.width",
    180,
)


warnings.filterwarnings(
    "default"
)


print(
    "Notebook 4 centralized imports loaded."
)


print(
    "Python:",
    platform.python_version(),
)


print(
    "NumPy:",
    np.__version__,
)


print(
    "pandas:",
    pd.__version__,
)


print(
    "SciPy:",
    stats.__version__
    if hasattr(
        stats,
        "__version__",
    )
    else
    "loaded",
)


print(
    "Matplotlib:",
    matplotlib.__version__,
)


print(
    "Plotly:",
    plotly.__version__,
)


print(
    "SHAP:",
    shap.__version__,
)


print(
    "Random seed:",
    RANDOM_SEED,
)

In [ ]:



PROJECT_ROOT = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PROJECT_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()


EXPECTED_PYTHON = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PYTHON",
        sys.executable,
    )
).expanduser().resolve()


EXPECTED_TMUX_SESSION = (
    "Robust_SHAP"
)



CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()


CURRENT_PYTHON = Path(
    sys.executable
).resolve()


inside_tmux = bool(
    os.environ.get(
        "TMUX"
    )
)


current_tmux_session = None


if inside_tmux:

    try:

        current_tmux_session = (
            subprocess.check_output(
                [
                    "tmux",
                    "display-message",
                    "-p",
                    "#S",
                ],
                text=True,
            )
            .strip()
        )

    except Exception:

        current_tmux_session = (
            "UNKNOWN"
        )



assert PROJECT_ROOT.exists(), (
    "Project root does not exist:\n"
    f"{PROJECT_ROOT}"
)


assert CURRENT_WORKING_DIRECTORY == PROJECT_ROOT, (
    "Notebook 4 should run from the project root.\n\n"
    f"Expected:\n{PROJECT_ROOT}\n\n"
    f"Current:\n{CURRENT_WORKING_DIRECTORY}"
)



assert CURRENT_PYTHON == EXPECTED_PYTHON, (
    "Notebook 4 is using the wrong Python interpreter.\n\n"
    f"Expected:\n{EXPECTED_PYTHON}\n\n"
    f"Current:\n{CURRENT_PYTHON}"
)



tmux_session_matches_expected = (
    inside_tmux
    and
    current_tmux_session
    ==
    EXPECTED_TMUX_SESSION
)


notebook4_runtime_audit = pd.DataFrame(
    [
        {
            "project_root": str(
                PROJECT_ROOT
            ),

            "working_directory": str(
                CURRENT_WORKING_DIRECTORY
            ),

            "working_directory_correct": (
                CURRENT_WORKING_DIRECTORY
                ==
                PROJECT_ROOT
            ),

            "python_executable": str(
                CURRENT_PYTHON
            ),

            "expected_python": str(
                EXPECTED_PYTHON
            ),

            "python_environment_correct": (
                CURRENT_PYTHON
                ==
                EXPECTED_PYTHON
            ),

            "inside_tmux": (
                inside_tmux
            ),

            "current_tmux_session": (
                current_tmux_session
            ),

            "recommended_tmux_session": (
                EXPECTED_TMUX_SESSION
            ),

            "recommended_tmux_session_active": (
                tmux_session_matches_expected
            ),

            "python_version": (
                platform.python_version()
            ),

            "platform": (
                platform.platform()
            ),
        }
    ]
)


display(
    notebook4_runtime_audit
)


print(
    "\nNOTEBOOK 4 RUNTIME CHECK"
)


print(
    "Project root:",
    PROJECT_ROOT,
)


print(
    "Working directory correct:",
    "YES",
)


print(
    "Kernel Python:",
    CURRENT_PYTHON,
)


print(
    "Expected Python:",
    EXPECTED_PYTHON,
)


print(
    "Python environment correct:",
    "YES",
)


print(
    "Inside tmux:",
    (
        "YES"
        if inside_tmux
        else
        "NO"
    ),
)


print(
    "Current tmux session:",
    current_tmux_session,
)


if inside_tmux:

    if tmux_session_matches_expected:

        print(
            "Recommended Notebook 4 tmux session:",
            "YES",
        )

    else:

        print(
            "Recommended Notebook 4 tmux session:",
            "NO — current session is still usable",
        )


print(
    "\nModel loaded:",
    "NO",
)


print(
    "SHAP bundles loaded:",
    "NO",
)


print(
    "Robust filtering performed:",
    "NO",
)

## Frozen hand-off loading and checksum verification

In [ ]:



NOTEBOOK3_HANDOFF_DIR = (
    PROJECT_ROOT
    / "results"
    / "handoff"
    / "notebook3"
)


NOTEBOOK4_HANDOFF_JSON = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_to_notebook4_handoff.json"
)


NOTEBOOK3_HANDOFF_CHECKSUM_MANIFEST = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_handoff_checksum_manifest.csv"
)


NOTEBOOK3_FINAL_ARTIFACT_MANIFEST = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_final_artifact_manifest.csv"
)


NOTEBOOK3_NOTEBOOK2_REFERENCE_MANIFEST = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_notebook2_reference_manifest.csv"
)


NOTEBOOK3_SHAP_CROSS_DATASET_AUDIT = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_shap_cross_dataset_handoff_audit.csv"
)


NOTEBOOK3_COMPLETION_AUDIT = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_completion_audit.csv"
)


required_primary_handoff_files = [
    NOTEBOOK4_HANDOFF_JSON,
    NOTEBOOK3_HANDOFF_CHECKSUM_MANIFEST,
    NOTEBOOK3_FINAL_ARTIFACT_MANIFEST,
    NOTEBOOK3_NOTEBOOK2_REFERENCE_MANIFEST,
    NOTEBOOK3_SHAP_CROSS_DATASET_AUDIT,
    NOTEBOOK3_COMPLETION_AUDIT,
]


for required_path in (
    required_primary_handoff_files
):

    assert required_path.exists(), (
        "Required Notebook 3 hand-off file is missing:\n"
        f"{required_path}"
    )



def notebook4_file_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()



notebook3_handoff_checksum_manifest = (
    pd.read_csv(
        NOTEBOOK3_HANDOFF_CHECKSUM_MANIFEST
    )
)


required_checksum_columns = {
    "artifact",
    "path",
    "sha256",
}


assert required_checksum_columns.issubset(
    notebook3_handoff_checksum_manifest.columns
)


notebook3_handoff_checksum_manifest[
    "current_sha256"
] = (
    notebook3_handoff_checksum_manifest[
        "path"
    ]
    .map(
        lambda path:
        notebook4_file_sha256(
            Path(
                path
            )
        )
    )
)


notebook3_handoff_checksum_manifest[
    "checksum_match"
] = (
    notebook3_handoff_checksum_manifest[
        "sha256"
    ]
    ==
    notebook3_handoff_checksum_manifest[
        "current_sha256"
    ]
)


assert notebook3_handoff_checksum_manifest[
    "checksum_match"
].all(), (
    "One or more Notebook 3 hand-off files failed "
    "SHA-256 verification."
)



with NOTEBOOK4_HANDOFF_JSON.open(
    "r",
    encoding="utf-8",
) as file:

    notebook4_handoff = json.load(
        file
    )


assert notebook4_handoff[
    "analysis_boundaries"
][
    "model_development_complete"
] is True


assert notebook4_handoff[
    "analysis_boundaries"
][
    "model_refitting_allowed"
] is False


assert notebook4_handoff[
    "analysis_boundaries"
][
    "recalibration_allowed"
] is False


assert notebook4_handoff[
    "analysis_boundaries"
][
    "threshold_reselection_allowed"
] is False


LOCKED_GROUPED_FEATURE_COUNT = int(
    notebook4_handoff[
        "feature_universe"
    ][
        "clinically_interpretable_features"
    ]
)


TECHNICAL_PREPARED_FEATURE_COUNT = int(
    notebook4_handoff[
        "feature_universe"
    ][
        "technical_prepared_model_columns"
    ]
)


LOCKED_FEATURE_ORDER_SHA256 = (
    notebook4_handoff[
        "feature_universe"
    ][
        "feature_order_sha256"
    ]
)


LOCKED_DECISION_THRESHOLD = float(
    notebook4_handoff[
        "model"
    ][
        "locked_threshold"
    ]
)


assert LOCKED_GROUPED_FEATURE_COUNT == 2495


assert TECHNICAL_PREPARED_FEATURE_COUNT == 2498


assert LOCKED_FEATURE_ORDER_SHA256 == (
    "9a4a106d770e2ce6edc5527a7b0f1b58"
    "e18e6db126a5f44a028d73eb54ae0587"
)


assert np.isclose(
    LOCKED_DECISION_THRESHOLD,
    0.282,
    rtol=0.0,
    atol=1e-12,
)



MIMIC_GROUPED_SHAP_BUNDLE_PATH = Path(
    notebook4_handoff[
        "mimic_internal_test"
    ][
        "grouped_shap_bundle"
    ]
)


MCMED_GROUPED_SHAP_BUNDLE_PATH = Path(
    notebook4_handoff[
        "mcmed_external"
    ][
        "grouped_shap_bundle"
    ]
)


CLINICIAN_FEATURE_UNIVERSE_PATH = Path(
    notebook4_handoff[
        "clinician_comparison"
    ][
        "feature_universe"
    ]
)


MIMIC_PREDICTIONS_PATH = Path(
    notebook4_handoff[
        "mimic_internal_test"
    ][
        "predictions"
    ]
)


MCMED_PREDICTIONS_PATH = Path(
    notebook4_handoff[
        "mcmed_external"
    ][
        "predictions"
    ]
)


MIMIC_METRICS_PATH = Path(
    notebook4_handoff[
        "mimic_internal_test"
    ][
        "metrics_with_ci"
    ]
)


MCMED_METRICS_PATH = Path(
    notebook4_handoff[
        "mcmed_external"
    ][
        "metrics_with_ci"
    ]
)


CROSS_DATASET_PERFORMANCE_PATH = Path(
    notebook4_handoff[
        "cross_dataset"
    ][
        "performance_comparison"
    ]
)


MCMED_CALIBRATION_DIAGNOSTICS_PATH = Path(
    notebook4_handoff[
        "mcmed_external"
    ][
        "calibration_diagnostics"
    ]
)


required_referenced_paths = [
    MIMIC_GROUPED_SHAP_BUNDLE_PATH,
    MCMED_GROUPED_SHAP_BUNDLE_PATH,
    CLINICIAN_FEATURE_UNIVERSE_PATH,
    MIMIC_PREDICTIONS_PATH,
    MCMED_PREDICTIONS_PATH,
    MIMIC_METRICS_PATH,
    MCMED_METRICS_PATH,
    CROSS_DATASET_PERFORMANCE_PATH,
    MCMED_CALIBRATION_DIAGNOSTICS_PATH,
]


for referenced_path in (
    required_referenced_paths
):

    assert referenced_path.exists(), (
        "A file referenced by the frozen hand-off is missing:\n"
        f"{referenced_path}"
    )



notebook2_reference_manifest = pd.read_csv(
    NOTEBOOK3_NOTEBOOK2_REFERENCE_MANIFEST
)


assert {
    "artifact",
    "canonical_path",
    "sha256",
}.issubset(
    notebook2_reference_manifest.columns
)


def expected_notebook2_sha256(
    artifact_name,
):

    matching_rows = (
        notebook2_reference_manifest.loc[
            notebook2_reference_manifest[
                "artifact"
            ]
            ==
            artifact_name
        ]
    )


    assert len(
        matching_rows
    ) == 1, (
        "Expected exactly one Notebook 2 reference for "
        f"{artifact_name}."
    )


    return str(
        matching_rows[
            "sha256"
        ]
        .iloc[
            0
        ]
    )



expected_mimic_shap_sha256 = (
    expected_notebook2_sha256(
        "mimic_grouped_shap_bundle"
    )
)


current_mimic_shap_sha256 = (
    notebook4_file_sha256(
        MIMIC_GROUPED_SHAP_BUNDLE_PATH
    )
)


assert (
    current_mimic_shap_sha256
    ==
    expected_mimic_shap_sha256
), (
    "MIMIC grouped SHAP bundle checksum mismatch."
)



expected_clinician_universe_sha256 = (
    expected_notebook2_sha256(
        "clinician_comparison_feature_universe"
    )
)


current_clinician_universe_sha256 = (
    notebook4_file_sha256(
        CLINICIAN_FEATURE_UNIVERSE_PATH
    )
)


assert (
    current_clinician_universe_sha256
    ==
    expected_clinician_universe_sha256
), (
    "Clinician comparison feature-universe checksum mismatch."
)



notebook3_final_artifact_manifest = pd.read_csv(
    NOTEBOOK3_FINAL_ARTIFACT_MANIFEST
)


assert {
    "relative_path",
    "sha256",
}.issubset(
    notebook3_final_artifact_manifest.columns
)


mcmed_shap_relative_path = str(
    MCMED_GROUPED_SHAP_BUNDLE_PATH.relative_to(
        PROJECT_ROOT
    )
)


matching_mcmed_manifest_rows = (
    notebook3_final_artifact_manifest.loc[
        notebook3_final_artifact_manifest[
            "relative_path"
        ]
        ==
        mcmed_shap_relative_path
    ]
)


assert len(
    matching_mcmed_manifest_rows
) == 1, (
    "MC-MED grouped SHAP bundle was not found uniquely "
    "in the Notebook 3 artifact manifest."
)


expected_mcmed_shap_sha256 = str(
    matching_mcmed_manifest_rows[
        "sha256"
    ]
    .iloc[
        0
    ]
)


current_mcmed_shap_sha256 = (
    notebook4_file_sha256(
        MCMED_GROUPED_SHAP_BUNDLE_PATH
    )
)


assert (
    current_mcmed_shap_sha256
    ==
    expected_mcmed_shap_sha256
), (
    "MC-MED grouped SHAP bundle checksum mismatch."
)



mimic_bundle_npz = np.load(
    MIMIC_GROUPED_SHAP_BUNDLE_PATH,
    allow_pickle=False,
)


mcmed_bundle_npz = np.load(
    MCMED_GROUPED_SHAP_BUNDLE_PATH,
    allow_pickle=False,
)


print(
    "MIMIC SHAP bundle keys:"
)


print(
    mimic_bundle_npz.files
)


print(
    "\nMC-MED SHAP bundle keys:"
)


print(
    mcmed_bundle_npz.files
)


required_bundle_keys = {
    "shap_values",
    "feature_values",
    "raw_feature_names",
    "outcome_binary",
}


assert required_bundle_keys.issubset(
    set(
        mimic_bundle_npz.files
    )
)


assert required_bundle_keys.issubset(
    set(
        mcmed_bundle_npz.files
    )
)



mimic_shap_values = np.asarray(
    mimic_bundle_npz[
        "shap_values"
    ],
    dtype=np.float32,
)


mimic_feature_values = np.asarray(
    mimic_bundle_npz[
        "feature_values"
    ],
    dtype=np.float32,
)


mimic_outcome = np.asarray(
    mimic_bundle_npz[
        "outcome_binary"
    ],
    dtype=np.int8,
)


mimic_feature_names = (
    mimic_bundle_npz[
        "raw_feature_names"
    ]
    .astype(str)
    .tolist()
)


mcmed_shap_values = np.asarray(
    mcmed_bundle_npz[
        "shap_values"
    ],
    dtype=np.float32,
)


mcmed_feature_values = np.asarray(
    mcmed_bundle_npz[
        "feature_values"
    ],
    dtype=np.float32,
)


mcmed_outcome = np.asarray(
    mcmed_bundle_npz[
        "outcome_binary"
    ],
    dtype=np.int8,
)


mcmed_feature_names = (
    mcmed_bundle_npz[
        "raw_feature_names"
    ]
    .astype(str)
    .tolist()
)


mimic_shap_bias = (
    np.asarray(
        mimic_bundle_npz[
            "bias"
        ],
        dtype=np.float32,
    )
    if "bias"
    in mimic_bundle_npz.files
    else None
)


mcmed_shap_bias = (
    np.asarray(
        mcmed_bundle_npz[
            "bias"
        ],
        dtype=np.float32,
    )
    if "bias"
    in mcmed_bundle_npz.files
    else None
)


mimic_bundle_npz.close()

mcmed_bundle_npz.close()



assert mimic_shap_values.shape == (
    1851,
    2495,
)


assert mimic_feature_values.shape == (
    1851,
    2495,
)


assert mimic_outcome.shape == (
    1851,
)


assert mcmed_shap_values.shape == (
    6707,
    2495,
)


assert mcmed_feature_values.shape == (
    6707,
    2495,
)


assert mcmed_outcome.shape == (
    6707,
)



assert len(
    mimic_feature_names
) == 2495


assert len(
    mcmed_feature_names
) == 2495


assert mimic_feature_names == (
    mcmed_feature_names
), (
    "MIMIC and MC-MED feature order differs."
)


LOCKED_RAW_FEATURE_ORDER = (
    mimic_feature_names
)


assert len(
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
) == 2495


calculated_feature_order_sha256 = hashlib.sha256(
    "\n".join(
        LOCKED_RAW_FEATURE_ORDER
    )
    .encode(
        "utf-8"
    )
).hexdigest()


assert calculated_feature_order_sha256 == (
    LOCKED_FEATURE_ORDER_SHA256
)



assert set(
    np.unique(
        mimic_outcome
    )
) == {
    0,
    1,
}


assert set(
    np.unique(
        mcmed_outcome
    )
) == {
    0,
    1,
}


assert int(
    (
        mimic_outcome
        ==
        0
    ).sum()
) == 1606


assert int(
    (
        mimic_outcome
        ==
        1
    ).sum()
) == 245


assert int(
    (
        mcmed_outcome
        ==
        0
    ).sum()
) == 4872


assert int(
    (
        mcmed_outcome
        ==
        1
    ).sum()
) == 1835



assert np.isfinite(
    mimic_shap_values
).all()


assert np.isfinite(
    mcmed_shap_values
).all()


assert not np.isinf(
    mimic_feature_values
).any()


assert not np.isinf(
    mcmed_feature_values
).any()



clinician_feature_universe = pd.read_csv(
    CLINICIAN_FEATURE_UNIVERSE_PATH
)


assert len(
    clinician_feature_universe
) == 2495


assert (
    clinician_feature_universe[
        "source_raw_feature"
    ]
    .astype(str)
    .tolist()
    ==
    LOCKED_RAW_FEATURE_ORDER
)


assert clinician_feature_universe[
    "source_raw_feature"
].is_unique



mimic_predictions = pd.read_parquet(
    MIMIC_PREDICTIONS_PATH
)


mcmed_predictions = pd.read_parquet(
    MCMED_PREDICTIONS_PATH
)


mimic_metrics_with_ci = pd.read_csv(
    MIMIC_METRICS_PATH
)


mcmed_metrics_with_ci = pd.read_csv(
    MCMED_METRICS_PATH
)


internal_external_performance = pd.read_csv(
    CROSS_DATASET_PERFORMANCE_PATH
)


mcmed_calibration_diagnostics = pd.read_csv(
    MCMED_CALIBRATION_DIAGNOSTICS_PATH
)


assert len(
    mimic_predictions
) == 1851


assert len(
    mcmed_predictions
) == 6707



notebook3_shap_cross_dataset_audit = pd.read_csv(
    NOTEBOOK3_SHAP_CROSS_DATASET_AUDIT
)


notebook3_completion_audit = pd.read_csv(
    NOTEBOOK3_COMPLETION_AUDIT
)


assert bool(
    notebook3_completion_audit[
        "notebook3_complete"
    ]
    .iloc[
        0
    ]
) is True



MIMIC_HOME_MASK = (
    mimic_outcome
    ==
    0
)


MIMIC_ADMITTED_MASK = (
    mimic_outcome
    ==
    1
)


MCMED_HOME_MASK = (
    mcmed_outcome
    ==
    0
)


MCMED_ADMITTED_MASK = (
    mcmed_outcome
    ==
    1
)


assert MIMIC_HOME_MASK.sum() == 1606

assert MIMIC_ADMITTED_MASK.sum() == 245

assert MCMED_HOME_MASK.sum() == 4872

assert MCMED_ADMITTED_MASK.sum() == 1835



RAW_FEATURE_POSITION = {
    feature_name: feature_position
    for feature_position, feature_name
    in enumerate(
        LOCKED_RAW_FEATURE_ORDER
    )
}


assert len(
    RAW_FEATURE_POSITION
) == 2495


assert RAW_FEATURE_POSITION[
    "sex_gender"
] >= 0


assert RAW_FEATURE_POSITION[
    "arrival_transport_grouped"
] >= 0



notebook4_handoff_loading_audit = pd.DataFrame(
    [
        {
            "mimic_rows": 1851,

            "mimic_home": int(
                MIMIC_HOME_MASK.sum()
            ),

            "mimic_admitted": int(
                MIMIC_ADMITTED_MASK.sum()
            ),

            "mcmed_rows": 6707,

            "mcmed_home": int(
                MCMED_HOME_MASK.sum()
            ),

            "mcmed_admitted": int(
                MCMED_ADMITTED_MASK.sum()
            ),

            "grouped_feature_count": (
                len(
                    LOCKED_RAW_FEATURE_ORDER
                )
            ),

            "technical_prepared_columns": (
                TECHNICAL_PREPARED_FEATURE_COUNT
            ),

            "mimic_shap_shape": str(
                mimic_shap_values.shape
            ),

            "mcmed_shap_shape": str(
                mcmed_shap_values.shape
            ),

            "mimic_feature_value_shape": str(
                mimic_feature_values.shape
            ),

            "mcmed_feature_value_shape": str(
                mcmed_feature_values.shape
            ),

            "same_feature_names": True,

            "same_feature_order": True,

            "feature_order_sha256": (
                calculated_feature_order_sha256
            ),

            "mimic_shap_checksum_verified": True,

            "mcmed_shap_checksum_verified": True,

            "clinician_universe_checksum_verified": True,

            "notebook3_handoff_checksums_verified": True,

            "model_loaded": False,

            "model_refitted": False,

            "preprocessor_refitted": False,

            "calibrator_refitted": False,

            "threshold_changed": False,

            "robust_shap_filtering_started": False,
        }
    ]
)


display(
    notebook4_handoff_loading_audit
)



print(
    "\n=================================================="
)


print(
    "NOTEBOOK 4 FROZEN HAND-OFF VERIFIED"
)


print(
    "=================================================="
)


print(
    "\nMIMIC-IV INTERNAL TEMPORAL TEST"
)


print(
    "Rows:",
    "1,851",
)


print(
    "HOME:",
    "1,606",
)


print(
    "ADMITTED:",
    "245",
)


print(
    "Grouped SHAP:",
    mimic_shap_values.shape,
)


print(
    "\nMC-MED EXTERNAL VALIDATION"
)


print(
    "Rows:",
    "6,707",
)


print(
    "HOME:",
    "4,872",
)


print(
    "ADMITTED:",
    "1,835",
)


print(
    "Grouped SHAP:",
    mcmed_shap_values.shape,
)


print(
    "\nFEATURE UNIVERSE"
)


print(
    "Clinically interpretable grouped features:",
    "2,495",
)


print(
    "Technical prepared XGBoost columns:",
    "2,498",
)


print(
    "Feature names identical:",
    "YES",
)


print(
    "Feature order identical:",
    "YES",
)


print(
    "Feature-order SHA-256:",
    calculated_feature_order_sha256,
)


print(
    "sex_gender grouped representation:",
    "VERIFIED",
)


print(
    "arrival_transport_grouped representation:",
    "VERIFIED",
)


print(
    "\nCHECKSUMS"
)


print(
    "Notebook 3 hand-off:",
    "PASSED",
)


print(
    "MIMIC grouped SHAP bundle:",
    "PASSED",
)


print(
    "MC-MED grouped SHAP bundle:",
    "PASSED",
)


print(
    "Clinician feature universe:",
    "PASSED",
)


print(
    "\nANALYSIS BOUNDARIES"
)


print(
    "Predictive model loaded:",
    "NO",
)


print(
    "Model refitted:",
    "NO",
)


print(
    "Preprocessor refitted:",
    "NO",
)


print(
    "Calibrator refitted:",
    "NO",
)


print(
    "Threshold changed:",
    "NO",
)


print(
    "Robust SHAP filtering started:",
    "NO",
)


print(
    "\nNOTEBOOK 4 READY FOR ROBUST SHAP ANALYSIS:",
    "YES",
)

##  81-combination SHAP stability and category-tier analysis

The previously established SHAP stability methodology is replicated independently for the MIMIC-IV internal temporal-test SHAP output and the MC-MED external-validation SHAP output.

The complete shared universe of 2,495 grouped clinically interpretable features is used. No arbitrary top-90 feature restriction is applied before stability analysis.

For each dataset, all features are first ranked by mean absolute SHAP:

`mean_abs_shap = mean(|SHAP|)`

For binary or low-cardinality features, the low and high feature-value groups are defined using the minimum and maximum observed values. For continuous features, the low group contains values at or below Q1 and the high group contains values at or above Q3. If Q1 and Q3 are identical, the minimum and maximum observed values are used instead.

For each feature:

- `high_mean_SHAP` = mean SHAP among high/present rows
- `low_mean_SHAP` = mean SHAP among low/absent rows
- `dominant_abs_SHAP` = max(|high_mean_SHAP|, |low_mean_SHAP|)
- `high_low_SHAP_separation` = |high_mean_SHAP − low_mean_SHAP|

`dominant_abs_SHAP` measures the dominant distance from zero, while `high_low_SHAP_separation` measures separation between the high and low feature-value groups.

For each dataset independently, P25, P50 and P75 thresholds are derived from its own eligible feature-level SHAP-effect distribution for both `dominant_abs_SHAP` and `high_low_SHAP_separation`.

Three minimum-support thresholds are evaluated and must be satisfied independently in both the high and low groups:

- loose = max(5, 0.5% of the dataset cohort)
- medium = max(5, 1% of the dataset cohort)
- strict = max(5, 2.5% of the dataset cohort)

Feature-rank sensitivity is evaluated using the complete feature universe N:

- loose = top N features
- medium = top ceil(2N/3) features
- strict = top ceil(N/3) features

For the current shared feature universe:

- N = 2,495
- ceil(2N/3) = 1,664
- ceil(N/3) = 832

Therefore, the robustness grid contains:

3 dominant-distance thresholds ×  
3 high-low-separation thresholds ×  
3 support thresholds ×  
3 feature-rank thresholds  
= 81 combinations.

A feature is selected in a combination only when it satisfies the corresponding feature-rank, support, distance-from-zero, high-low separation and directional-pattern requirements. Near-zero effects, weakly separated effects and same-side/mixed high-versus-low SHAP behaviour are rejected.

Feature stability is:

`stability_pct = selected combinations / 81 × 100`

After stability calculation, features selected at least once are assigned category-relative tags independently within each dataset and feature category:

- High = stability ≥ category P75
- Medium = stability ≥ category P50 and < P75
- Low = stability ≥ category P25 and < P50
- Below P25 = selected but below category P25

Below-P25 features remain stored because they may still become relevant during cross-dataset replication or clinician comparison.

The two datasets are analysed and archived separately. Within each dataset, full category-specific outputs are retained for initial context, triage vitals/pain, laboratory results, imaging/radiology, PMH diagnoses, PMH medication classes and PMH medication names. These tables form the direct inputs to the later clinician-versus-model category comparisons.

For the locked binary XGBoost model, positive grouped SHAP values push predictions toward ADMITTED and negative grouped SHAP values push predictions toward HOME.

This analysis is interpretive only. No model fitting, recalibration, threshold modification, predictive feature reselection or SHAP recalculation is performed.

In [ ]:



NB4_TOTAL_FEATURE_N = int(
    len(
        LOCKED_RAW_FEATURE_ORDER
    )
)


assert NB4_TOTAL_FEATURE_N == 2495


NB4_FEATURE_N_STRICT = int(
    np.ceil(
        NB4_TOTAL_FEATURE_N
        /
        3
    )
)


NB4_FEATURE_N_MEDIUM = int(
    np.ceil(
        2
        *
        NB4_TOTAL_FEATURE_N
        /
        3
    )
)


NB4_FEATURE_N_LOOSE = (
    NB4_TOTAL_FEATURE_N
)


assert NB4_FEATURE_N_STRICT == 832

assert NB4_FEATURE_N_MEDIUM == 1664

assert NB4_FEATURE_N_LOOSE == 2495


NB4_EXPECTED_COMBINATIONS = 81


NB4_CATEGORY_ORDER = [
    "Initial context",
    "Triage vitals / pain",
    "Lab result",
    "Imaging / radiology",
    "PMH ICD title",
    "PMH medication class",
    "PMH medication name",
]



required_nb4_stability_objects = [
    "PROJECT_ROOT",
    "LOCKED_RAW_FEATURE_ORDER",
    "mimic_shap_values",
    "mimic_feature_values",
    "mcmed_shap_values",
    "mcmed_feature_values",
    "clinician_feature_universe",
]


missing_nb4_stability_objects = [
    object_name
    for object_name
    in required_nb4_stability_objects
    if object_name not in globals()
]


if missing_nb4_stability_objects:

    raise RuntimeError(
        "Complete the frozen Notebook 4 hand-off loader "
        "before running SHAP stability analysis.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_nb4_stability_objects
        )
    )


assert mimic_shap_values.shape == (
    1851,
    2495,
)


assert mimic_feature_values.shape == (
    1851,
    2495,
)


assert mcmed_shap_values.shape == (
    6707,
    2495,
)


assert mcmed_feature_values.shape == (
    6707,
    2495,
)



nb4_family_lookup = (
    clinician_feature_universe[
        [
            "source_raw_feature",
            "family",
        ]
    ]
    .drop_duplicates(
        subset=[
            "source_raw_feature",
        ]
    )
    .set_index(
        "source_raw_feature"
    )[
        "family"
    ]
    .astype(str)
    .to_dict()
)


assert len(
    nb4_family_lookup
) == 2495



NB4_INITIAL_CONTEXT_FEATURES = {
    "age",
    "sex_gender",
    "arrival_transport_grouped",
}


def nb4_feature_category(
    feature_name,
):

    family = (
        nb4_family_lookup[
            feature_name
        ]
    )


    if family == "S2_lab_result":

        return "Lab result"


    if family == "S2_imaging_result":

        return "Imaging / radiology"


    if family == "S1_diagnosis":

        return "PMH ICD title"


    if family == "S1_medication_class":

        return "PMH medication class"


    if family == "S1_medication_name":

        return "PMH medication name"


    if family == "S0":

        if feature_name in (
            NB4_INITIAL_CONTEXT_FEATURES
        ):

            return "Initial context"


        return "Triage vitals / pain"


    raise RuntimeError(
        "Unrecognised frozen feature family:\n"
        f"{feature_name}: {family}"
    )



nb4_feature_category_table = pd.DataFrame(
    {
        "feature": (
            LOCKED_RAW_FEATURE_ORDER
        )
    }
)


nb4_feature_category_table[
    "category"
] = (
    nb4_feature_category_table[
        "feature"
    ]
    .map(
        nb4_feature_category
    )
)


assert len(
    nb4_feature_category_table
) == 2495


assert nb4_feature_category_table[
    "category"
].notna().all()


assert set(
    nb4_feature_category_table[
        "category"
    ]
) == set(
    NB4_CATEGORY_ORDER
)


nb4_feature_category_counts = (
    nb4_feature_category_table
    .groupby(
        "category",
        observed=True,
    )
    .size()
    .reindex(
        NB4_CATEGORY_ORDER
    )
    .reset_index(
        name="total_features_in_category"
    )
)


assert (
    nb4_feature_category_counts[
        "total_features_in_category"
    ].sum()
    ==
    2495
)



def nb4_clean_feature_name(
    feature_name,
):

    cleaned = str(
        feature_name
    )


    for prefix in [
        "poe_result__lab__",
        "poe_result__imaging_radiology__",
        "pmh_icd_title__",
        "pmh_med_class__",
        "pmh_med_name__",
    ]:

        if cleaned.startswith(
            prefix
        ):

            cleaned = cleaned[
                len(
                    prefix
                ):
            ]

            break


    return cleaned.replace(
        "__",
        " | ",
    )



mimic_nb4_shap_df = pd.DataFrame(
    mimic_shap_values,
    columns=(
        LOCKED_RAW_FEATURE_ORDER
    ),
)


mimic_nb4_feature_df = pd.DataFrame(
    mimic_feature_values,
    columns=(
        LOCKED_RAW_FEATURE_ORDER
    ),
)


mcmed_nb4_shap_df = pd.DataFrame(
    mcmed_shap_values,
    columns=(
        LOCKED_RAW_FEATURE_ORDER
    ),
)


mcmed_nb4_feature_df = pd.DataFrame(
    mcmed_feature_values,
    columns=(
        LOCKED_RAW_FEATURE_ORDER
    ),
)


assert mimic_nb4_shap_df.columns.tolist() == (
    LOCKED_RAW_FEATURE_ORDER
)


assert mcmed_nb4_shap_df.columns.tolist() == (
    LOCKED_RAW_FEATURE_ORDER
)



def nb4_compute_feature_metrics(
    shap_df,
    feature_df,
):

    assert shap_df.shape == (
        feature_df.shape
    )


    assert shap_df.shape[
        1
    ] == NB4_TOTAL_FEATURE_N


    assert shap_df.columns.tolist() == (
        feature_df.columns.tolist()
    )


    mean_abs_shap = (
        shap_df
        .abs()
        .mean(
            axis=0
        )
        .sort_values(
            ascending=False
        )
    )


    assert len(
        mean_abs_shap
    ) == NB4_TOTAL_FEATURE_N


    metric_rows = []



    for global_rank, (
        feature_name,
        feature_mean_abs_shap,
    ) in enumerate(
        mean_abs_shap.items(),
        start=1,
    ):

        shap_series = pd.to_numeric(
            shap_df[
                feature_name
            ],
            errors="coerce",
        )


        feature_series = pd.to_numeric(
            feature_df[
                feature_name
            ],
            errors="coerce",
        )


        valid_mask = (
            shap_series.notna()
            &
            feature_series.notna()
        )


        valid_shap = (
            shap_series[
                valid_mask
            ]
        )


        valid_feature = (
            feature_series[
                valid_mask
            ]
        )


        n_valid = int(
            valid_mask.sum()
        )


        high_mean = np.nan

        low_mean = np.nan

        high_median = np.nan

        low_median = np.nan

        high_group_description = (
            "not available"
        )

        low_group_description = (
            "not available"
        )

        high_low_definition = (
            "not available"
        )

        n_high = 0

        n_low = 0

        dominant_abs_shap = np.nan

        abs_high_low_separation = np.nan


        if (
            n_valid
            >=
            2
            and
            valid_feature.nunique(
                dropna=True
            )
            >=
            2
        ):

            unique_values = np.sort(
                valid_feature
                .dropna()
                .unique()
            )



            if len(
                unique_values
            ) <= 4:

                low_value = (
                    unique_values[
                        0
                    ]
                )


                high_value = (
                    unique_values[
                        -1
                    ]
                )


                low_mask = (
                    valid_feature.eq(
                        low_value
                    )
                )


                high_mask = (
                    valid_feature.eq(
                        high_value
                    )
                )


                low_group_description = (
                    f"= {low_value:.4g}"
                )


                high_group_description = (
                    f"= {high_value:.4g}"
                )


                high_low_definition = (
                    "min value vs max value"
                )



            else:

                q1 = float(
                    valid_feature.quantile(
                        0.25
                    )
                )


                q3 = float(
                    valid_feature.quantile(
                        0.75
                    )
                )


                if np.isclose(
                    q1,
                    q3,
                ):

                    low_value = (
                        unique_values[
                            0
                        ]
                    )


                    high_value = (
                        unique_values[
                            -1
                        ]
                    )


                    low_mask = (
                        valid_feature.eq(
                            low_value
                        )
                    )


                    high_mask = (
                        valid_feature.eq(
                            high_value
                        )
                    )


                    low_group_description = (
                        f"= {low_value:.4g}"
                    )


                    high_group_description = (
                        f"= {high_value:.4g}"
                    )


                    high_low_definition = (
                        "min value vs max value "
                        "because Q1 = Q3"
                    )


                else:

                    low_mask = (
                        valid_feature.le(
                            q1
                        )
                    )


                    high_mask = (
                        valid_feature.ge(
                            q3
                        )
                    )


                    low_group_description = (
                        f"≤ {q1:.4g}"
                    )


                    high_group_description = (
                        f"≥ {q3:.4g}"
                    )


                    high_low_definition = (
                        "bottom quartile vs top quartile"
                    )


            n_low = int(
                low_mask.sum()
            )


            n_high = int(
                high_mask.sum()
            )


            if (
                n_low > 0
                and
                n_high > 0
            ):

                low_shap = (
                    valid_shap[
                        low_mask
                    ]
                )


                high_shap = (
                    valid_shap[
                        high_mask
                    ]
                )


                low_mean = float(
                    low_shap.mean()
                )


                high_mean = float(
                    high_shap.mean()
                )


                low_median = float(
                    low_shap.median()
                )


                high_median = float(
                    high_shap.median()
                )


                dominant_abs_shap = float(
                    max(
                        abs(
                            high_mean
                        ),
                        abs(
                            low_mean
                        ),
                    )
                )


                abs_high_low_separation = float(
                    abs(
                        high_mean
                        -
                        low_mean
                    )
                )


        metric_rows.append(
            {
                "global_rank": (
                    global_rank
                ),

                "category": (
                    nb4_feature_category(
                        feature_name
                    )
                ),

                "feature": (
                    feature_name
                ),

                "feature_clean": (
                    nb4_clean_feature_name(
                        feature_name
                    )
                ),

                "mean_abs_shap": float(
                    feature_mean_abs_shap
                ),

                "mean_shap_high_value": (
                    high_mean
                ),

                "mean_shap_low_value": (
                    low_mean
                ),

                "median_shap_high_value": (
                    high_median
                ),

                "median_shap_low_value": (
                    low_median
                ),

                "dominant_abs_shap": (
                    dominant_abs_shap
                ),

                "abs_high_low_separation": (
                    abs_high_low_separation
                ),

                "high_value_group": (
                    high_group_description
                ),

                "low_value_group": (
                    low_group_description
                ),

                "high_low_definition": (
                    high_low_definition
                ),

                "n_high": (
                    n_high
                ),

                "n_low": (
                    n_low
                ),

                "n_valid": (
                    n_valid
                ),
            }
        )


    metrics_df = pd.DataFrame(
        metric_rows
    )


    assert len(
        metrics_df
    ) == NB4_TOTAL_FEATURE_N


    assert metrics_df[
        "global_rank"
    ].min() == 1


    assert metrics_df[
        "global_rank"
    ].max() == (
        NB4_TOTAL_FEATURE_N
    )


    metrics_df[
        "category"
    ] = pd.Categorical(
        metrics_df[
            "category"
        ],
        categories=(
            NB4_CATEGORY_ORDER
        ),
        ordered=True,
    )


    return metrics_df



def nb4_check_pattern(
    high_mean,
    low_mean,
    min_side_abs_shap,
    min_abs_separation,
):

    if (
        pd.isna(
            high_mean
        )
        or
        pd.isna(
            low_mean
        )
    ):

        return (
            False,
            "not_interpretable",
            "No clear pattern",
        )


    dominant_abs = max(
        abs(
            high_mean
        ),
        abs(
            low_mean
        ),
    )


    abs_separation = abs(
        high_mean
        -
        low_mean
    )



    if dominant_abs < (
        min_side_abs_shap
    ):

        return (
            False,
            "weak_dominant_shap",
            (
                "Dominant SHAP magnitude "
                "too close to zero"
            ),
        )



    if abs_separation < (
        min_abs_separation
    ):

        return (
            False,
            "weak_separation",
            (
                "High-low SHAP separation "
                "too small"
            ),
        )


    eps = float(
        min_side_abs_shap
    )



    if (
        high_mean >= eps
        and
        low_mean <= eps
    ):

        return (
            True,
            "high_present_increases_admitted",
            (
                "High/present values push toward ADMITTED; "
                "low/absent values are near zero or HOME"
            ),
        )



    if (
        high_mean <= -eps
        and
        low_mean >= -eps
    ):

        return (
            True,
            "high_present_increases_home",
            (
                "High/present values push toward HOME; "
                "low/absent values are near zero or ADMITTED"
            ),
        )



    if (
        low_mean >= eps
        and
        high_mean <= eps
    ):

        return (
            True,
            "low_absent_increases_admitted",
            (
                "Low/absent values push toward ADMITTED; "
                "high/present values are near zero or HOME"
            ),
        )



    if (
        low_mean <= -eps
        and
        high_mean >= -eps
    ):

        return (
            True,
            "low_absent_increases_home",
            (
                "Low/absent values push toward HOME; "
                "high/present values are near zero or ADMITTED"
            ),
        )


    return (
        False,
        "same_side_or_mixed",
        (
            "High and low groups have same-side "
            "or mixed SHAP behaviour"
        ),
    )



def nb4_make_percentile_thresholds(
    values,
    threshold_type,
):

    clean_values = (
        pd.Series(
            values
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    clean_values = (
        clean_values.loc[
            clean_values
            >
            0
        ]
    )


    percentiles = [
        25,
        50,
        75,
    ]


    levels = [
        "loose_p25",
        "medium_p50",
        "strict_p75",
    ]


    if clean_values.empty:

        threshold_values = [
            0.0,
            0.0,
            0.0,
        ]


    else:

        threshold_values = [
            float(
                np.nanpercentile(
                    clean_values,
                    percentile,
                )
            )
            for percentile
            in percentiles
        ]


    return pd.DataFrame(
        {
            "threshold_type": (
                threshold_type
            ),

            "level": (
                levels
            ),

            "percentile": (
                percentiles
            ),

            "threshold_value": (
                threshold_values
            ),
        }
    )



def nb4_make_support_thresholds(
    n_rows,
):

    support_specs = [
        (
            "loose_max5_or_0.5pct",
            0.005,
        ),

        (
            "medium_max5_or_1pct",
            0.010,
        ),

        (
            "strict_max5_or_2.5pct",
            0.025,
        ),
    ]


    support_rows = []


    for level, fraction in (
        support_specs
    ):

        support_n = int(
            np.ceil(
                max(
                    5,
                    fraction
                    *
                    n_rows,
                )
            )
        )


        support_rows.append(
            {
                "threshold_type": (
                    "minimum_support_each_high_low_group"
                ),

                "level": (
                    level
                ),

                "fraction": (
                    fraction
                ),

                "dataset_rows": (
                    n_rows
                ),

                "threshold_value": (
                    support_n
                ),
            }
        )


    support_df = pd.DataFrame(
        support_rows
    )


    assert len(
        support_df
    ) == 3


    return support_df



def nb4_make_total_n_rank_thresholds():

    rank_thresholds_df = pd.DataFrame(
        [
            {
                "threshold_type": (
                    "ranked_feature_pool"
                ),

                "level": (
                    "loose_total_N"
                ),

                "fraction_of_total_N": (
                    1.0
                ),

                "threshold_value": (
                    NB4_FEATURE_N_LOOSE
                ),
            },

            {
                "threshold_type": (
                    "ranked_feature_pool"
                ),

                "level": (
                    "medium_two_thirds_N"
                ),

                "fraction_of_total_N": (
                    2.0
                    /
                    3.0
                ),

                "threshold_value": (
                    NB4_FEATURE_N_MEDIUM
                ),
            },

            {
                "threshold_type": (
                    "ranked_feature_pool"
                ),

                "level": (
                    "strict_one_third_N"
                ),

                "fraction_of_total_N": (
                    1.0
                    /
                    3.0
                ),

                "threshold_value": (
                    NB4_FEATURE_N_STRICT
                ),
            },
        ]
    )


    assert rank_thresholds_df[
        "threshold_value"
    ].tolist() == [
        2495,
        1664,
        832,
    ]


    return rank_thresholds_df



def nb4_run_81_combination_stability(
    shap_df,
    feature_df,
    dataset_name,
):

    n_rows = int(
        len(
            shap_df
        )
    )



    metrics_df = (
        nb4_compute_feature_metrics(
            shap_df=shap_df,
            feature_df=feature_df,
        )
    )


    assert len(
        metrics_df
    ) == NB4_TOTAL_FEATURE_N



    support_thresholds_df = (
        nb4_make_support_thresholds(
            n_rows
        )
    )


    minimum_support_for_percentile_reference = int(
        support_thresholds_df[
            "threshold_value"
        ]
        .min()
    )



    percentile_reference_df = (
        metrics_df.loc[
            metrics_df[
                "dominant_abs_shap"
            ].notna()
            &
            metrics_df[
                "abs_high_low_separation"
            ].notna()
            &
            metrics_df[
                "n_high"
            ].ge(
                minimum_support_for_percentile_reference
            )
            &
            metrics_df[
                "n_low"
            ].ge(
                minimum_support_for_percentile_reference
            )
        ]
        .copy()
    )


    if len(
        percentile_reference_df
    ) < 5:

        percentile_reference_df = (
            metrics_df.loc[
                metrics_df[
                    "dominant_abs_shap"
                ].notna()
                &
                metrics_df[
                    "abs_high_low_separation"
                ].notna()
            ]
            .copy()
        )


    assert not percentile_reference_df.empty



    distance_thresholds_df = (
        nb4_make_percentile_thresholds(
            percentile_reference_df[
                "dominant_abs_shap"
            ],
            (
                "dominant_abs_shap_"
                "distance_from_zero"
            ),
        )
    )



    separation_thresholds_df = (
        nb4_make_percentile_thresholds(
            percentile_reference_df[
                "abs_high_low_separation"
            ],
            (
                "absolute_high_low_shap_separation"
            ),
        )
    )



    rank_thresholds_df = (
        nb4_make_total_n_rank_thresholds()
    )


    thresholds_df = pd.concat(
        [
            distance_thresholds_df,
            separation_thresholds_df,
            support_thresholds_df,
            rank_thresholds_df,
        ],
        ignore_index=True,
    )



    grid_rows = []

    selection_event_rows = []

    combo_id = 0


    for rank_row in (
        rank_thresholds_df
        .itertuples(
            index=False
        )
    ):

        rank_level = (
            rank_row.level
        )


        maximum_global_rank = int(
            rank_row.threshold_value
        )


        rank_candidate_df = (
            metrics_df.loc[
                metrics_df[
                    "global_rank"
                ]
                <=
                maximum_global_rank
            ]
            .copy()
        )


        for distance_row in (
            distance_thresholds_df
            .itertuples(
                index=False
            )
        ):

            distance_level = (
                distance_row.level
            )


            minimum_dominant_abs_shap = float(
                distance_row.threshold_value
            )


            for separation_row in (
                separation_thresholds_df
                .itertuples(
                    index=False
                )
            ):

                separation_level = (
                    separation_row.level
                )


                minimum_abs_separation = float(
                    separation_row.threshold_value
                )


                for support_row in (
                    support_thresholds_df
                    .itertuples(
                        index=False
                    )
                ):

                    support_level = (
                        support_row.level
                    )


                    minimum_group_n = int(
                        support_row.threshold_value
                    )


                    combo_id += 1


                    grid_rows.append(
                        {
                            "dataset": (
                                dataset_name
                            ),

                            "combo_id": (
                                combo_id
                            ),

                            "rank_level": (
                                rank_level
                            ),

                            "maximum_global_rank": (
                                maximum_global_rank
                            ),

                            "distance_level": (
                                distance_level
                            ),

                            "minimum_dominant_abs_shap": (
                                minimum_dominant_abs_shap
                            ),

                            "separation_level": (
                                separation_level
                            ),

                            "minimum_abs_separation": (
                                minimum_abs_separation
                            ),

                            "support_level": (
                                support_level
                            ),

                            "minimum_group_n": (
                                minimum_group_n
                            ),
                        }
                    )



                    for feature_row in (
                        rank_candidate_df
                        .itertuples(
                            index=False
                        )
                    ):

                        support_ok = bool(
                            int(
                                feature_row.n_high
                            )
                            >=
                            minimum_group_n
                            and
                            int(
                                feature_row.n_low
                            )
                            >=
                            minimum_group_n
                        )


                        (
                            useful_pattern,
                            pattern_code,
                            pattern_description,
                        ) = nb4_check_pattern(
                            high_mean=(
                                feature_row.mean_shap_high_value
                            ),

                            low_mean=(
                                feature_row.mean_shap_low_value
                            ),

                            min_side_abs_shap=(
                                minimum_dominant_abs_shap
                            ),

                            min_abs_separation=(
                                minimum_abs_separation
                            ),
                        )


                        selected = bool(
                            support_ok
                            and
                            useful_pattern
                        )


                        if selected:

                            selection_event_rows.append(
                                {
                                    "dataset": (
                                        dataset_name
                                    ),

                                    "combo_id": (
                                        combo_id
                                    ),

                                    "rank_level": (
                                        rank_level
                                    ),

                                    "maximum_global_rank": (
                                        maximum_global_rank
                                    ),

                                    "distance_level": (
                                        distance_level
                                    ),

                                    "minimum_dominant_abs_shap": (
                                        minimum_dominant_abs_shap
                                    ),

                                    "separation_level": (
                                        separation_level
                                    ),

                                    "minimum_abs_separation": (
                                        minimum_abs_separation
                                    ),

                                    "support_level": (
                                        support_level
                                    ),

                                    "minimum_group_n": (
                                        minimum_group_n
                                    ),

                                    "global_rank": (
                                        feature_row.global_rank
                                    ),

                                    "category": (
                                        str(
                                            feature_row.category
                                        )
                                    ),

                                    "feature": (
                                        feature_row.feature
                                    ),

                                    "feature_clean": (
                                        feature_row.feature_clean
                                    ),

                                    "mean_abs_shap": (
                                        feature_row.mean_abs_shap
                                    ),

                                    "dominant_abs_shap": (
                                        feature_row.dominant_abs_shap
                                    ),

                                    "abs_high_low_separation": (
                                        feature_row.abs_high_low_separation
                                    ),

                                    "mean_shap_high_value": (
                                        feature_row.mean_shap_high_value
                                    ),

                                    "mean_shap_low_value": (
                                        feature_row.mean_shap_low_value
                                    ),

                                    "high_value_group": (
                                        feature_row.high_value_group
                                    ),

                                    "low_value_group": (
                                        feature_row.low_value_group
                                    ),

                                    "n_high": (
                                        feature_row.n_high
                                    ),

                                    "n_low": (
                                        feature_row.n_low
                                    ),

                                    "pattern_code": (
                                        pattern_code
                                    ),

                                    "pattern": (
                                        pattern_description
                                    ),
                                }
                            )


    stability_grid_df = pd.DataFrame(
        grid_rows
    )


    selection_events_df = pd.DataFrame(
        selection_event_rows
    )


    total_combinations = int(
        len(
            stability_grid_df
        )
    )


    assert total_combinations == 81


    assert combo_id == 81



    if selection_events_df.empty:

        selected_counts = pd.DataFrame(
            {
                "feature": (
                    metrics_df[
                        "feature"
                    ]
                ),

                "times_selected": 0,
            }
        )


    else:

        selected_counts = (
            selection_events_df
            .groupby(
                "feature",
                as_index=False,
            )[
                "combo_id"
            ]
            .nunique()
            .rename(
                columns={
                    "combo_id": (
                        "times_selected"
                    ),
                }
            )
        )


    stability_all_features_df = (
        metrics_df
        .merge(
            selected_counts,
            on="feature",
            how="left",
            validate="one_to_one",
        )
    )


    stability_all_features_df[
        "times_selected"
    ] = (
        stability_all_features_df[
            "times_selected"
        ]
        .fillna(
            0
        )
        .astype(int)
    )


    stability_all_features_df[
        "total_combinations"
    ] = 81


    stability_all_features_df[
        "stability"
    ] = (
        stability_all_features_df[
            "times_selected"
        ]
        /
        81
    )


    stability_all_features_df[
        "stability_pct"
    ] = (
        100.0
        *
        stability_all_features_df[
            "stability"
        ]
    )



    if not selection_events_df.empty:

        pattern_summary = (
            selection_events_df
            .groupby(
                "feature"
            )
            .agg(
                most_common_pattern_code=(
                    "pattern_code",
                    lambda values:
                    values.value_counts().index[
                        0
                    ],
                ),

                most_common_pattern=(
                    "pattern",
                    lambda values:
                    values.value_counts().index[
                        0
                    ],
                ),

                selected_minimum_global_rank_limit=(
                    "maximum_global_rank",
                    "min",
                ),

                selected_maximum_global_rank_limit=(
                    "maximum_global_rank",
                    "max",
                ),

                selected_minimum_support_n=(
                    "minimum_group_n",
                    "min",
                ),

                selected_maximum_support_n=(
                    "minimum_group_n",
                    "max",
                ),
            )
            .reset_index()
        )


        stability_all_features_df = (
            stability_all_features_df
            .merge(
                pattern_summary,
                on="feature",
                how="left",
                validate="one_to_one",
            )
        )


    stability_all_features_df[
        "dataset"
    ] = (
        dataset_name
    )


    stability_all_features_df[
        "category"
    ] = pd.Categorical(
        stability_all_features_df[
            "category"
        ],
        categories=(
            NB4_CATEGORY_ORDER
        ),
        ordered=True,
    )


    stability_all_features_df = (
        stability_all_features_df
        .sort_values(
            [
                "category",
                "stability_pct",
                "mean_abs_shap",
                "global_rank",
            ],
            ascending=[
                True,
                False,
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    return {
        "dataset": (
            dataset_name
        ),

        "n_rows": (
            n_rows
        ),

        "total_feature_N": (
            NB4_TOTAL_FEATURE_N
        ),

        "rank_N_loose": (
            NB4_FEATURE_N_LOOSE
        ),

        "rank_N_medium": (
            NB4_FEATURE_N_MEDIUM
        ),

        "rank_N_strict": (
            NB4_FEATURE_N_STRICT
        ),

        "total_combinations": 81,

        "thresholds_df": (
            thresholds_df
        ),

        "stability_grid_df": (
            stability_grid_df
        ),

        "feature_metrics_df": (
            metrics_df
        ),

        "percentile_reference_df": (
            percentile_reference_df
        ),

        "selection_events_df": (
            selection_events_df
        ),

        "stability_all_features_df": (
            stability_all_features_df
        ),
    }



def nb4_assign_category_tiers(
    stability_result,
):

    full_df = (
        stability_result[
            "stability_all_features_df"
        ]
        .copy()
    )


    full_df[
        "importance_tag"
    ] = "Not selected"


    selected_df = (
        full_df.loc[
            full_df[
                "times_selected"
            ]
            >
            0
        ]
        .copy()
    )


    threshold_rows = []


    for category in (
        NB4_CATEGORY_ORDER
    ):

        category_selected_df = (
            selected_df.loc[
                selected_df[
                    "category"
                ]
                .astype(str)
                .eq(
                    category
                )
            ]
            .copy()
        )


        if category_selected_df.empty:

            threshold_rows.append(
                {
                    "dataset": (
                        stability_result[
                            "dataset"
                        ]
                    ),

                    "category": (
                        category
                    ),

                    "n_features_in_category": int(
                        (
                            full_df[
                                "category"
                            ]
                            .astype(str)
                            ==
                            category
                        ).sum()
                    ),

                    "n_selected_features_for_tiering": 0,

                    "stability_p25_low_threshold": np.nan,

                    "stability_p50_medium_threshold": np.nan,

                    "stability_p75_high_threshold": np.nan,
                }
            )


            continue


        p25 = float(
            np.nanpercentile(
                category_selected_df[
                    "stability_pct"
                ],
                25,
            )
        )


        p50 = float(
            np.nanpercentile(
                category_selected_df[
                    "stability_pct"
                ],
                50,
            )
        )


        p75 = float(
            np.nanpercentile(
                category_selected_df[
                    "stability_pct"
                ],
                75,
            )
        )


        threshold_rows.append(
            {
                "dataset": (
                    stability_result[
                        "dataset"
                    ]
                ),

                "category": (
                    category
                ),

                "n_features_in_category": int(
                    (
                        full_df[
                            "category"
                        ]
                        .astype(str)
                        ==
                        category
                    ).sum()
                ),

                "n_selected_features_for_tiering": int(
                    len(
                        category_selected_df
                    )
                ),

                "stability_p25_low_threshold": (
                    p25
                ),

                "stability_p50_medium_threshold": (
                    p50
                ),

                "stability_p75_high_threshold": (
                    p75
                ),
            }
        )


        for feature_name in (
            category_selected_df[
                "feature"
            ]
        ):

            stability_pct = float(
                full_df.loc[
                    full_df[
                        "feature"
                    ]
                    .eq(
                        feature_name
                    ),
                    "stability_pct",
                ]
                .iloc[
                    0
                ]
            )


            if stability_pct >= p75:

                tag = "High"


            elif stability_pct >= p50:

                tag = "Medium"


            elif stability_pct >= p25:

                tag = "Low"


            else:

                tag = "Below P25"


            full_df.loc[
                full_df[
                    "feature"
                ]
                .eq(
                    feature_name
                ),
                "importance_tag",
            ] = tag


    tier_thresholds_df = pd.DataFrame(
        threshold_rows
    )



    tiered_feature_table = (
        full_df.loc[
            full_df[
                "times_selected"
            ]
            >
            0
        ]
        .copy()
    )


    tier_sort_order = {
        "High": 1,
        "Medium": 2,
        "Low": 3,
        "Below P25": 4,
    }


    tiered_feature_table[
        "_tier_sort"
    ] = (
        tiered_feature_table[
            "importance_tag"
        ]
        .map(
            tier_sort_order
        )
    )


    tiered_feature_table = (
        tiered_feature_table
        .sort_values(
            [
                "category",
                "_tier_sort",
                "stability_pct",
                "mean_abs_shap",
                "global_rank",
            ],
            ascending=[
                True,
                True,
                False,
                False,
                True,
            ],
        )
        .drop(
            columns=[
                "_tier_sort",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    tiered_feature_table[
        "category_rank_by_stability"
    ] = (
        tiered_feature_table
        .groupby(
            "category",
            observed=True,
        )
        .cumcount()
        +
        1
    )



    category_tables = {}


    for category in (
        NB4_CATEGORY_ORDER
    ):

        category_tables[
            category
        ] = (
            tiered_feature_table.loc[
                tiered_feature_table[
                    "category"
                ]
                .astype(str)
                .eq(
                    category
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


    return (
        full_df,
        tiered_feature_table,
        tier_thresholds_df,
        category_tables,
    )



nb4_stability_start_time = (
    time.perf_counter()
)


nb4_mimic_stability_result = (
    nb4_run_81_combination_stability(
        shap_df=(
            mimic_nb4_shap_df
        ),

        feature_df=(
            mimic_nb4_feature_df
        ),

        dataset_name=(
            "MIMIC-IV internal temporal test"
        ),
    )
)


(
    nb4_mimic_all_stability_features,
    nb4_mimic_tiered_feature_table,
    nb4_mimic_tier_thresholds_df,
    nb4_mimic_tiered_tables_by_category,
) = nb4_assign_category_tiers(
    nb4_mimic_stability_result
)



nb4_mcmed_stability_result = (
    nb4_run_81_combination_stability(
        shap_df=(
            mcmed_nb4_shap_df
        ),

        feature_df=(
            mcmed_nb4_feature_df
        ),

        dataset_name=(
            "MC-MED external validation"
        ),
    )
)


(
    nb4_mcmed_all_stability_features,
    nb4_mcmed_tiered_feature_table,
    nb4_mcmed_tier_thresholds_df,
    nb4_mcmed_tiered_tables_by_category,
) = nb4_assign_category_tiers(
    nb4_mcmed_stability_result
)


nb4_stability_elapsed_seconds = (
    time.perf_counter()
    -
    nb4_stability_start_time
)



nb4_mimic_initial_context_tiered_df = (
    nb4_mimic_tiered_tables_by_category[
        "Initial context"
    ]
)


nb4_mimic_triage_vitals_pain_tiered_df = (
    nb4_mimic_tiered_tables_by_category[
        "Triage vitals / pain"
    ]
)


nb4_mimic_lab_result_tiered_df = (
    nb4_mimic_tiered_tables_by_category[
        "Lab result"
    ]
)


nb4_mimic_imaging_radiology_tiered_df = (
    nb4_mimic_tiered_tables_by_category[
        "Imaging / radiology"
    ]
)


nb4_mimic_pmh_icd_title_tiered_df = (
    nb4_mimic_tiered_tables_by_category[
        "PMH ICD title"
    ]
)


nb4_mimic_pmh_med_class_tiered_df = (
    nb4_mimic_tiered_tables_by_category[
        "PMH medication class"
    ]
)


nb4_mimic_pmh_med_name_tiered_df = (
    nb4_mimic_tiered_tables_by_category[
        "PMH medication name"
    ]
)



nb4_mcmed_initial_context_tiered_df = (
    nb4_mcmed_tiered_tables_by_category[
        "Initial context"
    ]
)


nb4_mcmed_triage_vitals_pain_tiered_df = (
    nb4_mcmed_tiered_tables_by_category[
        "Triage vitals / pain"
    ]
)


nb4_mcmed_lab_result_tiered_df = (
    nb4_mcmed_tiered_tables_by_category[
        "Lab result"
    ]
)


nb4_mcmed_imaging_radiology_tiered_df = (
    nb4_mcmed_tiered_tables_by_category[
        "Imaging / radiology"
    ]
)


nb4_mcmed_pmh_icd_title_tiered_df = (
    nb4_mcmed_tiered_tables_by_category[
        "PMH ICD title"
    ]
)


nb4_mcmed_pmh_med_class_tiered_df = (
    nb4_mcmed_tiered_tables_by_category[
        "PMH medication class"
    ]
)


nb4_mcmed_pmh_med_name_tiered_df = (
    nb4_mcmed_tiered_tables_by_category[
        "PMH medication name"
    ]
)



nb4_mimic_category_tier_summary = (
    nb4_mimic_tiered_feature_table
    .groupby(
        [
            "category",
            "importance_tag",
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="n_features"
    )
)


nb4_mimic_category_tier_summary.insert(
    0,
    "dataset",
    "MIMIC-IV internal temporal test",
)


nb4_mcmed_category_tier_summary = (
    nb4_mcmed_tiered_feature_table
    .groupby(
        [
            "category",
            "importance_tag",
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="n_features"
    )
)


nb4_mcmed_category_tier_summary.insert(
    0,
    "dataset",
    "MC-MED external validation",
)


nb4_category_tier_summary = pd.concat(
    [
        nb4_mimic_category_tier_summary,
        nb4_mcmed_category_tier_summary,
    ],
    ignore_index=True,
)



def nb4_make_category_selection_summary(
    full_stability_df,
    dataset_name,
):

    rows = []


    for category in (
        NB4_CATEGORY_ORDER
    ):

        category_full = (
            full_stability_df.loc[
                full_stability_df[
                    "category"
                ]
                .astype(str)
                .eq(
                    category
                )
            ]
            .copy()
        )


        category_selected = (
            category_full.loc[
                category_full[
                    "times_selected"
                ]
                >
                0
            ]
            .copy()
        )


        rows.append(
            {
                "dataset": (
                    dataset_name
                ),

                "category": (
                    category
                ),

                "total_model_features_in_category": int(
                    len(
                        category_full
                    )
                ),

                "selected_at_least_once": int(
                    len(
                        category_selected
                    )
                ),

                "not_selected": int(
                    (
                        category_full[
                            "times_selected"
                        ]
                        ==
                        0
                    ).sum()
                ),

                "selected_pct_of_category": float(
                    100.0
                    *
                    len(
                        category_selected
                    )
                    /
                    len(
                        category_full
                    )
                )
                if len(
                    category_full
                )
                >
                0
                else np.nan,

                "high": int(
                    (
                        category_selected[
                            "importance_tag"
                        ]
                        ==
                        "High"
                    ).sum()
                ),

                "medium": int(
                    (
                        category_selected[
                            "importance_tag"
                        ]
                        ==
                        "Medium"
                    ).sum()
                ),

                "low": int(
                    (
                        category_selected[
                            "importance_tag"
                        ]
                        ==
                        "Low"
                    ).sum()
                ),

                "below_p25": int(
                    (
                        category_selected[
                            "importance_tag"
                        ]
                        ==
                        "Below P25"
                    ).sum()
                ),
            }
        )


    return pd.DataFrame(
        rows
    )


nb4_mimic_category_selection_summary = (
    nb4_make_category_selection_summary(
        nb4_mimic_all_stability_features,
        "MIMIC-IV internal temporal test",
    )
)


nb4_mcmed_category_selection_summary = (
    nb4_make_category_selection_summary(
        nb4_mcmed_all_stability_features,
        "MC-MED external validation",
    )
)


nb4_category_selection_summary = pd.concat(
    [
        nb4_mimic_category_selection_summary,
        nb4_mcmed_category_selection_summary,
    ],
    ignore_index=True,
)



NOTEBOOK4_STABILITY_ROOT = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook4"
    / "shap_stability"
)


NOTEBOOK4_MIMIC_STABILITY_DIR = (
    NOTEBOOK4_STABILITY_ROOT
    / "mimic"
)


NOTEBOOK4_MCMED_STABILITY_DIR = (
    NOTEBOOK4_STABILITY_ROOT
    / "mcmed"
)


NOTEBOOK4_COMBINED_STABILITY_DIR = (
    NOTEBOOK4_STABILITY_ROOT
    / "combined_summary"
)


for output_directory in [
    NOTEBOOK4_STABILITY_ROOT,
    NOTEBOOK4_MIMIC_STABILITY_DIR,
    NOTEBOOK4_MCMED_STABILITY_DIR,
    NOTEBOOK4_COMBINED_STABILITY_DIR,
]:

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )



def nb4_category_filename(
    category,
):

    return (
        category
        .lower()
        .replace(
            " / ",
            "_"
        )
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )



def nb4_save_dataset_stability_outputs(
    dataset_prefix,
    output_directory,
    stability_result,
    all_features_df,
    tiered_features_df,
    tier_thresholds_df,
    category_tables,
    category_selection_summary,
    category_tier_summary,
):


    all_features_df.to_csv(
        output_directory
        /
        f"{dataset_prefix}_all_2495_stability_features.csv",
        index=False,
    )


    tiered_features_df.to_csv(
        output_directory
        /
        f"{dataset_prefix}_selected_tiered_features.csv",
        index=False,
    )


    stability_result[
        "feature_metrics_df"
    ].to_csv(
        output_directory
        /
        f"{dataset_prefix}_all_2495_feature_shap_metrics.csv",
        index=False,
    )


    stability_result[
        "thresholds_df"
    ].to_csv(
        output_directory
        /
        f"{dataset_prefix}_81_combination_thresholds.csv",
        index=False,
    )


    stability_result[
        "stability_grid_df"
    ].to_csv(
        output_directory
        /
        f"{dataset_prefix}_81_combination_grid.csv",
        index=False,
    )


    stability_result[
        "selection_events_df"
    ].to_parquet(
        output_directory
        /
        f"{dataset_prefix}_81_combination_selection_events.parquet",
        index=False,
        engine="pyarrow",
    )


    tier_thresholds_df.to_csv(
        output_directory
        /
        f"{dataset_prefix}_category_relative_tier_thresholds.csv",
        index=False,
    )


    category_selection_summary.to_csv(
        output_directory
        /
        f"{dataset_prefix}_category_selection_summary.csv",
        index=False,
    )


    category_tier_summary.to_csv(
        output_directory
        /
        f"{dataset_prefix}_category_tier_summary.csv",
        index=False,
    )



    category_root = (
        output_directory
        /
        "categories"
    )


    category_root.mkdir(
        parents=True,
        exist_ok=True,
    )


    for category, category_df in (
        category_tables.items()
    ):

        safe_category = (
            nb4_category_filename(
                category
            )
        )


        category_df.to_csv(
            category_root
            /
            (
                f"{dataset_prefix}_"
                f"{safe_category}_tiered_features.csv"
            ),
            index=False,
        )


        full_category_df = (
            all_features_df.loc[
                all_features_df[
                    "category"
                ]
                .astype(str)
                .eq(
                    category
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        full_category_df.to_csv(
            category_root
            /
            (
                f"{dataset_prefix}_"
                f"{safe_category}_all_model_features.csv"
            ),
            index=False,
        )



nb4_save_dataset_stability_outputs(
    dataset_prefix="mimic",
    output_directory=(
        NOTEBOOK4_MIMIC_STABILITY_DIR
    ),
    stability_result=(
        nb4_mimic_stability_result
    ),
    all_features_df=(
        nb4_mimic_all_stability_features
    ),
    tiered_features_df=(
        nb4_mimic_tiered_feature_table
    ),
    tier_thresholds_df=(
        nb4_mimic_tier_thresholds_df
    ),
    category_tables=(
        nb4_mimic_tiered_tables_by_category
    ),
    category_selection_summary=(
        nb4_mimic_category_selection_summary
    ),
    category_tier_summary=(
        nb4_mimic_category_tier_summary
    ),
)



nb4_save_dataset_stability_outputs(
    dataset_prefix="mcmed",
    output_directory=(
        NOTEBOOK4_MCMED_STABILITY_DIR
    ),
    stability_result=(
        nb4_mcmed_stability_result
    ),
    all_features_df=(
        nb4_mcmed_all_stability_features
    ),
    tiered_features_df=(
        nb4_mcmed_tiered_feature_table
    ),
    tier_thresholds_df=(
        nb4_mcmed_tier_thresholds_df
    ),
    category_tables=(
        nb4_mcmed_tiered_tables_by_category
    ),
    category_selection_summary=(
        nb4_mcmed_category_selection_summary
    ),
    category_tier_summary=(
        nb4_mcmed_category_tier_summary
    ),
)



nb4_category_selection_summary.to_csv(
    NOTEBOOK4_COMBINED_STABILITY_DIR
    /
    "mimic_mcmed_category_selection_summary.csv",
    index=False,
)


nb4_category_tier_summary.to_csv(
    NOTEBOOK4_COMBINED_STABILITY_DIR
    /
    "mimic_mcmed_category_tier_summary.csv",
    index=False,
)


nb4_feature_category_counts.to_csv(
    NOTEBOOK4_COMBINED_STABILITY_DIR
    /
    "shared_2495_feature_category_counts.csv",
    index=False,
)



nb4_stability_methodology_audit = pd.DataFrame(
    [
        {
            "method": (
                "Replicated 81-combination "
                "Pathway 5 SHAP stability"
            ),

            "total_feature_N": (
                NB4_TOTAL_FEATURE_N
            ),

            "rank_loose_N": (
                NB4_FEATURE_N_LOOSE
            ),

            "rank_medium_ceil_2N_over_3": (
                NB4_FEATURE_N_MEDIUM
            ),

            "rank_strict_ceil_N_over_3": (
                NB4_FEATURE_N_STRICT
            ),

            "dominant_abs_thresholds": (
                "dataset-specific P25 | P50 | P75"
            ),

            "separation_thresholds": (
                "dataset-specific P25 | P50 | P75"
            ),

            "support_thresholds": (
                "max(5, 0.5%) | max(5, 1%) | "
                "max(5, 2.5%)"
            ),

            "total_combinations_per_dataset": 81,

            "mimic_analyzed_independently": True,

            "mcmed_analyzed_independently": True,

            "category_tiering_independent_per_dataset": True,

            "category_tiering_relative_within_category": True,

            "reject_near_zero": True,

            "reject_weak_separation": True,

            "reject_same_side_or_mixed": True,

            "below_p25_retained": True,

            "category_specific_outputs_saved": True,

            "model_refitted": False,

            "shap_recalculated": False,
        }
    ]
)


nb4_stability_methodology_audit.to_csv(
    NOTEBOOK4_COMBINED_STABILITY_DIR
    /
    "shap_stability_methodology_audit.csv",
    index=False,
)



print(
    "Shared 2,495-feature category universe:"
)


display(
    nb4_feature_category_counts
)



print(
    "\n=================================================="
)


print(
    "MIMIC-IV SHAP STABILITY RESULTS"
)


print(
    "=================================================="
)


print(
    "\nMIMIC 81-grid thresholds:"
)


display(
    nb4_mimic_stability_result[
        "thresholds_df"
    ].round(
        6
    )
)


print(
    "\nMIMIC category tier thresholds:"
)


display(
    nb4_mimic_tier_thresholds_df.round(
        4
    )
)


print(
    "\nMIMIC category selection summary:"
)


display(
    nb4_mimic_category_selection_summary.round(
        4
    )
)


print(
    "\nMIMIC category × tier counts:"
)


display(
    nb4_mimic_category_tier_summary
)



for category_name in (
    NB4_CATEGORY_ORDER
):

    category_df = (
        nb4_mimic_tiered_tables_by_category[
            category_name
        ]
    )


    print(
        "\nMIMIC-IV —",
        category_name,
        f"— selected features: {len(category_df):,}",
    )


    display(
        category_df[
            [
                "category_rank_by_stability",
                "global_rank",
                "feature_clean",
                "feature",
                "importance_tag",
                "times_selected",
                "total_combinations",
                "stability_pct",
                "mean_abs_shap",
                "dominant_abs_shap",
                "abs_high_low_separation",
                "mean_shap_high_value",
                "mean_shap_low_value",
                "n_high",
                "n_low",
                "most_common_pattern_code",
            ]
        ]
        .round(
            5
        )
    )



print(
    "\n=================================================="
)


print(
    "MC-MED SHAP STABILITY RESULTS"
)


print(
    "=================================================="
)


print(
    "\nMC-MED 81-grid thresholds:"
)


display(
    nb4_mcmed_stability_result[
        "thresholds_df"
    ].round(
        6
    )
)


print(
    "\nMC-MED category tier thresholds:"
)


display(
    nb4_mcmed_tier_thresholds_df.round(
        4
    )
)


print(
    "\nMC-MED category selection summary:"
)


display(
    nb4_mcmed_category_selection_summary.round(
        4
    )
)


print(
    "\nMC-MED category × tier counts:"
)


display(
    nb4_mcmed_category_tier_summary
)



for category_name in (
    NB4_CATEGORY_ORDER
):

    category_df = (
        nb4_mcmed_tiered_tables_by_category[
            category_name
        ]
    )


    print(
        "\nMC-MED —",
        category_name,
        f"— selected features: {len(category_df):,}",
    )


    display(
        category_df[
            [
                "category_rank_by_stability",
                "global_rank",
                "feature_clean",
                "feature",
                "importance_tag",
                "times_selected",
                "total_combinations",
                "stability_pct",
                "mean_abs_shap",
                "dominant_abs_shap",
                "abs_high_low_separation",
                "mean_shap_high_value",
                "mean_shap_low_value",
                "n_high",
                "n_low",
                "most_common_pattern_code",
            ]
        ]
        .round(
            5
        )
    )



assert (
    nb4_mimic_stability_result[
        "total_combinations"
    ]
    ==
    81
)


assert (
    nb4_mcmed_stability_result[
        "total_combinations"
    ]
    ==
    81
)


assert (
    len(
        nb4_mimic_all_stability_features
    )
    ==
    2495
)


assert (
    len(
        nb4_mcmed_all_stability_features
    )
    ==
    2495
)



print(
    "\n=================================================="
)


print(
    "81-COMBINATION SHAP STABILITY ANALYSIS COMPLETE"
)


print(
    "=================================================="
)


print(
    "\nFEATURE-RANK DIMENSION"
)


print(
    "Total N:",
    NB4_FEATURE_N_LOOSE,
)


print(
    "ceil(2N/3):",
    NB4_FEATURE_N_MEDIUM,
)


print(
    "ceil(N/3):",
    NB4_FEATURE_N_STRICT,
)


print(
    "\nNOT top 90/60/30:",
    "VERIFIED",
)


print(
    "\nROBUSTNESS DIMENSIONS"
)


print(
    "Distance from zero:",
    "P25 / P50 / P75",
)


print(
    "High-low separation:",
    "P25 / P50 / P75",
)


print(
    "Support:",
    "0.5% / 1% / 2.5%, minimum 5",
)


print(
    "Feature rank:",
    "N / ceil(2N/3) / ceil(N/3)",
)


print(
    "Combinations per dataset:",
    81,
)


print(
    "\nDATASET HANDLING"
)


print(
    "MIMIC analyzed independently:",
    "YES",
)


print(
    "MC-MED analyzed independently:",
    "YES",
)


print(
    "MIMIC category outputs stored separately:",
    "YES",
)


print(
    "MC-MED category outputs stored separately:",
    "YES",
)


print(
    "\nCATEGORY OUTPUTS"
)


for category_name in (
    NB4_CATEGORY_ORDER
):

    print(
        " ",
        category_name,
    )


print(
    "\nNear-zero effects rejected:",
    "YES",
)


print(
    "Weak separation rejected:",
    "YES",
)


print(
    "Same-side/mixed effects rejected:",
    "YES",
)


print(
    "Below-P25 selected features retained:",
    "YES",
)


print(
    "\nPositive SHAP:",
    "toward ADMITTED",
)


print(
    "Negative SHAP:",
    "toward HOME",
)


print(
    "\nModel refitted:",
    "NO",
)


print(
    "SHAP recalculated:",
    "NO",
)


print(
    "\nRuntime seconds:",
    round(
        nb4_stability_elapsed_seconds,
        3,
    ),
)


print(
    "\nREADY FOR CROSS-DATASET CATEGORY COMPARISON:",
    "YES",
)


print(
    "READY FOR LATER CLINICIAN CATEGORY COMPARISON:",
    "YES",
)

## Cross-dataset overlap of robust SHAP features

Robust SHAP features selected by the 81-combination stability analysis are compared between MIMIC-IV and MC-MED.

All features selected at least once are included, including features assigned the category-relative `Below P25` stability tag.

For each clinical feature category, features are classified as:

- selected in both MIMIC-IV and MC-MED
- selected only in MIMIC-IV
- selected only in MC-MED

For features selected in both datasets, the analysis retains the dataset-specific stability percentage, stability tag, global SHAP rank, mean absolute SHAP, dominant SHAP magnitude, high-low SHAP separation and directional pattern.

A conservative cross-dataset stability measure is additionally reported as the minimum of the two dataset-specific stability percentages. This value is used only for ranking replicated features and does not create a new stability tier.

Features selected in both datasets represent stronger evidence of cross-dataset SHAP robustness and transportability. Dataset-specific selections remain important findings because differences may reflect population, practice-pattern, coding or measurement differences between MIMIC-IV and MC-MED.

All overall and category-specific overlap tables are archived for subsequent clinician-versus-model comparison.

In [ ]:



required_cross_dataset_objects = [
    "PROJECT_ROOT",
    "NB4_CATEGORY_ORDER",
    "nb4_mimic_tiered_feature_table",
    "nb4_mcmed_tiered_feature_table",
]


missing_cross_dataset_objects = [
    object_name
    for object_name
    in required_cross_dataset_objects
    if object_name not in globals()
]


if missing_cross_dataset_objects:

    raise RuntimeError(
        "Run the 81-combination SHAP stability cell first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_cross_dataset_objects
        )
    )



assert (
    nb4_mimic_tiered_feature_table[
        "times_selected"
    ]
    >
    0
).all()


assert (
    nb4_mcmed_tiered_feature_table[
        "times_selected"
    ]
    >
    0
).all()


assert not (
    nb4_mimic_tiered_feature_table[
        "importance_tag"
    ]
    ==
    "Not selected"
).any()


assert not (
    nb4_mcmed_tiered_feature_table[
        "importance_tag"
    ]
    ==
    "Not selected"
).any()



cross_dataset_evidence_columns = [
    "category",
    "feature",
    "feature_clean",
    "global_rank",
    "importance_tag",
    "times_selected",
    "total_combinations",
    "stability_pct",
    "mean_abs_shap",
    "dominant_abs_shap",
    "abs_high_low_separation",
    "mean_shap_high_value",
    "mean_shap_low_value",
    "high_value_group",
    "low_value_group",
    "n_high",
    "n_low",
    "most_common_pattern_code",
    "most_common_pattern",
]


available_mimic_columns = [
    column_name
    for column_name
    in cross_dataset_evidence_columns
    if column_name
    in nb4_mimic_tiered_feature_table.columns
]


available_mcmed_columns = [
    column_name
    for column_name
    in cross_dataset_evidence_columns
    if column_name
    in nb4_mcmed_tiered_feature_table.columns
]



mimic_overlap_source = (
    nb4_mimic_tiered_feature_table[
        available_mimic_columns
    ]
    .copy()
)


mimic_rename_map = {
    column_name: (
        column_name
        if column_name == "feature"
        else f"mimic_{column_name}"
    )
    for column_name
    in mimic_overlap_source.columns
}


mimic_overlap_source = (
    mimic_overlap_source
    .rename(
        columns=mimic_rename_map
    )
)


mimic_overlap_source[
    "mimic_selected"
] = True


assert "feature" in (
    mimic_overlap_source.columns
)


assert "mimic_selected" in (
    mimic_overlap_source.columns
)


assert "mimic_category" in (
    mimic_overlap_source.columns
)


assert "mimic_feature_clean" in (
    mimic_overlap_source.columns
)


assert "mimic_stability_pct" in (
    mimic_overlap_source.columns
)


assert "mimic_importance_tag" in (
    mimic_overlap_source.columns
)


assert mimic_overlap_source[
    "feature"
].is_unique



mcmed_overlap_source = (
    nb4_mcmed_tiered_feature_table[
        available_mcmed_columns
    ]
    .copy()
)


mcmed_rename_map = {
    column_name: (
        column_name
        if column_name == "feature"
        else f"mcmed_{column_name}"
    )
    for column_name
    in mcmed_overlap_source.columns
}


mcmed_overlap_source = (
    mcmed_overlap_source
    .rename(
        columns=mcmed_rename_map
    )
)


mcmed_overlap_source[
    "mcmed_selected"
] = True


assert "feature" in (
    mcmed_overlap_source.columns
)


assert "mcmed_selected" in (
    mcmed_overlap_source.columns
)


assert "mcmed_category" in (
    mcmed_overlap_source.columns
)


assert "mcmed_feature_clean" in (
    mcmed_overlap_source.columns
)


assert "mcmed_stability_pct" in (
    mcmed_overlap_source.columns
)


assert "mcmed_importance_tag" in (
    mcmed_overlap_source.columns
)


assert mcmed_overlap_source[
    "feature"
].is_unique



print(
    "MIMIC overlap source:",
    mimic_overlap_source.shape,
)


print(
    "MC-MED overlap source:",
    mcmed_overlap_source.shape,
)


print(
    "\nRequired selection flags:"
)


print(
    "mimic_selected:",
    (
        "YES"
        if "mimic_selected"
        in mimic_overlap_source.columns
        else "NO"
    ),
)


print(
    "mcmed_selected:",
    (
        "YES"
        if "mcmed_selected"
        in mcmed_overlap_source.columns
        else "NO"
    ),
)


print(
    "\nReady for outer cross-dataset merge:",
    "YES",
)


nb4_cross_dataset_feature_overlap = (
    mimic_overlap_source
    .merge(
        mcmed_overlap_source,
        on="feature",
        how="outer",
        validate="one_to_one",
    )
)


nb4_cross_dataset_feature_overlap[
    "mimic_selected"
] = (
    nb4_cross_dataset_feature_overlap[
        "mimic_selected"
    ]
    .fillna(
        False
    )
    .astype(bool)
)


nb4_cross_dataset_feature_overlap[
    "mcmed_selected"
] = (
    nb4_cross_dataset_feature_overlap[
        "mcmed_selected"
    ]
    .fillna(
        False
    )
    .astype(bool)
)



nb4_cross_dataset_feature_overlap[
    "category"
] = (
    nb4_cross_dataset_feature_overlap[
        "mimic_category"
    ]
    .combine_first(
        nb4_cross_dataset_feature_overlap[
            "mcmed_category"
        ]
    )
)


nb4_cross_dataset_feature_overlap[
    "feature_clean"
] = (
    nb4_cross_dataset_feature_overlap[
        "mimic_feature_clean"
    ]
    .combine_first(
        nb4_cross_dataset_feature_overlap[
            "mcmed_feature_clean"
        ]
    )
)


both_category_available = (
    nb4_cross_dataset_feature_overlap[
        "mimic_category"
    ].notna()
    &
    nb4_cross_dataset_feature_overlap[
        "mcmed_category"
    ].notna()
)


assert (
    nb4_cross_dataset_feature_overlap.loc[
        both_category_available,
        "mimic_category",
    ]
    .astype(str)
    .to_numpy()
    ==
    nb4_cross_dataset_feature_overlap.loc[
        both_category_available,
        "mcmed_category",
    ]
    .astype(str)
    .to_numpy()
).all()



nb4_cross_dataset_feature_overlap[
    "selection_overlap"
] = np.select(
    [
        (
            nb4_cross_dataset_feature_overlap[
                "mimic_selected"
            ]
            &
            nb4_cross_dataset_feature_overlap[
                "mcmed_selected"
            ]
        ),

        (
            nb4_cross_dataset_feature_overlap[
                "mimic_selected"
            ]
            &
            ~nb4_cross_dataset_feature_overlap[
                "mcmed_selected"
            ]
        ),

        (
            ~nb4_cross_dataset_feature_overlap[
                "mimic_selected"
            ]
            &
            nb4_cross_dataset_feature_overlap[
                "mcmed_selected"
            ]
        ),
    ],

    [
        "BOTH",
        "MIMIC-IV only",
        "MC-MED only",
    ],

    default="ERROR",
)


assert not (
    nb4_cross_dataset_feature_overlap[
        "selection_overlap"
    ]
    ==
    "ERROR"
).any()



both_selected_mask = (
    nb4_cross_dataset_feature_overlap[
        "selection_overlap"
    ]
    ==
    "BOTH"
)


nb4_cross_dataset_feature_overlap[
    "cross_dataset_min_stability_pct"
] = np.nan


nb4_cross_dataset_feature_overlap[
    "cross_dataset_mean_stability_pct"
] = np.nan


nb4_cross_dataset_feature_overlap.loc[
    both_selected_mask,
    "cross_dataset_min_stability_pct",
] = (
    nb4_cross_dataset_feature_overlap.loc[
        both_selected_mask,
        [
            "mimic_stability_pct",
            "mcmed_stability_pct",
        ],
    ]
    .min(
        axis=1
    )
)


nb4_cross_dataset_feature_overlap.loc[
    both_selected_mask,
    "cross_dataset_mean_stability_pct",
] = (
    nb4_cross_dataset_feature_overlap.loc[
        both_selected_mask,
        [
            "mimic_stability_pct",
            "mcmed_stability_pct",
        ],
    ]
    .mean(
        axis=1
    )
)



nb4_cross_dataset_feature_overlap[
    "same_stability_tag"
] = False


nb4_cross_dataset_feature_overlap.loc[
    both_selected_mask,
    "same_stability_tag",
] = (
    nb4_cross_dataset_feature_overlap.loc[
        both_selected_mask,
        "mimic_importance_tag",
    ]
    .astype(str)
    .to_numpy()
    ==
    nb4_cross_dataset_feature_overlap.loc[
        both_selected_mask,
        "mcmed_importance_tag",
    ]
    .astype(str)
    .to_numpy()
)



nb4_cross_dataset_feature_overlap[
    "same_direction_pattern"
] = False


pattern_columns_available = (
    "mimic_most_common_pattern_code"
    in
    nb4_cross_dataset_feature_overlap.columns
    and
    "mcmed_most_common_pattern_code"
    in
    nb4_cross_dataset_feature_overlap.columns
)


if pattern_columns_available:

    direction_comparable_mask = (
        both_selected_mask
        &
        nb4_cross_dataset_feature_overlap[
            "mimic_most_common_pattern_code"
        ].notna()
        &
        nb4_cross_dataset_feature_overlap[
            "mcmed_most_common_pattern_code"
        ].notna()
    )


    nb4_cross_dataset_feature_overlap.loc[
        direction_comparable_mask,
        "same_direction_pattern",
    ] = (
        nb4_cross_dataset_feature_overlap.loc[
            direction_comparable_mask,
            "mimic_most_common_pattern_code",
        ]
        .astype(str)
        .to_numpy()
        ==
        nb4_cross_dataset_feature_overlap.loc[
            direction_comparable_mask,
            "mcmed_most_common_pattern_code",
        ]
        .astype(str)
        .to_numpy()
    )



nb4_features_selected_in_both = (
    nb4_cross_dataset_feature_overlap.loc[
        nb4_cross_dataset_feature_overlap[
            "selection_overlap"
        ]
        ==
        "BOTH"
    ]
    .copy()
)


nb4_features_selected_mimic_only = (
    nb4_cross_dataset_feature_overlap.loc[
        nb4_cross_dataset_feature_overlap[
            "selection_overlap"
        ]
        ==
        "MIMIC-IV only"
    ]
    .copy()
)


nb4_features_selected_mcmed_only = (
    nb4_cross_dataset_feature_overlap.loc[
        nb4_cross_dataset_feature_overlap[
            "selection_overlap"
        ]
        ==
        "MC-MED only"
    ]
    .copy()
)



nb4_features_selected_in_both = (
    nb4_features_selected_in_both
    .sort_values(
        [
            "category",
            "cross_dataset_min_stability_pct",
            "cross_dataset_mean_stability_pct",
            "mimic_stability_pct",
            "mcmed_stability_pct",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


nb4_features_selected_mimic_only = (
    nb4_features_selected_mimic_only
    .sort_values(
        [
            "category",
            "mimic_stability_pct",
            "mimic_mean_abs_shap",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


nb4_features_selected_mcmed_only = (
    nb4_features_selected_mcmed_only
    .sort_values(
        [
            "category",
            "mcmed_stability_pct",
            "mcmed_mean_abs_shap",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



nb4_cross_dataset_overlap_summary_by_category = (
    nb4_cross_dataset_feature_overlap
    .groupby(
        [
            "category",
            "selection_overlap",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        NB4_CATEGORY_ORDER
    )
    .reset_index()
)


for required_column in [
    "BOTH",
    "MIMIC-IV only",
    "MC-MED only",
]:

    if required_column not in (
        nb4_cross_dataset_overlap_summary_by_category.columns
    ):

        nb4_cross_dataset_overlap_summary_by_category[
            required_column
        ] = 0


nb4_cross_dataset_overlap_summary_by_category[
    "selected_union"
] = (
    nb4_cross_dataset_overlap_summary_by_category[
        "BOTH"
    ]
    +
    nb4_cross_dataset_overlap_summary_by_category[
        "MIMIC-IV only"
    ]
    +
    nb4_cross_dataset_overlap_summary_by_category[
        "MC-MED only"
    ]
)


nb4_cross_dataset_overlap_summary_by_category[
    "jaccard_overlap"
] = np.divide(
    nb4_cross_dataset_overlap_summary_by_category[
        "BOTH"
    ],
    nb4_cross_dataset_overlap_summary_by_category[
        "selected_union"
    ],
    out=np.zeros(
        len(
            nb4_cross_dataset_overlap_summary_by_category
        ),
        dtype=float,
    ),
    where=(
        nb4_cross_dataset_overlap_summary_by_category[
            "selected_union"
        ]
        >
        0
    ),
)


nb4_cross_dataset_overlap_summary_by_category[
    "jaccard_overlap_pct"
] = (
    100
    *
    nb4_cross_dataset_overlap_summary_by_category[
        "jaccard_overlap"
    ]
)



nb4_cross_dataset_overlap_summary_overall = (
    nb4_cross_dataset_feature_overlap[
        "selection_overlap"
    ]
    .value_counts()
    .reindex(
        [
            "BOTH",
            "MIMIC-IV only",
            "MC-MED only",
        ],
        fill_value=0,
    )
    .rename_axis(
        "selection_overlap"
    )
    .reset_index(
        name="n_features"
    )
)


overall_both = int(
    (
        nb4_cross_dataset_feature_overlap[
            "selection_overlap"
        ]
        ==
        "BOTH"
    ).sum()
)


overall_union = int(
    len(
        nb4_cross_dataset_feature_overlap
    )
)


overall_jaccard = (
    overall_both
    /
    overall_union
    if overall_union > 0
    else np.nan
)



nb4_cross_dataset_tables_by_category = {}

nb4_both_tables_by_category = {}

nb4_mimic_only_tables_by_category = {}

nb4_mcmed_only_tables_by_category = {}


for category_name in (
    NB4_CATEGORY_ORDER
):

    category_all = (
        nb4_cross_dataset_feature_overlap.loc[
            nb4_cross_dataset_feature_overlap[
                "category"
            ]
            .astype(str)
            .eq(
                category_name
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    category_both = (
        nb4_features_selected_in_both.loc[
            nb4_features_selected_in_both[
                "category"
            ]
            .astype(str)
            .eq(
                category_name
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    category_mimic_only = (
        nb4_features_selected_mimic_only.loc[
            nb4_features_selected_mimic_only[
                "category"
            ]
            .astype(str)
            .eq(
                category_name
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    category_mcmed_only = (
        nb4_features_selected_mcmed_only.loc[
            nb4_features_selected_mcmed_only[
                "category"
            ]
            .astype(str)
            .eq(
                category_name
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    nb4_cross_dataset_tables_by_category[
        category_name
    ] = category_all


    nb4_both_tables_by_category[
        category_name
    ] = category_both


    nb4_mimic_only_tables_by_category[
        category_name
    ] = category_mimic_only


    nb4_mcmed_only_tables_by_category[
        category_name
    ] = category_mcmed_only



nb4_both_initial_context_df = (
    nb4_both_tables_by_category[
        "Initial context"
    ]
)


nb4_both_triage_vitals_pain_df = (
    nb4_both_tables_by_category[
        "Triage vitals / pain"
    ]
)


nb4_both_lab_result_df = (
    nb4_both_tables_by_category[
        "Lab result"
    ]
)


nb4_both_imaging_radiology_df = (
    nb4_both_tables_by_category[
        "Imaging / radiology"
    ]
)


nb4_both_pmh_icd_title_df = (
    nb4_both_tables_by_category[
        "PMH ICD title"
    ]
)


nb4_both_pmh_med_class_df = (
    nb4_both_tables_by_category[
        "PMH medication class"
    ]
)


nb4_both_pmh_med_name_df = (
    nb4_both_tables_by_category[
        "PMH medication name"
    ]
)



NOTEBOOK4_CROSS_DATASET_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook4"
    / "cross_dataset_stability"
)


NOTEBOOK4_CROSS_DATASET_CATEGORY_DIR = (
    NOTEBOOK4_CROSS_DATASET_DIR
    / "categories"
)


NOTEBOOK4_CROSS_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


NOTEBOOK4_CROSS_DATASET_CATEGORY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)



nb4_cross_dataset_feature_overlap.to_csv(
    NOTEBOOK4_CROSS_DATASET_DIR
    /
    "all_selected_feature_overlap.csv",
    index=False,
)


nb4_cross_dataset_feature_overlap.to_parquet(
    NOTEBOOK4_CROSS_DATASET_DIR
    /
    "all_selected_feature_overlap.parquet",
    index=False,
    engine="pyarrow",
)


nb4_features_selected_in_both.to_csv(
    NOTEBOOK4_CROSS_DATASET_DIR
    /
    "features_selected_in_both_datasets.csv",
    index=False,
)


nb4_features_selected_mimic_only.to_csv(
    NOTEBOOK4_CROSS_DATASET_DIR
    /
    "features_selected_mimic_only.csv",
    index=False,
)


nb4_features_selected_mcmed_only.to_csv(
    NOTEBOOK4_CROSS_DATASET_DIR
    /
    "features_selected_mcmed_only.csv",
    index=False,
)


nb4_cross_dataset_overlap_summary_by_category.to_csv(
    NOTEBOOK4_CROSS_DATASET_DIR
    /
    "cross_dataset_overlap_summary_by_category.csv",
    index=False,
)


nb4_cross_dataset_overlap_summary_overall.to_csv(
    NOTEBOOK4_CROSS_DATASET_DIR
    /
    "cross_dataset_overlap_summary_overall.csv",
    index=False,
)



def nb4_safe_category_filename(
    category_name,
):

    return (
        category_name
        .lower()
        .replace(
            " / ",
            "_"
        )
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )


for category_name in (
    NB4_CATEGORY_ORDER
):

    safe_category = (
        nb4_safe_category_filename(
            category_name
        )
    )


    nb4_cross_dataset_tables_by_category[
        category_name
    ].to_csv(
        NOTEBOOK4_CROSS_DATASET_CATEGORY_DIR
        /
        f"{safe_category}_all_selected_overlap.csv",
        index=False,
    )


    nb4_both_tables_by_category[
        category_name
    ].to_csv(
        NOTEBOOK4_CROSS_DATASET_CATEGORY_DIR
        /
        f"{safe_category}_selected_in_both.csv",
        index=False,
    )


    nb4_mimic_only_tables_by_category[
        category_name
    ].to_csv(
        NOTEBOOK4_CROSS_DATASET_CATEGORY_DIR
        /
        f"{safe_category}_mimic_only.csv",
        index=False,
    )


    nb4_mcmed_only_tables_by_category[
        category_name
    ].to_csv(
        NOTEBOOK4_CROSS_DATASET_CATEGORY_DIR
        /
        f"{safe_category}_mcmed_only.csv",
        index=False,
    )



print(
    "Cross-dataset stable-feature overlap summary:"
)


display(
    nb4_cross_dataset_overlap_summary_by_category.round(
        2
    )
)


print(
    "\nOverall:"
)


display(
    nb4_cross_dataset_overlap_summary_overall
)


print(
    "\nOverall Jaccard overlap:",
    round(
        100
        *
        overall_jaccard,
        2,
    ),
    "%",
)



print(
    "\n=================================================="
)


print(
    "FEATURES SELECTED IN BOTH DATASETS"
)


print(
    "Includes High / Medium / Low / Below P25"
)


print(
    "=================================================="
)


for category_name in (
    NB4_CATEGORY_ORDER
):

    category_both_df = (
        nb4_both_tables_by_category[
            category_name
        ]
    )


    print(
        "\n",
        category_name,
        "— BOTH datasets:",
        len(
            category_both_df
        ),
    )


    if category_both_df.empty:

        print(
            "No common selected features."
        )


    else:

        display(
            category_both_df[
                [
                    "feature_clean",
                    "feature",
                    "mimic_importance_tag",
                    "mimic_stability_pct",
                    "mcmed_importance_tag",
                    "mcmed_stability_pct",
                    "cross_dataset_min_stability_pct",
                    "cross_dataset_mean_stability_pct",
                    "same_stability_tag",
                    "mimic_most_common_pattern_code",
                    "mcmed_most_common_pattern_code",
                    "same_direction_pattern",
                ]
            ]
            .round(
                3
            )
        )



print(
    "\n=================================================="
)


print(
    "FEATURES SELECTED IN ONLY ONE DATASET"
)


print(
    "=================================================="
)


for category_name in (
    NB4_CATEGORY_ORDER
):

    mimic_only_df = (
        nb4_mimic_only_tables_by_category[
            category_name
        ]
    )


    mcmed_only_df = (
        nb4_mcmed_only_tables_by_category[
            category_name
        ]
    )


    print(
        "\n",
        category_name,
    )


    print(
        "MIMIC-IV only:",
        len(
            mimic_only_df
        ),
    )


    if not mimic_only_df.empty:

        display(
            mimic_only_df[
                [
                    "feature_clean",
                    "feature",
                    "mimic_importance_tag",
                    "mimic_stability_pct",
                    "mimic_global_rank",
                    "mimic_mean_abs_shap",
                    "mimic_most_common_pattern_code",
                ]
            ]
            .round(
                3
            )
        )


    print(
        "MC-MED only:",
        len(
            mcmed_only_df
        ),
    )


    if not mcmed_only_df.empty:

        display(
            mcmed_only_df[
                [
                    "feature_clean",
                    "feature",
                    "mcmed_importance_tag",
                    "mcmed_stability_pct",
                    "mcmed_global_rank",
                    "mcmed_mean_abs_shap",
                    "mcmed_most_common_pattern_code",
                ]
            ]
            .round(
                3
            )
        )



assert (
    len(
        nb4_features_selected_in_both
    )
    +
    len(
        nb4_features_selected_mimic_only
    )
    +
    len(
        nb4_features_selected_mcmed_only
    )
    ==
    len(
        nb4_cross_dataset_feature_overlap
    )
)


print(
    "\n=================================================="
)


print(
    "CROSS-DATASET STABLE-FEATURE COMPARISON COMPLETE"
)


print(
    "=================================================="
)


print(
    "\nSelected in BOTH datasets:",
    len(
        nb4_features_selected_in_both
    ),
)


print(
    "Selected only in MIMIC-IV:",
    len(
        nb4_features_selected_mimic_only
    ),
)


print(
    "Selected only in MC-MED:",
    len(
        nb4_features_selected_mcmed_only
    ),
)


print(
    "\nBelow-P25 features included:",
    "YES",
)


print(
    "Category-wise comparison:",
    "YES",
)


print(
    "Exact directional-pattern agreement reported:",
    "YES",
)


print(
    "\nImportant replicated features saved to disk:",
    "YES",
)


print(
    "Dataset-specific selected features saved to disk:",
    "YES",
)


print(
    "Category-specific tables saved to disk:",
    "YES",
)


print(
    "\nOutput directory:"
)


print(
    NOTEBOOK4_CROSS_DATASET_DIR
)


print(
    "\nREADY FOR CLINICIAN-VERSUS-MODEL COMPARISON:",
    "YES",
)

## Ten-clinician consensus parsing and feature-universe validation

The cleaned clinician-review Excel workbook is used as the authoritative source for external clinical validation.

The workbook contains clinician consensus at the clinical-group level for PMH Medication, PMH Diagnosis, Laboratory and Imaging features. Consensus values may be stored either as numeric proportions (for example 0.7, 0.5, 0 or 1) or as percentage-formatted/text values (for example 70%, 50% or 30%). All values are normalized to a proportion between 0 and 1 and converted to the equivalent number of clinicians out of 10.

The nine S0 features are not contained in the workbook. Based on the documented clinician review, all three Initial Context features and all six Triage Vital features are added explicitly as unanimous 10/10 clinician-important features. These additions are retained separately in the audit trail rather than represented as if they originated from the Excel workbook.

Because some technical features belong to more than one clinician-defined group, group-feature memberships are preserved before deriving a unique feature-level consensus table. For features with multiple group memberships, the maximum clinician vote across applicable groups is retained as the conservative primary feature-level clinician support, while all memberships and their individual votes remain available.

The parsing stage creates and freezes:

- clinician group consensus
- clinician group-feature memberships
- manual S0 consensus additions
- unique feature-level clinician consensus
- clinician vote-distribution summaries
- source-file checksum and parsing audit
- exact alignment with the locked 2,495-feature model universe

No clinician-versus-model comparison is performed in this section.

In [ ]:



required_clinician_parser_objects = [
    "PROJECT_ROOT",
    "LOCKED_RAW_FEATURE_ORDER",
    "clinician_feature_universe",
    "notebook4_file_sha256",
]


missing_clinician_parser_objects = [
    object_name
    for object_name
    in required_clinician_parser_objects
    if object_name not in globals()
]


if missing_clinician_parser_objects:

    raise RuntimeError(
        "Run the Notebook 4 frozen hand-off loader first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_clinician_parser_objects
        )
    )


if "re" not in globals():

    raise RuntimeError(
        "The Python 're' module is not loaded.\n"
        "Add 'import re' to the centralized Notebook 4 "
        "imports cell and rerun that imports cell."
    )


assert len(
    LOCKED_RAW_FEATURE_ORDER
) == 2495


assert len(
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
) == 2495



CLINICIAN_REVIEW_PATH = (
    PROJECT_ROOT
    / "data"
    / "clinical_validation"
    / "raw"
    / "external_clinician_feature_group_review_10_clinicians.xlsx"
)


if not CLINICIAN_REVIEW_PATH.exists():

    clinical_source_directory = (
        CLINICIAN_REVIEW_PATH.parent
    )


    available_excel_files = (
        sorted(
            clinical_source_directory.glob(
                "*.xlsx"
            )
        )
        if clinical_source_directory.exists()
        else []
    )


    available_text = (
        "\n".join(
            str(path)
            for path
            in available_excel_files
        )
        if available_excel_files
        else "No .xlsx files found."
    )


    raise FileNotFoundError(
        "Canonical clinician-review Excel file not found:\n\n"
        f"{CLINICIAN_REVIEW_PATH}\n\n"
        "Available Excel files in the directory:\n"
        f"{available_text}\n\n"
        "Rename the frozen analysis copy to:\n"
        "external_clinician_feature_group_review_10_clinicians.xlsx"
    )



CLINICIAN_REVIEW_SHEET = (
    "Clinician Review"
)


N_CLINICIANS = 10


EXCEL_EXPECTED_GROUPS = 73


EXCEL_EXPECTED_FEATURE_MEMBERSHIPS = 2493


EXCEL_EXPECTED_UNIQUE_FEATURES = 2486


MANUAL_S0_EXPECTED_FEATURES = 9


COMBINED_EXPECTED_UNIQUE_FEATURES = 2495


KNOWN_EXCEL_DOMAINS = {
    "MEDICATION": "PMH Medication",
    "DIAGNOSIS": "PMH Diagnosis",
    "LABORATORY": "Laboratory",
    "IMAGING": "Imaging",
}


EXPECTED_EXCEL_DOMAIN_MEMBERSHIPS = {
    "MEDICATION": 1178,
    "DIAGNOSIS": 881,
    "LABORATORY": 351,
    "IMAGING": 83,
}


CLINICIAN_IMAGING_MODEL_PREFIX = (
    "poe_result__imaging_radiology__"
)



def clinician_clean_text(
    value,
):

    if pd.isna(
        value
    ):

        return ""


    value = str(
        value
    )


    value = (
        value
        .replace(
            "\xa0",
            " "
        )
        .replace(
            "\r",
            " "
        )
        .replace(
            "\n",
            " "
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


def clinician_clean_feature_code(
    value,
):

    if pd.isna(
        value
    ):

        return ""


    value = str(
        value
    )


    value = (
        value
        .replace(
            "\xa0",
            ""
        )
        .replace(
            "\r",
            ""
        )
        .replace(
            "\n",
            ""
        )
    )


    value = re.sub(
        r"\s+",
        "",
        value,
    )


    return value.strip()


def clinician_clean_feature_label(
    value,
):

    value = clinician_clean_text(
        value
    )


    value = re.sub(
        r"^[↳→]+\s*",
        "",
        value,
    )


    return value.strip()



locked_feature_set = set(
    LOCKED_RAW_FEATURE_ORDER
)


assert len(
    locked_feature_set
) == 2495


locked_feature_family_lookup = (
    clinician_feature_universe[
        [
            "source_raw_feature",
            "family",
        ]
    ]
    .drop_duplicates(
        subset=[
            "source_raw_feature",
        ]
    )
    .set_index(
        "source_raw_feature"
    )[
        "family"
    ]
    .astype(str)
    .to_dict()
)


assert len(
    locked_feature_family_lookup
) == 2495


assert set(
    locked_feature_family_lookup
) == locked_feature_set



CLINICIAN_EXPLICIT_FEATURE_ALIASES = {
    (
        "IMAGING",
        "ct_abd_w_contrast",
    ): (
        "poe_result__imaging_radiology__"
        "ct_abdomen_w_contrast"
    ),
}


for (
    alias_domain,
    alias_source_feature,
), alias_target_feature in (
    CLINICIAN_EXPLICIT_FEATURE_ALIASES.items()
):

    assert alias_target_feature in locked_feature_set, (
        "Explicit clinician feature alias target is not "
        "present in the frozen model feature universe:\n"
        f"{alias_domain}: "
        f"{alias_source_feature} -> {alias_target_feature}"
    )


clinician_explicit_feature_aliases_df = pd.DataFrame(
    [
        {
            "clinician_domain_code": alias_domain,
            "excel_technical_feature": alias_source_feature,
            "canonical_model_feature": alias_target_feature,
            "mapping_method": "documented_exact_alias",
            "fuzzy_matching_used": False,
        }
        for (
            alias_domain,
            alias_source_feature,
        ), alias_target_feature in (
            CLINICIAN_EXPLICIT_FEATURE_ALIASES.items()
        )
    ]
)



def clinician_model_category_from_feature(
    feature_code,
):

    feature_code = (
        clinician_clean_feature_code(
            feature_code
        )
    )


    if feature_code not in (
        locked_feature_set
    ):

        return None


    family = (
        locked_feature_family_lookup[
            feature_code
        ]
    )


    if family == "S1_medication_name":

        return "PMH medication name"


    if family == "S1_medication_class":

        return "PMH medication class"


    if family == "S1_diagnosis":

        return "PMH ICD title"


    if family == "S2_lab_result":

        return "Lab result"


    if family == "S2_imaging_result":

        return "Imaging / radiology"


    if family == "S0":

        if feature_code in {
            "age",
            "sex_gender",
            "arrival_transport_grouped",
        }:

            return "Initial context"


        return "Triage vitals"


    raise RuntimeError(
        "Locked feature has an unexpected family:\n"
        f"Feature: {feature_code}\n"
        f"Family: {family}"
    )



def clinician_canonicalize_feature_code(
    raw_feature_code,
    clinician_domain_code,
):

    raw_feature_code = (
        clinician_clean_feature_code(
            raw_feature_code
        )
    )


    if raw_feature_code == "":

        return {
            "raw_feature_code": "",
            "canonical_feature": "",
            "mapping_method": "blank",
            "present_in_locked_model": False,
        }



    if raw_feature_code in (
        locked_feature_set
    ):

        return {
            "raw_feature_code": (
                raw_feature_code
            ),

            "canonical_feature": (
                raw_feature_code
            ),

            "mapping_method": (
                "exact_locked_feature_match"
            ),

            "present_in_locked_model": (
                True
            ),
        }



    alias_key = (
        clinician_domain_code,
        raw_feature_code,
    )


    if alias_key in (
        CLINICIAN_EXPLICIT_FEATURE_ALIASES
    ):

        alias_candidate = (
            CLINICIAN_EXPLICIT_FEATURE_ALIASES[
                alias_key
            ]
        )


        assert alias_candidate in (
            locked_feature_set
        )


        return {
            "raw_feature_code": (
                raw_feature_code
            ),

            "canonical_feature": (
                alias_candidate
            ),

            "mapping_method": (
                "documented_exact_alias"
            ),

            "present_in_locked_model": (
                True
            ),
        }



    if (
        clinician_domain_code
        ==
        "IMAGING"
    ):

        imaging_candidate = (
            CLINICIAN_IMAGING_MODEL_PREFIX
            +
            raw_feature_code
        )


        if imaging_candidate in (
            locked_feature_set
        ):

            return {
                "raw_feature_code": (
                    raw_feature_code
                ),

                "canonical_feature": (
                    imaging_candidate
                ),

                "mapping_method": (
                    "imaging_prefix_canonicalization"
                ),

                "present_in_locked_model": (
                    True
                ),
            }



    return {
        "raw_feature_code": (
            raw_feature_code
        ),

        "canonical_feature": (
            raw_feature_code
        ),

        "mapping_method": (
            "unmatched"
        ),

        "present_in_locked_model": (
            False
        ),
    }



def clinician_extract_group_number(
    group_text,
):

    group_text = clinician_clean_text(
        group_text
    )


    match = re.search(
        r"\bGroup\s+(\d+)\b",
        group_text,
        flags=re.IGNORECASE,
    )


    if match is None:

        return np.nan


    return int(
        match.group(
            1
        )
    )


def clinician_extract_group_name(
    group_text,
):

    group_text = clinician_clean_text(
        group_text
    )


    group_name = re.sub(
        r"^\s*Group\s+\d+\s*",
        "",
        group_text,
        flags=re.IGNORECASE,
    )


    group_name = re.sub(
        r"^[\s\-–—:]+",
        "",
        group_name,
    )


    return group_name.strip()



def clinician_parse_declared_feature_count(
    value,
):

    value = clinician_clean_text(
        value
    )


    match = re.search(
        r"(\d[\d,]*)\s*features?",
        value,
        flags=re.IGNORECASE,
    )


    if match is None:

        return np.nan


    return int(
        match.group(
            1
        )
        .replace(
            ",",
            ""
        )
    )



def clinician_parse_vote(
    raw_value,
):

    if pd.isna(
        raw_value
    ):

        raise ValueError(
            "Missing clinician consensus value."
        )


    parsing_mode = None



    if isinstance(
        raw_value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        numeric_value = float(
            raw_value
        )


        if (
            0.0
            <=
            numeric_value
            <=
            1.0
        ):

            vote_prop = (
                numeric_value
            )


            parsing_mode = (
                "numeric_proportion"
            )


        elif (
            1.0
            <
            numeric_value
            <=
            100.0
        ):

            vote_prop = (
                numeric_value
                /
                100.0
            )


            parsing_mode = (
                "numeric_percent_scale"
            )


        else:

            raise ValueError(
                "Numeric clinician consensus is outside "
                "the expected 0–1 or 0–100 range:\n"
                f"{raw_value!r}"
            )



    else:

        text_value = clinician_clean_text(
            raw_value
        )


        if text_value == "":

            raise ValueError(
                "Blank clinician consensus value."
            )


        percent_match = re.fullmatch(
            r"([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*%",
            text_value,
        )


        numeric_match = re.fullmatch(
            r"([+-]?(?:\d+(?:\.\d*)?|\.\d+))",
            text_value,
        )


        if percent_match is not None:

            vote_prop = (
                float(
                    percent_match.group(
                        1
                    )
                )
                /
                100.0
            )


            parsing_mode = (
                "text_percentage"
            )


        elif numeric_match is not None:

            numeric_value = float(
                numeric_match.group(
                    1
                )
            )


            if (
                0.0
                <=
                numeric_value
                <=
                1.0
            ):

                vote_prop = (
                    numeric_value
                )


                parsing_mode = (
                    "text_proportion"
                )


            elif (
                1.0
                <
                numeric_value
                <=
                100.0
            ):

                vote_prop = (
                    numeric_value
                    /
                    100.0
                )


                parsing_mode = (
                    "text_percent_scale"
                )


            else:

                raise ValueError(
                    "Text clinician consensus is outside "
                    "the expected range:\n"
                    f"{raw_value!r}"
                )


        else:

            raise ValueError(
                "Clinician consensus must be a numeric "
                "proportion or percentage.\n"
                f"Unrecognised value: {raw_value!r}"
            )



    if not (
        0.0
        <=
        vote_prop
        <=
        1.0
    ):

        raise ValueError(
            "Normalized clinician proportion is outside "
            f"0–1: {vote_prop}"
        )


    vote_n_float = (
        vote_prop
        *
        N_CLINICIANS
    )


    vote_n = int(
        round(
            vote_n_float
        )
    )


    if not np.isclose(
        vote_n_float,
        vote_n,
        atol=1e-8,
        rtol=0.0,
    ):

        raise ValueError(
            "Consensus does not correspond to a whole "
            "number of clinicians out of 10.\n"
            f"Raw value: {raw_value!r}\n"
            f"Normalized proportion: {vote_prop}\n"
            f"Equivalent votes: {vote_n_float}"
        )


    if not (
        0
        <=
        vote_n
        <=
        N_CLINICIANS
    ):

        raise ValueError(
            f"Derived clinician vote count invalid: {vote_n}"
        )


    canonical_vote_prop = (
        vote_n
        /
        N_CLINICIANS
    )


    return {
        "vote_raw": (
            raw_value
        ),

        "vote_parsing_mode": (
            parsing_mode
        ),

        "clinician_vote_n": (
            vote_n
        ),

        "clinician_vote_prop": (
            canonical_vote_prop
        ),

        "clinician_vote_pct": (
            100.0
            *
            canonical_vote_prop
        ),
    }



def clinician_consensus_band(
    vote_n,
):

    vote_n = int(
        vote_n
    )


    if vote_n == 10:

        return (
            "Unanimous important"
        )


    if vote_n >= 8:

        return (
            "Strong consensus important"
        )


    if vote_n >= 6:

        return (
            "Majority important"
        )


    if vote_n >= 4:

        return (
            "Divided / uncertain"
        )


    if vote_n >= 1:

        return (
            "Low clinician support"
        )


    return (
        "Unanimous not important"
    )



clinician_excel_raw = pd.read_excel(
    CLINICIAN_REVIEW_PATH,
    sheet_name=(
        CLINICIAN_REVIEW_SHEET
    ),
    header=None,
)


assert not clinician_excel_raw.empty



header_candidates = clinician_excel_raw.index[
    clinician_excel_raw[
        0
    ]
    .astype(str)
    .str.strip()
    .eq(
        "Domain"
    )
    &
    clinician_excel_raw[
        1
    ]
    .astype(str)
    .str.strip()
    .eq(
        "Group / Feature"
    )
].tolist()


assert len(
    header_candidates
) == 1, (
    "Expected exactly one internal clinician-review "
    "header row."
)


CLINICIAN_EXCEL_HEADER_ROW_INDEX = int(
    header_candidates[
        0
    ]
)


CLINICIAN_EXCEL_HEADER_ROW_NUMBER = (
    CLINICIAN_EXCEL_HEADER_ROW_INDEX
    +
    1
)



group_records = []

membership_records = []

unmatched_feature_candidate_records = []


current_group = None


for row_index in range(
    CLINICIAN_EXCEL_HEADER_ROW_INDEX
    +
    1,
    len(
        clinician_excel_raw
    ),
):

    row = clinician_excel_raw.iloc[
        row_index
    ]


    domain_raw = clinician_clean_text(
        row.iloc[
            0
        ]
    ).upper()


    group_or_feature_raw = clinician_clean_text(
        row.iloc[
            1
        ]
    )


    subgroup_or_count_raw = clinician_clean_text(
        row.iloc[
            3
        ]
    )


    technical_or_vote_raw = (
        row.iloc[
            4
        ]
    )


    source_excel_number_raw = (
        row.iloc[
            5
        ]
        if len(
            row
        )
        >
        5
        else np.nan
    )


    clinician_comments_raw = (
        row.iloc[
            6
        ]
        if len(
            row
        )
        >
        6
        else np.nan
    )



    is_group_row = (
        domain_raw
        in
        KNOWN_EXCEL_DOMAINS
        and
        bool(
            re.match(
                r"^\s*Group\s+\d+\b",
                group_or_feature_raw,
                flags=re.IGNORECASE,
            )
        )
    )


    if is_group_row:

        group_number = (
            clinician_extract_group_number(
                group_or_feature_raw
            )
        )


        if pd.isna(
            group_number
        ):

            raise ValueError(
                "Could not parse clinician group number.\n"
                f"Excel row: {row_index + 1}\n"
                f"Group text: {group_or_feature_raw!r}"
            )


        group_name = (
            clinician_extract_group_name(
                group_or_feature_raw
            )
        )


        declared_feature_count = (
            clinician_parse_declared_feature_count(
                subgroup_or_count_raw
            )
        )


        vote_result = (
            clinician_parse_vote(
                technical_or_vote_raw
            )
        )


        group_key = (
            f"{domain_raw}:"
            f"{int(group_number):02d}"
        )


        current_group = {
            "source_type": (
                "Excel"
            ),

            "clinician_domain_code": (
                domain_raw
            ),

            "clinician_domain": (
                KNOWN_EXCEL_DOMAINS[
                    domain_raw
                ]
            ),

            "clinician_group_key": (
                group_key
            ),

            "clinician_group_number": int(
                group_number
            ),

            "clinician_group_name": (
                group_name
            ),

            "declared_feature_count": (
                int(
                    declared_feature_count
                )
                if pd.notna(
                    declared_feature_count
                )
                else np.nan
            ),

            "vote_raw": (
                vote_result[
                    "vote_raw"
                ]
            ),

            "vote_parsing_mode": (
                vote_result[
                    "vote_parsing_mode"
                ]
            ),

            "clinician_vote_n": (
                vote_result[
                    "clinician_vote_n"
                ]
            ),

            "clinician_vote_prop": (
                vote_result[
                    "clinician_vote_prop"
                ]
            ),

            "clinician_vote_pct": (
                vote_result[
                    "clinician_vote_pct"
                ]
            ),

            "clinician_consensus_band": (
                clinician_consensus_band(
                    vote_result[
                        "clinician_vote_n"
                    ]
                )
            ),

            "clinician_majority_important": bool(
                vote_result[
                    "clinician_vote_n"
                ]
                >=
                6
            ),

            "clinician_unanimous_important": bool(
                vote_result[
                    "clinician_vote_n"
                ]
                ==
                10
            ),

            "clinician_unanimous_not_important": bool(
                vote_result[
                    "clinician_vote_n"
                ]
                ==
                0
            ),

            "source_excel_row": (
                row_index
                +
                1
            ),

            "clinician_comments": (
                clinician_clean_text(
                    clinician_comments_raw
                )
            ),
        }


        group_records.append(
            current_group.copy()
        )


        continue



    raw_feature_code = (
        clinician_clean_feature_code(
            technical_or_vote_raw
        )
    )


    if raw_feature_code == "":

        continue


    if current_group is None:

        continue



    feature_mapping = (
        clinician_canonicalize_feature_code(
            raw_feature_code=(
                raw_feature_code
            ),

            clinician_domain_code=(
                current_group[
                    "clinician_domain_code"
                ]
            ),
        )
    )


    feature_code = (
        feature_mapping[
            "canonical_feature"
        ]
    )


    feature_in_locked_model = bool(
        feature_mapping[
            "present_in_locked_model"
        ]
    )


    feature_mapping_method = (
        feature_mapping[
            "mapping_method"
        ]
    )


    model_category = (
        clinician_model_category_from_feature(
            feature_code
        )
        if feature_in_locked_model
        else None
    )


    is_feature_row = (
        feature_in_locked_model
        and
        model_category is not None
    )



    if is_feature_row:

        feature_label = (
            clinician_clean_feature_label(
                group_or_feature_raw
            )
        )


        membership_records.append(
            {
                "source_type": (
                    "Excel"
                ),

                "clinician_domain_code": (
                    current_group[
                        "clinician_domain_code"
                    ]
                ),

                "clinician_domain": (
                    current_group[
                        "clinician_domain"
                    ]
                ),

                "clinician_group_key": (
                    current_group[
                        "clinician_group_key"
                    ]
                ),

                "clinician_group_number": (
                    current_group[
                        "clinician_group_number"
                    ]
                ),

                "clinician_group_name": (
                    current_group[
                        "clinician_group_name"
                    ]
                ),

                "feature_label": (
                    feature_label
                ),

                "subgroup_label": (
                    subgroup_or_count_raw
                ),

                "excel_technical_feature": (
                    raw_feature_code
                ),

                "feature": (
                    feature_code
                ),

                "feature_key": (
                    feature_code.lower()
                ),

                "feature_mapping_method": (
                    feature_mapping_method
                ),

                "model_category": (
                    model_category
                ),

                "clinician_vote_n": (
                    current_group[
                        "clinician_vote_n"
                    ]
                ),

                "clinician_vote_prop": (
                    current_group[
                        "clinician_vote_prop"
                    ]
                ),

                "clinician_vote_pct": (
                    current_group[
                        "clinician_vote_pct"
                    ]
                ),

                "clinician_consensus_band": (
                    current_group[
                        "clinician_consensus_band"
                    ]
                ),

                "clinician_majority_important": (
                    current_group[
                        "clinician_majority_important"
                    ]
                ),

                "clinician_unanimous_important": (
                    current_group[
                        "clinician_unanimous_important"
                    ]
                ),

                "clinician_unanimous_not_important": (
                    current_group[
                        "clinician_unanimous_not_important"
                    ]
                ),

                "source_excel_number": (
                    source_excel_number_raw
                ),

                "source_excel_row": (
                    row_index
                    +
                    1
                ),
            }
        )


        continue



    looks_like_feature_row = (
        group_or_feature_raw.startswith(
            (
                "↳",
                "→",
            )
        )
        or
        "__"
        in
        raw_feature_code
        or
        (
            current_group[
                "clinician_domain_code"
            ]
            ==
            "IMAGING"
            and
            raw_feature_code
            !=
            ""
        )
    )


    if looks_like_feature_row:

        unmatched_feature_candidate_records.append(
            {
                "source_excel_row": (
                    row_index
                    +
                    1
                ),

                "clinician_domain_code": (
                    current_group[
                        "clinician_domain_code"
                    ]
                ),

                "clinician_group_key": (
                    current_group[
                        "clinician_group_key"
                    ]
                ),

                "group_or_feature_raw": (
                    group_or_feature_raw
                ),

                "excel_technical_feature": (
                    raw_feature_code
                ),

                "canonical_candidate": (
                    feature_code
                ),

                "mapping_method": (
                    feature_mapping_method
                ),

                "present_in_locked_model": (
                    feature_in_locked_model
                ),
            }
        )



clinician_excel_group_consensus_df = pd.DataFrame(
    group_records
)


clinician_excel_feature_membership_df = pd.DataFrame(
    membership_records
)


clinician_excel_unmatched_feature_candidates_df = (
    pd.DataFrame(
        unmatched_feature_candidate_records
    )
)


print(
    "Groups parsed:",
    len(
        clinician_excel_group_consensus_df
    ),
)


print(
    "Feature-membership rows parsed:",
    len(
        clinician_excel_feature_membership_df
    ),
)


print(
    "Unmatched candidate feature rows:",
    len(
        clinician_excel_unmatched_feature_candidates_df
    ),
)


assert not clinician_excel_group_consensus_df.empty, (
    "No clinician groups were parsed."
)


assert not clinician_excel_feature_membership_df.empty, (
    "No clinician feature memberships were parsed."
)



clinician_feature_mapping_audit_df = (
    clinician_excel_feature_membership_df
    .groupby(
        [
            "clinician_domain_code",
            "feature_mapping_method",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": (
                "n_feature_memberships"
            )
        }
    )
    .sort_values(
        [
            "clinician_domain_code",
            "feature_mapping_method",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nTechnical feature mapping audit:"
)


display(
    clinician_feature_mapping_audit_df
)


if not clinician_excel_unmatched_feature_candidates_df.empty:

    print(
        "\nUNMATCHED CANDIDATE FEATURES:"
    )


    display(
        clinician_excel_unmatched_feature_candidates_df
    )


assert (
    clinician_excel_unmatched_feature_candidates_df.empty
), (
    "At least one clinician technical feature could not "
    "be deterministically reconciled with the locked model "
    "feature universe."
)



clinician_excel_membership_domain_audit = (
    clinician_excel_feature_membership_df
    .groupby(
        "clinician_domain_code",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": (
                "parsed_feature_memberships"
            )
        }
    )
)


expected_domain_memberships_df = pd.DataFrame(
    {
        "clinician_domain_code": list(
            EXPECTED_EXCEL_DOMAIN_MEMBERSHIPS.keys()
        ),

        "expected_feature_memberships": list(
            EXPECTED_EXCEL_DOMAIN_MEMBERSHIPS.values()
        ),
    }
)


clinician_excel_membership_domain_audit = (
    expected_domain_memberships_df
    .merge(
        clinician_excel_membership_domain_audit,
        on="clinician_domain_code",
        how="left",
        validate="one_to_one",
    )
)


clinician_excel_membership_domain_audit[
    "parsed_feature_memberships"
] = (
    clinician_excel_membership_domain_audit[
        "parsed_feature_memberships"
    ]
    .fillna(
        0
    )
    .astype(int)
)


clinician_excel_membership_domain_audit[
    "count_match"
] = (
    clinician_excel_membership_domain_audit[
        "expected_feature_memberships"
    ]
    ==
    clinician_excel_membership_domain_audit[
        "parsed_feature_memberships"
    ]
)


print(
    "\nDomain-level membership audit:"
)


display(
    clinician_excel_membership_domain_audit
)


assert clinician_excel_membership_domain_audit[
    "count_match"
].all(), (
    "At least one clinician domain did not parse the "
    "expected number of feature-membership rows."
)


assert (
    clinician_excel_membership_domain_audit[
        "parsed_feature_memberships"
    ].sum()
    ==
    EXCEL_EXPECTED_FEATURE_MEMBERSHIPS
)



assert len(
    clinician_excel_group_consensus_df
) == EXCEL_EXPECTED_GROUPS, (
    "Unexpected number of clinician groups parsed from "
    f"Excel: {len(clinician_excel_group_consensus_df)} "
    f"(expected {EXCEL_EXPECTED_GROUPS})."
)


assert len(
    clinician_excel_feature_membership_df
) == EXCEL_EXPECTED_FEATURE_MEMBERSHIPS, (
    "Unexpected number of group-feature membership rows: "
    f"{len(clinician_excel_feature_membership_df)} "
    f"(expected {EXCEL_EXPECTED_FEATURE_MEMBERSHIPS})."
)


assert (
    clinician_excel_feature_membership_df[
        "feature"
    ]
    .nunique()
    ==
    EXCEL_EXPECTED_UNIQUE_FEATURES
), (
    "Unexpected number of unique canonical technical "
    "features in the clinician Excel workbook.\n"
    f"Observed: "
    f"{clinician_excel_feature_membership_df['feature'].nunique()}\n"
    f"Expected: {EXCEL_EXPECTED_UNIQUE_FEATURES}"
)


assert clinician_excel_group_consensus_df[
    "clinician_group_key"
].is_unique



parsed_group_feature_counts = (
    clinician_excel_feature_membership_df
    .groupby(
        "clinician_group_key",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": (
                "parsed_feature_memberships"
            )
        }
    )
)


clinician_excel_group_consensus_df = (
    clinician_excel_group_consensus_df
    .merge(
        parsed_group_feature_counts,
        on="clinician_group_key",
        how="left",
        validate="one_to_one",
    )
)


clinician_excel_group_consensus_df[
    "parsed_feature_memberships"
] = (
    clinician_excel_group_consensus_df[
        "parsed_feature_memberships"
    ]
    .fillna(
        0
    )
    .astype(int)
)


clinician_excel_group_consensus_df[
    "declared_count_matches_parsed"
] = (
    clinician_excel_group_consensus_df[
        "declared_feature_count"
    ]
    ==
    clinician_excel_group_consensus_df[
        "parsed_feature_memberships"
    ]
)


group_count_mismatches = (
    clinician_excel_group_consensus_df.loc[
        ~clinician_excel_group_consensus_df[
            "declared_count_matches_parsed"
        ]
    ]
    .copy()
)


if not group_count_mismatches.empty:

    print(
        "\nGROUP FEATURE-COUNT MISMATCHES:"
    )


    display(
        group_count_mismatches[
            [
                "clinician_domain",
                "clinician_group_key",
                "clinician_group_name",
                "declared_feature_count",
                "parsed_feature_memberships",
            ]
        ]
    )


assert group_count_mismatches.empty, (
    "One or more clinician groups have a mismatch between "
    "the Excel-declared feature count and parsed membership "
    "count."
)



assert clinician_excel_group_consensus_df[
    "clinician_vote_n"
].between(
    0,
    10
).all()


assert np.allclose(
    clinician_excel_group_consensus_df[
        "clinician_vote_prop"
    ]
    *
    10,
    clinician_excel_group_consensus_df[
        "clinician_vote_n"
    ],
    atol=1e-8,
    rtol=0.0,
)



s0_feature_df = (
    clinician_feature_universe.loc[
        clinician_feature_universe[
            "family"
        ]
        .astype(str)
        .eq(
            "S0"
        )
    ]
    .copy()
)


assert len(
    s0_feature_df
) == MANUAL_S0_EXPECTED_FEATURES


S0_FEATURES = (
    s0_feature_df[
        "source_raw_feature"
    ]
    .astype(str)
    .tolist()
)



EXPECTED_S0_FEATURES = {
    "age",
    "sex_gender",
    "arrival_transport_grouped",
    "temperature_c",
    "heart_rate",
    "respiratory_rate",
    "oxygen_saturation",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
}


assert set(
    S0_FEATURES
) == EXPECTED_S0_FEATURES, (
    "Frozen S0 feature universe differs from expected "
    "9 publication-model S0 features.\n\n"
    f"Observed:\n{sorted(S0_FEATURES)}\n\n"
    f"Expected:\n{sorted(EXPECTED_S0_FEATURES)}"
)



INITIAL_CONTEXT_FEATURES = {
    "age",
    "sex_gender",
    "arrival_transport_grouped",
}


TRIAGE_VITAL_FEATURES = {
    "temperature_c",
    "heart_rate",
    "respiratory_rate",
    "oxygen_saturation",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
}


assert len(
    INITIAL_CONTEXT_FEATURES
) == 3


assert len(
    TRIAGE_VITAL_FEATURES
) == 6


assert (
    INITIAL_CONTEXT_FEATURES
    |
    TRIAGE_VITAL_FEATURES
) == EXPECTED_S0_FEATURES


assert (
    INITIAL_CONTEXT_FEATURES
    &
    TRIAGE_VITAL_FEATURES
) == set()


print(
    "\nFrozen S0 clinician-validation features:"
)


print(
    "Initial context:",
    sorted(
        INITIAL_CONTEXT_FEATURES
    ),
)


print(
    "Triage vitals:",
    sorted(
        TRIAGE_VITAL_FEATURES
    ),
)


print(
    "Total S0 features:",
    len(
        EXPECTED_S0_FEATURES
    ),
)


print(
    "Clinician consensus:",
    "10/10 for all 9 S0 features",
)


manual_s0_group_records = [
    {
        "source_type": (
            "Manual documented consensus addition"
        ),

        "clinician_domain_code": (
            "INITIAL_CONTEXT"
        ),

        "clinician_domain": (
            "Initial context"
        ),

        "clinician_group_key": (
            "MANUAL_S0:INITIAL_CONTEXT"
        ),

        "clinician_group_number": (
            np.nan
        ),

        "clinician_group_name": (
            "Initial context"
        ),

        "declared_feature_count": (
            3
        ),

        "vote_raw": (
            1.0
        ),

        "vote_parsing_mode": (
            "manual_documented_10_of_10"
        ),

        "clinician_vote_n": (
            10
        ),

        "clinician_vote_prop": (
            1.0
        ),

        "clinician_vote_pct": (
            100.0
        ),

        "clinician_consensus_band": (
            "Unanimous important"
        ),

        "clinician_majority_important": (
            True
        ),

        "clinician_unanimous_important": (
            True
        ),

        "clinician_unanimous_not_important": (
            False
        ),

        "source_excel_row": (
            np.nan
        ),

        "clinician_comments": (
            "All 10 clinicians treated Initial Context "
            "as clinically important; not recorded in Excel."
        ),

        "parsed_feature_memberships": (
            3
        ),

        "declared_count_matches_parsed": (
            True
        ),
    },

    {
        "source_type": (
            "Manual documented consensus addition"
        ),

        "clinician_domain_code": (
            "TRIAGE_VITALS"
        ),

        "clinician_domain": (
            "Triage vitals"
        ),

        "clinician_group_key": (
            "MANUAL_S0:TRIAGE_VITALS"
        ),

        "clinician_group_number": (
            np.nan
        ),

        "clinician_group_name": (
            "Triage vitals"
        ),

        "declared_feature_count": (
            6
        ),

        "vote_raw": (
            1.0
        ),

        "vote_parsing_mode": (
            "manual_documented_10_of_10"
        ),

        "clinician_vote_n": (
            10
        ),

        "clinician_vote_prop": (
            1.0
        ),

        "clinician_vote_pct": (
            100.0
        ),

        "clinician_consensus_band": (
            "Unanimous important"
        ),

        "clinician_majority_important": (
            True
        ),

        "clinician_unanimous_important": (
            True
        ),

        "clinician_unanimous_not_important": (
            False
        ),

        "source_excel_row": (
            np.nan
        ),

        "clinician_comments": (
            "All 10 clinicians treated all six Triage "
            "Vital features as clinically important; "
            "not recorded in Excel."
        ),

        "parsed_feature_memberships": (
            6
        ),

        "declared_count_matches_parsed": (
            True
        ),
    },
]


clinician_manual_s0_group_consensus_df = (
    pd.DataFrame(
        manual_s0_group_records
    )
)



manual_s0_membership_records = []


for feature_name in (
    LOCKED_RAW_FEATURE_ORDER
):

    if feature_name not in (
        EXPECTED_S0_FEATURES
    ):

        continue


    if feature_name in (
        INITIAL_CONTEXT_FEATURES
    ):

        manual_domain_code = (
            "INITIAL_CONTEXT"
        )


        manual_domain = (
            "Initial context"
        )


        manual_group_key = (
            "MANUAL_S0:INITIAL_CONTEXT"
        )


        manual_group_name = (
            "Initial context"
        )


        model_category = (
            "Initial context"
        )


    else:

        manual_domain_code = (
            "TRIAGE_VITALS"
        )


        manual_domain = (
            "Triage vitals"
        )


        manual_group_key = (
            "MANUAL_S0:TRIAGE_VITALS"
        )


        manual_group_name = (
            "Triage vitals"
        )


        model_category = (
            "Triage vitals"
        )


    manual_s0_membership_records.append(
        {
            "source_type": (
                "Manual documented consensus addition"
            ),

            "clinician_domain_code": (
                manual_domain_code
            ),

            "clinician_domain": (
                manual_domain
            ),

            "clinician_group_key": (
                manual_group_key
            ),

            "clinician_group_number": (
                np.nan
            ),

            "clinician_group_name": (
                manual_group_name
            ),

            "feature_label": (
                feature_name
            ),

            "subgroup_label": (
                manual_group_name
            ),

            "excel_technical_feature": (
                np.nan
            ),

            "feature": (
                feature_name
            ),

            "feature_key": (
                feature_name.lower()
            ),

            "feature_mapping_method": (
                "manual_documented_s0"
            ),

            "model_category": (
                model_category
            ),

            "clinician_vote_n": (
                10
            ),

            "clinician_vote_prop": (
                1.0
            ),

            "clinician_vote_pct": (
                100.0
            ),

            "clinician_consensus_band": (
                "Unanimous important"
            ),

            "clinician_majority_important": (
                True
            ),

            "clinician_unanimous_important": (
                True
            ),

            "clinician_unanimous_not_important": (
                False
            ),

            "source_excel_number": (
                np.nan
            ),

            "source_excel_row": (
                np.nan
            ),
        }
    )


clinician_manual_s0_feature_membership_df = (
    pd.DataFrame(
        manual_s0_membership_records
    )
)


assert len(
    clinician_manual_s0_feature_membership_df
) == 9



clinician_group_consensus_df = pd.concat(
    [
        clinician_excel_group_consensus_df,
        clinician_manual_s0_group_consensus_df,
    ],
    ignore_index=True,
    sort=False,
)


clinician_feature_membership_df = pd.concat(
    [
        clinician_excel_feature_membership_df,
        clinician_manual_s0_feature_membership_df,
    ],
    ignore_index=True,
    sort=False,
)


assert len(
    clinician_group_consensus_df
) == (
    EXCEL_EXPECTED_GROUPS
    +
    2
)


assert len(
    clinician_feature_membership_df
) == (
    EXCEL_EXPECTED_FEATURE_MEMBERSHIPS
    +
    MANUAL_S0_EXPECTED_FEATURES
)



excel_unique_feature_set = set(
    clinician_excel_feature_membership_df[
        "feature"
    ]
)


manual_s0_feature_set = set(
    clinician_manual_s0_feature_membership_df[
        "feature"
    ]
)


assert excel_unique_feature_set.isdisjoint(
    manual_s0_feature_set
)


expected_excel_feature_set = (
    locked_feature_set
    -
    manual_s0_feature_set
)


excel_features_not_in_model = (
    excel_unique_feature_set
    -
    locked_feature_set
)


model_non_s0_features_missing_from_excel = (
    expected_excel_feature_set
    -
    excel_unique_feature_set
)


if excel_features_not_in_model:

    print(
        "\nExcel features not in locked model:"
    )


    display(
        pd.DataFrame(
            {
                "feature": sorted(
                    excel_features_not_in_model
                )
            }
        )
    )


if model_non_s0_features_missing_from_excel:

    print(
        "\nLocked non-S0 model features missing from Excel:"
    )


    display(
        pd.DataFrame(
            {
                "feature": sorted(
                    model_non_s0_features_missing_from_excel
                )
            }
        )
    )


assert not excel_features_not_in_model, (
    "Clinician Excel contains canonical technical features "
    "not present in the locked model feature universe."
)


assert not model_non_s0_features_missing_from_excel, (
    "One or more locked non-S0 model features are missing "
    "from the clinician Excel workbook."
)


assert excel_unique_feature_set == (
    expected_excel_feature_set
)



combined_unique_feature_set = set(
    clinician_feature_membership_df[
        "feature"
    ]
)


assert len(
    combined_unique_feature_set
) == COMBINED_EXPECTED_UNIQUE_FEATURES


assert combined_unique_feature_set == (
    locked_feature_set
)



def clinician_join_unique_strings(
    values,
):

    clean_values = [
        str(
            value
        )
        for value
        in values
        if pd.notna(
            value
        )
        and
        str(
            value
        ).strip()
        !=
        ""
    ]


    return " | ".join(
        dict.fromkeys(
            clean_values
        )
    )


def clinician_join_vote_values(
    values,
):

    clean_values = [
        int(
            value
        )
        for value
        in values
        if pd.notna(
            value
        )
    ]


    return " | ".join(
        str(
            value
        )
        for value
        in clean_values
    )



clinician_feature_consensus_df = (
    clinician_feature_membership_df
    .groupby(
        "feature",
        as_index=False,
        sort=False,
    )
    .agg(
        feature_key=(
            "feature_key",
            "first",
        ),

        model_category=(
            "model_category",
            "first",
        ),

        n_group_memberships=(
            "clinician_group_key",
            "nunique",
        ),

        clinician_group_keys=(
            "clinician_group_key",
            clinician_join_unique_strings,
        ),

        clinician_group_names=(
            "clinician_group_name",
            clinician_join_unique_strings,
        ),

        clinician_domains=(
            "clinician_domain",
            clinician_join_unique_strings,
        ),

        feature_mapping_methods=(
            "feature_mapping_method",
            clinician_join_unique_strings,
        ),

        all_clinician_vote_n=(
            "clinician_vote_n",
            clinician_join_vote_values,
        ),

        clinician_vote_n_max=(
            "clinician_vote_n",
            "max",
        ),

        clinician_vote_n_min=(
            "clinician_vote_n",
            "min",
        ),

        clinician_vote_n_mean=(
            "clinician_vote_n",
            "mean",
        ),

        clinician_vote_prop_max=(
            "clinician_vote_prop",
            "max",
        ),

        clinician_vote_prop_min=(
            "clinician_vote_prop",
            "min",
        ),

        clinician_vote_prop_mean=(
            "clinician_vote_prop",
            "mean",
        ),
    )
)



clinician_feature_consensus_df[
    "clinician_vote_n"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n_max"
    ]
    .astype(int)
)


clinician_feature_consensus_df[
    "clinician_vote_prop"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    /
    N_CLINICIANS
)


clinician_feature_consensus_df[
    "clinician_vote_pct"
] = (
    100.0
    *
    clinician_feature_consensus_df[
        "clinician_vote_prop"
    ]
)


clinician_feature_consensus_df[
    "clinician_consensus_band"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    .map(
        clinician_consensus_band
    )
)


clinician_feature_consensus_df[
    "clinician_majority_important"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    >=
    6
)


clinician_feature_consensus_df[
    "clinician_unanimous_important"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    ==
    10
)


clinician_feature_consensus_df[
    "clinician_divided_or_uncertain"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    .between(
        4,
        5,
    )
)


clinician_feature_consensus_df[
    "clinician_low_support"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    .between(
        1,
        3,
    )
)


clinician_feature_consensus_df[
    "clinician_unanimous_not_important"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    ==
    0
)



feature_order_df = pd.DataFrame(
    {
        "feature": (
            LOCKED_RAW_FEATURE_ORDER
        ),

        "locked_feature_position": np.arange(
            len(
                LOCKED_RAW_FEATURE_ORDER
            ),
            dtype=int,
        ),
    }
)


clinician_feature_consensus_df = (
    feature_order_df
    .merge(
        clinician_feature_consensus_df,
        on="feature",
        how="left",
        validate="one_to_one",
    )
)


assert len(
    clinician_feature_consensus_df
) == 2495


assert clinician_feature_consensus_df[
    "clinician_vote_n"
].notna().all()


assert (
    clinician_feature_consensus_df[
        "feature"
    ]
    .tolist()
    ==
    LOCKED_RAW_FEATURE_ORDER
)



clinician_group_vote_distribution_df = (
    clinician_group_consensus_df
    .groupby(
        [
            "clinician_domain",
            "clinician_vote_n",
            "clinician_consensus_band",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": (
                "n_groups"
            )
        }
    )
    .sort_values(
        [
            "clinician_domain",
            "clinician_vote_n",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



clinician_feature_vote_distribution_df = (
    clinician_feature_consensus_df
    .groupby(
        [
            "model_category",
            "clinician_vote_n",
            "clinician_consensus_band",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": (
                "n_unique_features"
            )
        }
    )
    .sort_values(
        [
            "model_category",
            "clinician_vote_n",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



clinician_multi_group_features_df = (
    clinician_feature_consensus_df.loc[
        clinician_feature_consensus_df[
            "n_group_memberships"
        ]
        >
        1
    ]
    .copy()
    .reset_index(
        drop=True
    )
)



CLINICIAN_REVIEW_SHA256 = (
    notebook4_file_sha256(
        CLINICIAN_REVIEW_PATH
    )
)


clinician_source_checksum_df = pd.DataFrame(
    [
        {
            "source_file": (
                CLINICIAN_REVIEW_PATH.name
            ),

            "source_path": str(
                CLINICIAN_REVIEW_PATH
            ),

            "sheet_name": (
                CLINICIAN_REVIEW_SHEET
            ),

            "size_bytes": int(
                CLINICIAN_REVIEW_PATH
                .stat()
                .st_size
            ),

            "sha256": (
                CLINICIAN_REVIEW_SHA256
            ),

            "source_frozen_for_analysis": (
                True
            ),
        }
    ]
)



clinician_source_parsing_audit_df = pd.DataFrame(
    [
        {
            "source_sheet": (
                CLINICIAN_REVIEW_SHEET
            ),

            "excel_header_row": (
                CLINICIAN_EXCEL_HEADER_ROW_NUMBER
            ),

            "number_of_clinicians": (
                N_CLINICIANS
            ),

            "excel_groups_parsed": int(
                len(
                    clinician_excel_group_consensus_df
                )
            ),

            "expected_excel_groups": (
                EXCEL_EXPECTED_GROUPS
            ),

            "excel_feature_membership_rows": int(
                len(
                    clinician_excel_feature_membership_df
                )
            ),

            "expected_excel_feature_membership_rows": (
                EXCEL_EXPECTED_FEATURE_MEMBERSHIPS
            ),

            "excel_unique_canonical_features": int(
                clinician_excel_feature_membership_df[
                    "feature"
                ]
                .nunique()
            ),

            "expected_excel_unique_features": (
                EXCEL_EXPECTED_UNIQUE_FEATURES
            ),

            "exact_feature_memberships": int(
                (
                    clinician_excel_feature_membership_df[
                        "feature_mapping_method"
                    ]
                    ==
                    "exact_locked_feature_match"
                ).sum()
            ),

            "imaging_prefix_mapped_memberships": int(
                (
                    clinician_excel_feature_membership_df[
                        "feature_mapping_method"
                    ]
                    ==
                    "imaging_prefix_canonicalization"
                ).sum()
            ),

            "documented_exact_alias_memberships": int(
                (
                    clinician_excel_feature_membership_df[
                        "feature_mapping_method"
                    ]
                    ==
                    "documented_exact_alias"
                ).sum()
            ),

            "explicit_feature_aliases_defined": int(
                len(
                    CLINICIAN_EXPLICIT_FEATURE_ALIASES
                )
            ),

            "unmatched_feature_candidates": int(
                len(
                    clinician_excel_unmatched_feature_candidates_df
                )
            ),

            "manual_s0_groups_added": (
                2
            ),

            "manual_s0_features_added": int(
                len(
                    clinician_manual_s0_feature_membership_df
                )
            ),

            "combined_group_rows": int(
                len(
                    clinician_group_consensus_df
                )
            ),

            "combined_feature_membership_rows": int(
                len(
                    clinician_feature_membership_df
                )
            ),

            "combined_unique_features": int(
                clinician_feature_consensus_df[
                    "feature"
                ]
                .nunique()
            ),

            "expected_locked_features": (
                COMBINED_EXPECTED_UNIQUE_FEATURES
            ),

            "excel_features_not_in_locked_model": int(
                len(
                    excel_features_not_in_model
                )
            ),

            "non_s0_model_features_missing_from_excel": int(
                len(
                    model_non_s0_features_missing_from_excel
                )
            ),

            "combined_clinician_features_missing_from_model": int(
                len(
                    combined_unique_feature_set
                    -
                    locked_feature_set
                )
            ),

            "locked_model_features_missing_clinician_consensus": int(
                len(
                    locked_feature_set
                    -
                    combined_unique_feature_set
                )
            ),

            "groups_with_feature_count_mismatch": int(
                len(
                    group_count_mismatches
                )
            ),

            "features_with_multiple_group_memberships": int(
                len(
                    clinician_multi_group_features_df
                )
            ),

            "feature_level_multi_membership_rule": (
                "maximum clinician vote across memberships"
            ),

            "initial_context_vote": (
                "10/10"
            ),

            "triage_vitals_vote": (
                "10/10"
            ),

            "word_documents_used": (
                False
            ),

            "fuzzy_feature_matching_used": (
                False
            ),

            "clinician_model_comparison_started": (
                False
            ),

            "complete_2495_feature_alignment": (
                True
            ),
        }
    ]
)



NOTEBOOK4_CLINICAL_VALIDATION_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook4"
    / "clinical_validation"
)


NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR = (
    NOTEBOOK4_CLINICAL_VALIDATION_DIR
    / "clinician_consensus_source"
)


NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)



CLINICIAN_GROUP_CONSENSUS_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_group_consensus.csv"
)


CLINICIAN_FEATURE_MEMBERSHIP_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_feature_membership.csv"
)


CLINICIAN_FEATURE_CONSENSUS_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_feature_consensus.csv"
)


CLINICIAN_MANUAL_S0_GROUPS_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_manual_s0_group_consensus.csv"
)


CLINICIAN_MANUAL_S0_MEMBERSHIP_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_manual_s0_feature_membership.csv"
)


CLINICIAN_MULTI_GROUP_FEATURES_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_features_with_multiple_group_memberships.csv"
)


CLINICIAN_GROUP_VOTE_DISTRIBUTION_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_group_vote_distribution.csv"
)


CLINICIAN_FEATURE_VOTE_DISTRIBUTION_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_feature_vote_distribution.csv"
)


CLINICIAN_SOURCE_CHECKSUM_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_source_checksum.csv"
)


CLINICIAN_PARSING_AUDIT_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_source_parsing_audit.csv"
)


CLINICIAN_MAPPING_AUDIT_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_feature_mapping_audit.csv"
)


CLINICIAN_DOMAIN_AUDIT_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_excel_membership_domain_audit.csv"
)


CLINICIAN_EXPLICIT_ALIAS_AUDIT_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_explicit_feature_aliases.csv"
)


CLINICIAN_UNMATCHED_FEATURES_PATH = (
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_unmatched_feature_candidates.csv"
)



clinician_group_consensus_df.to_csv(
    CLINICIAN_GROUP_CONSENSUS_PATH,
    index=False,
)


clinician_feature_membership_df.to_csv(
    CLINICIAN_FEATURE_MEMBERSHIP_PATH,
    index=False,
)


clinician_feature_consensus_df.to_csv(
    CLINICIAN_FEATURE_CONSENSUS_PATH,
    index=False,
)


clinician_manual_s0_group_consensus_df.to_csv(
    CLINICIAN_MANUAL_S0_GROUPS_PATH,
    index=False,
)


clinician_manual_s0_feature_membership_df.to_csv(
    CLINICIAN_MANUAL_S0_MEMBERSHIP_PATH,
    index=False,
)


clinician_multi_group_features_df.to_csv(
    CLINICIAN_MULTI_GROUP_FEATURES_PATH,
    index=False,
)


clinician_group_vote_distribution_df.to_csv(
    CLINICIAN_GROUP_VOTE_DISTRIBUTION_PATH,
    index=False,
)


clinician_feature_vote_distribution_df.to_csv(
    CLINICIAN_FEATURE_VOTE_DISTRIBUTION_PATH,
    index=False,
)


clinician_source_checksum_df.to_csv(
    CLINICIAN_SOURCE_CHECKSUM_PATH,
    index=False,
)


clinician_source_parsing_audit_df.to_csv(
    CLINICIAN_PARSING_AUDIT_PATH,
    index=False,
)


clinician_feature_mapping_audit_df.to_csv(
    CLINICIAN_MAPPING_AUDIT_PATH,
    index=False,
)


clinician_excel_membership_domain_audit.to_csv(
    CLINICIAN_DOMAIN_AUDIT_PATH,
    index=False,
)


clinician_explicit_feature_aliases_df.to_csv(
    CLINICIAN_EXPLICIT_ALIAS_AUDIT_PATH,
    index=False,
)


clinician_excel_unmatched_feature_candidates_df.to_csv(
    CLINICIAN_UNMATCHED_FEATURES_PATH,
    index=False,
)



clinician_feature_membership_df.to_parquet(
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_feature_membership.parquet",
    index=False,
    engine="pyarrow",
)


clinician_feature_consensus_df.to_parquet(
    NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
    / "clinician_feature_consensus.parquet",
    index=False,
    engine="pyarrow",
)



reloaded_clinician_feature_consensus = (
    pd.read_parquet(
        NOTEBOOK4_CLINICAL_VALIDATION_SOURCE_DIR
        / "clinician_feature_consensus.parquet",
        engine="pyarrow",
    )
)


assert len(
    reloaded_clinician_feature_consensus
) == 2495


assert (
    reloaded_clinician_feature_consensus[
        "feature"
    ]
    .tolist()
    ==
    LOCKED_RAW_FEATURE_ORDER
)


assert reloaded_clinician_feature_consensus[
    "clinician_vote_n"
].notna().all()



print(
    "\nClinician consensus source parsing audit:"
)


display(
    clinician_source_parsing_audit_df
)


print(
    "\n10-clinician group vote distribution:"
)


display(
    clinician_group_vote_distribution_df
)


print(
    "\nFeature-level clinician vote distribution:"
)


display(
    clinician_feature_vote_distribution_df
)


print(
    "\nFeatures belonging to >1 clinician-defined group:"
)


print(
    "Count:",
    len(
        clinician_multi_group_features_df
    ),
)


if not clinician_multi_group_features_df.empty:

    display(
        clinician_multi_group_features_df[
            [
                "feature",
                "model_category",
                "n_group_memberships",
                "clinician_group_keys",
                "clinician_group_names",
                "all_clinician_vote_n",
                "clinician_vote_n_min",
                "clinician_vote_n_max",
                "clinician_vote_n",
                "clinician_consensus_band",
            ]
        ]
        .sort_values(
            [
                "model_category",
                "feature",
            ]
        )
        .reset_index(
            drop=True
        )
    )


print(
    "\nManual S0 clinician consensus:"
)


display(
    clinician_manual_s0_feature_membership_df[
        [
            "model_category",
            "feature",
            "clinician_vote_n",
            "clinician_vote_pct",
            "clinician_consensus_band",
            "source_type",
        ]
    ]
)



assert len(
    clinician_excel_group_consensus_df
) == 73


assert len(
    clinician_excel_feature_membership_df
) == 2493


assert (
    clinician_excel_feature_membership_df[
        "feature"
    ].nunique()
    ==
    2486
)


n_imaging_prefix_mapped = int(
    (
        clinician_excel_feature_membership_df[
            "feature_mapping_method"
        ]
        ==
        "imaging_prefix_canonicalization"
    ).sum()
)


n_documented_alias_mapped = int(
    (
        clinician_excel_feature_membership_df[
            "feature_mapping_method"
        ]
        ==
        "documented_exact_alias"
    ).sum()
)


assert n_imaging_prefix_mapped == 82, (
    "Expected exactly 82 standard Imaging namespace-prefix "
    f"mappings; observed {n_imaging_prefix_mapped}."
)


assert n_documented_alias_mapped == 1, (
    "Expected exactly 1 documented exact feature alias; "
    f"observed {n_documented_alias_mapped}."
)


assert (
    n_imaging_prefix_mapped
    +
    n_documented_alias_mapped
) == 83, (
    "Expected exactly 83 Imaging feature memberships after "
    "deterministic namespace reconciliation."
)


assert (
    clinician_excel_unmatched_feature_candidates_df.empty
)


assert len(
    clinician_feature_consensus_df
) == 2495


assert set(
    clinician_feature_consensus_df[
        "feature"
    ]
) == locked_feature_set



print(
    "\n=================================================="
)


print(
    "10-CLINICIAN CONSENSUS SOURCE VERIFIED"
)


print(
    "=================================================="
)


print(
    "\nSOURCE"
)


print(
    "Excel:",
    CLINICIAN_REVIEW_PATH,
)


print(
    "Sheet:",
    CLINICIAN_REVIEW_SHEET,
)


print(
    "SHA-256:",
    CLINICIAN_REVIEW_SHA256,
)


print(
    "Word files used:",
    "NO",
)


print(
    "\nEXCEL CLINICIAN REVIEW"
)


print(
    "Clinicians:",
    N_CLINICIANS,
)


print(
    "Groups parsed:",
    len(
        clinician_excel_group_consensus_df
    ),
)


print(
    "Feature membership rows:",
    len(
        clinician_excel_feature_membership_df
    ),
)


print(
    "Unique canonical Excel technical features:",
    clinician_excel_feature_membership_df[
        "feature"
    ].nunique(),
)


print(
    "Imaging memberships mapped by deterministic prefix:",
    n_imaging_prefix_mapped,
)


print(
    "Imaging memberships mapped by documented exact alias:",
    n_documented_alias_mapped,
)


print(
    "Explicit documented aliases defined:",
    len(
        CLINICIAN_EXPLICIT_FEATURE_ALIASES
    ),
)


print(
    "Unmatched candidate features:",
    len(
        clinician_excel_unmatched_feature_candidates_df
    ),
)


print(
    "\nMANUAL S0 CONSENSUS"
)


print(
    "Initial context:",
    "3 features — 10/10",
)


print(
    "Triage vitals:",
    "6 features — 10/10",
)


print(
    "Manual S0 features:",
    len(
        clinician_manual_s0_feature_membership_df
    ),
)


print(
    "\nCOMBINED CLINICIAN FEATURE UNIVERSE"
)


print(
    "Unique clinician-covered technical features:",
    clinician_feature_consensus_df[
        "feature"
    ].nunique(),
)


print(
    "Locked model features:",
    len(
        LOCKED_RAW_FEATURE_ORDER
    ),
)


print(
    "Exact 2,495-feature alignment:",
    "YES",
)


print(
    "Missing clinician consensus:",
    "0",
)


print(
    "Extra clinician features:",
    "0",
)


print(
    "\nMULTIPLE GROUP MEMBERSHIPS"
)


print(
    "Features in >1 clinician group:",
    len(
        clinician_multi_group_features_df
    ),
)


print(
    "Primary feature-level clinician rule:",
    "MAXIMUM vote across valid group memberships",
)


print(
    "\nCONSENSUS INTERPRETATION"
)


print(
    "10/10:",
    "Unanimous important",
)


print(
    "8–9/10:",
    "Strong consensus important",
)


print(
    "6–7/10:",
    "Majority important",
)


print(
    "4–5/10:",
    "Divided / uncertain",
)


print(
    "1–3/10:",
    "Low clinician support",
)


print(
    "0/10:",
    "Unanimous not important",
)


print(
    "\nFEATURE MAPPING"
)


print(
    "Exact feature matching:",
    "YES",
)


print(
    "Imaging namespace-prefix restoration:",
    "YES",
)


print(
    "Documented exact alias reconciliation:",
    "YES — 1 alias",
)


print(
    "Fuzzy matching:",
    "NO",
)


print(
    "\nMODEL COMPARISON STARTED:",
    "NO",
)


print(
    "\nCanonical group consensus saved:"
)


print(
    CLINICIAN_GROUP_CONSENSUS_PATH
)


print(
    "\nCanonical feature membership saved:"
)


print(
    CLINICIAN_FEATURE_MEMBERSHIP_PATH
)


print(
    "\nCanonical feature consensus saved:"
)


print(
    CLINICIAN_FEATURE_CONSENSUS_PATH
)


print(
    "\nFeature mapping audit saved:"
)


print(
    CLINICIAN_MAPPING_AUDIT_PATH
)


print(
    "\nExplicit feature alias audit saved:"
)


print(
    CLINICIAN_EXPLICIT_ALIAS_AUDIT_PATH
)


print(
    "\nREADY FOR CATEGORY-WISE CLINICIAN VALIDATION:",
    "YES",
)

#### Ten-clinician clinical validation of cross-dataset robust SHAP features

Clinical validation is performed after completion of model development, internal temporal testing, external validation and SHAP robustness analysis. No model fitting, recalibration, threshold optimization, SHAP recalculation or predictive feature reselection is performed during this stage.

Clinical opinion was obtained from 10 clinicians at the clinician-defined group level. The clinician workbook contains 73 reviewed groups covering PMH Medication, PMH Diagnosis, Laboratory and Imaging features. Initial Context and Triage Vital features were additionally documented as unanimously important by all 10 clinicians. The final clinician-validation universe therefore covers exactly the same 2,495 clinically interpretable model features used in the SHAP analysis.

Clinician consensus is represented as the number of clinicians, out of 10, who considered a group clinically important. The following prespecified consensus categories are used:

| Clinician votes | Consensus tag | Methodological interpretation |
|---|---|---|
| 10/10 | Unanimous important | All clinicians considered the group important |
| 8–9/10 | Strong consensus important | Very strong clinician support |
| 6–7/10 | Majority important | More than half of clinicians considered the group important |
| 4–5/10 | Divided / uncertain | Clinician opinion was mixed and no clear majority support was present |
| 1–3/10 | Low clinician support | Only a minority of clinicians considered the group important |
| 0/10 | Unanimous not important | No clinician considered the group important |

For feature-level analysis, the terms `clinician-unselected`, `clinician-underrecognized`, `clinician-divided/uncertain` and `clinician-supported` are used operationally:

| Clinician votes | Feature-level operational tag |
|---|---|
| 0/10 | Clinician-unselected |
| 1–3/10 | Clinician-underrecognized |
| 4–5/10 | Clinician-divided / uncertain |
| 6–10/10 | Clinician-supported |

Some technical features belong to more than one clinician-defined group. All original memberships and votes are retained. For the unique feature-level comparison, the maximum clinician vote across valid group memberships is used. This conservative rule prevents a feature from being labelled clinician-unselected or clinician-underrecognized when clinicians strongly recognised the same feature within another legitimate clinical grouping.

Model robustness is defined from the completed 81-combination SHAP stability analysis. A feature is considered selected in a dataset when it was selected in at least one robustness combination (`times_selected > 0`). This includes High, Medium, Low and Below-P25 category-relative stability tiers. The 2,495-feature universe is classified into four mutually exclusive model-evidence states:

| Model robustness status | Definition |
|---|---|
| BOTH | Robustly selected in both MIMIC-IV and MC-MED |
| MIMIC-IV only | Robustly selected only in MIMIC-IV |
| MC-MED only | Robustly selected only in MC-MED |
| Neither | Not robustly selected in either dataset |

The primary discovery findings require replication across both datasets. A feature selected in BOTH datasets with 0/10 clinician support is tagged `Replicated clinician-unselected model signal`. A feature selected in BOTH datasets with 1–3/10 support is tagged `Replicated clinician-underrecognized model signal`. These form the primary candidate pool for subsequent literature review and clinical adjudication. They are not described as novel features at this stage.

Features selected in BOTH datasets with 4–5/10 clinician support are tagged `Replicated model signal with divided clinician opinion`. Features selected in BOTH datasets with at least 6/10 support are tagged `Replicated clinician-model concordant signal`.

Secondary discovery findings are features selected in exactly one dataset but receiving 0–3/10 clinician support. They are classified as `One-dataset clinician-unselected model signal` or `One-dataset clinician-underrecognized model signal`. The specific dataset, MIMIC-IV or MC-MED, is retained separately.

The reverse-discrepancy analysis identifies features receiving at least 6/10 clinician support that were not robustly selected in either dataset. These are tagged `Clinician-supported feature not robustly selected in either dataset`. This analysis ensures that clinical validation examines both model findings not strongly recognised by clinicians and clinician-prioritised features not robustly captured by the model.

The principal feature-level validation tags are therefore:

| Model evidence | Clinician support | Clinical-validation tag | Role |
|---|---:|---|---|
| BOTH | 0/10 | Replicated clinician-unselected model signal | Primary discovery |
| BOTH | 1–3/10 | Replicated clinician-underrecognized model signal | Primary discovery |
| BOTH | 4–5/10 | Replicated model signal with divided clinician opinion | Primary contextual finding |
| BOTH | 6–10/10 | Replicated clinician-model concordant signal | Primary concordance |
| One dataset | 0/10 | One-dataset clinician-unselected model signal | Secondary discovery |
| One dataset | 1–3/10 | One-dataset clinician-underrecognized model signal | Secondary discovery |
| One dataset | 4–5/10 | One-dataset model signal with divided clinician opinion | Secondary contextual finding |
| One dataset | 6–10/10 | One-dataset clinician-model concordant signal | Secondary concordance |
| Neither | 6–10/10 | Clinician-supported feature not robustly selected in either dataset | Reverse discrepancy |

Clinical validation is performed at two complementary levels. The clinician-group level is the primary validation unit because clinicians judged groups directly. The individual-feature level is the discovery unit because SHAP robustness operates on individual model features. Group-level analysis records the number and proportion of features within each clinician group that were robust in both datasets, one dataset or neither dataset.

All feature-level and group-level analyses are archived overall and separately by clinical category. Primary replicated clinician-unselected and clinician-underrecognized findings, secondary one-dataset findings, clinician-model concordant findings, divided-opinion findings and reverse-discrepancy findings are retained as separate outputs for later interpretation and literature-based novelty assessment.

In [ ]:



required_clinical_validation_objects = [
    "PROJECT_ROOT",
    "LOCKED_RAW_FEATURE_ORDER",
    "clinician_feature_consensus_df",
    "clinician_feature_membership_df",
    "clinician_group_consensus_df",
    "nb4_mimic_all_stability_features",
    "nb4_mcmed_all_stability_features",
]


missing_clinical_validation_objects = [
    object_name
    for object_name
    in required_clinical_validation_objects
    if object_name not in globals()
]


if missing_clinical_validation_objects:

    raise RuntimeError(
        "Required frozen Notebook 4 objects are missing.\n\n"
        "Missing:\n"
        +
        "\n".join(
            missing_clinical_validation_objects
        )
    )


locked_feature_set = set(
    LOCKED_RAW_FEATURE_ORDER
)


GENERIC_IMAGING_PARENT_GROUP_MAP = {

    (
        "poe_result__imaging_radiology__"
        "imaging_radiology_orders_ct_scan"
    ): [
        "IMAGING:02",
        "IMAGING:05",
        "IMAGING:11",
    ],

    (
        "poe_result__imaging_radiology__"
        "imaging_radiology_orders_ultrasound"
    ): [
        "IMAGING:01",
        "IMAGING:03",
        "IMAGING:04",
        "IMAGING:08",
        "IMAGING:09",
        "IMAGING:12",
    ],

    (
        "poe_result__imaging_radiology__"
        "imaging_radiology_orders_general_xray"
    ): [
        "IMAGING:06",
        "IMAGING:10",
        "IMAGING:18",
    ],
}


semantic_membership_records = []


for feature_name, target_group_keys in (
    GENERIC_IMAGING_PARENT_GROUP_MAP.items()
):

    assert feature_name in locked_feature_set, (
        "Generic Imaging feature not found in frozen "
        f"model universe:\n{feature_name}"
    )


    existing_group_keys = set(
        clinician_feature_membership_df.loc[
            clinician_feature_membership_df[
                "feature"
            ].eq(feature_name),
            "clinician_group_key",
        ]
        .astype(str)
    )


    for target_group_key in target_group_keys:

        target_group_rows = (
            clinician_group_consensus_df.loc[
                clinician_group_consensus_df[
                    "clinician_group_key"
                ].eq(
                    target_group_key
                )
            ]
            .copy()
        )


        assert len(
            target_group_rows
        ) == 1, (
            "Each Imaging semantic mapping must resolve "
            "to exactly one clinician group.\n"
            f"Feature: {feature_name}\n"
            f"Group: {target_group_key}"
        )


        target_group = (
            target_group_rows
            .iloc[0]
        )


        assert (
            target_group[
                "clinician_domain"
            ]
            ==
            "Imaging"
        )


        if target_group_key in existing_group_keys:
            continue


        semantic_membership_records.append(
            {
                "source_type": (
                    "Documented semantic membership reconciliation"
                ),

                "clinician_domain_code": (
                    target_group[
                        "clinician_domain_code"
                    ]
                ),

                "clinician_domain": (
                    target_group[
                        "clinician_domain"
                    ]
                ),

                "clinician_group_key": (
                    target_group[
                        "clinician_group_key"
                    ]
                ),

                "clinician_group_number": (
                    target_group[
                        "clinician_group_number"
                    ]
                ),

                "clinician_group_name": (
                    target_group[
                        "clinician_group_name"
                    ]
                ),

                "feature_label": (
                    feature_name.split("__")[-1]
                ),

                "subgroup_label": (
                    "Generic modality-level imaging-order variable"
                ),

                "excel_technical_feature": np.nan,

                "feature": feature_name,

                "feature_key": (
                    feature_name.lower()
                ),

                "feature_mapping_method": (
                    "generic_modality_to_specific_clinician_groups"
                ),

                "model_category": (
                    "Imaging / radiology"
                ),

                "clinician_vote_n": int(
                    target_group[
                        "clinician_vote_n"
                    ]
                ),

                "clinician_vote_prop": float(
                    target_group[
                        "clinician_vote_prop"
                    ]
                ),

                "clinician_vote_pct": float(
                    target_group[
                        "clinician_vote_pct"
                    ]
                ),

                "clinician_consensus_band": (
                    target_group[
                        "clinician_consensus_band"
                    ]
                ),

                "clinician_majority_important": bool(
                    target_group[
                        "clinician_majority_important"
                    ]
                ),

                "clinician_unanimous_important": bool(
                    target_group[
                        "clinician_unanimous_important"
                    ]
                ),

                "clinician_unanimous_not_important": bool(
                    target_group[
                        "clinician_unanimous_not_important"
                    ]
                ),

                "source_excel_number": np.nan,
                "source_excel_row": np.nan,
            }
        )


if semantic_membership_records:

    clinician_imaging_semantic_membership_df = (
        pd.DataFrame(
            semantic_membership_records
        )
    )


    clinician_feature_membership_df = (
        pd.concat(
            [
                clinician_feature_membership_df,
                clinician_imaging_semantic_membership_df,
            ],
            ignore_index=True,
            sort=False,
        )
    )


for feature_name, target_group_keys in (
    GENERIC_IMAGING_PARENT_GROUP_MAP.items()
):

    actual_group_keys = set(
        clinician_feature_membership_df.loc[
            clinician_feature_membership_df[
                "feature"
            ].eq(feature_name),
            "clinician_group_key",
        ]
        .astype(str)
    )


    assert set(
        target_group_keys
    ).issubset(
        actual_group_keys
    )


clinician_feature_consensus_df = (
    clinician_feature_membership_df
    .groupby(
        "feature",
        as_index=False,
        sort=False,
    )
    .agg(
        feature_key=(
            "feature_key",
            "first",
        ),

        model_category=(
            "model_category",
            "first",
        ),

        n_group_memberships=(
            "clinician_group_key",
            "nunique",
        ),

        clinician_group_keys=(
            "clinician_group_key",
            clinician_join_unique_strings,
        ),

        clinician_group_names=(
            "clinician_group_name",
            clinician_join_unique_strings,
        ),

        clinician_domains=(
            "clinician_domain",
            clinician_join_unique_strings,
        ),

        feature_mapping_methods=(
            "feature_mapping_method",
            clinician_join_unique_strings,
        ),

        all_clinician_vote_n=(
            "clinician_vote_n",
            clinician_join_vote_values,
        ),

        clinician_vote_n_max=(
            "clinician_vote_n",
            "max",
        ),

        clinician_vote_n_min=(
            "clinician_vote_n",
            "min",
        ),

        clinician_vote_n_mean=(
            "clinician_vote_n",
            "mean",
        ),

        clinician_vote_prop_max=(
            "clinician_vote_prop",
            "max",
        ),

        clinician_vote_prop_min=(
            "clinician_vote_prop",
            "min",
        ),

        clinician_vote_prop_mean=(
            "clinician_vote_prop",
            "mean",
        ),
    )
)


clinician_feature_consensus_df[
    "clinician_vote_n"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n_max"
    ]
    .astype(int)
)


clinician_feature_consensus_df[
    "clinician_vote_prop"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    /
    N_CLINICIANS
)


clinician_feature_consensus_df[
    "clinician_vote_pct"
] = (
    100.0
    *
    clinician_feature_consensus_df[
        "clinician_vote_prop"
    ]
)


clinician_feature_consensus_df[
    "clinician_consensus_band"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    .map(
        clinician_consensus_band
    )
)


clinician_feature_consensus_df[
    "clinician_majority_important"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    >=
    6
)


clinician_feature_consensus_df[
    "clinician_unanimous_important"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    ==
    10
)


clinician_feature_consensus_df[
    "clinician_divided_or_uncertain"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    .between(
        4,
        5,
    )
)


clinician_feature_consensus_df[
    "clinician_low_support"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    .between(
        1,
        3,
    )
)


clinician_feature_consensus_df[
    "clinician_unanimous_not_important"
] = (
    clinician_feature_consensus_df[
        "clinician_vote_n"
    ]
    ==
    0
)


feature_order_df = pd.DataFrame(
    {
        "feature": (
            LOCKED_RAW_FEATURE_ORDER
        ),

        "locked_feature_position": np.arange(
            len(
                LOCKED_RAW_FEATURE_ORDER
            ),
            dtype=int,
        ),
    }
)


clinician_feature_consensus_df = (
    feature_order_df
    .merge(
        clinician_feature_consensus_df,
        on="feature",
        how="left",
        validate="one_to_one",
    )
)


assert len(
    LOCKED_RAW_FEATURE_ORDER
) == 2495


assert len(
    clinician_feature_consensus_df
) == 2495


assert clinician_feature_consensus_df[
    "feature"
].is_unique


assert (
    set(
        clinician_feature_consensus_df[
            "feature"
        ]
    )
    ==
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
)



CLINICAL_VALIDATION_CATEGORY_ORDER = [
    "Initial context",
    "Triage vitals",
    "PMH medication class",
    "PMH medication name",
    "PMH ICD title",
    "Lab result",
    "Imaging / radiology",
]


CLINICAL_VALIDATION_DOMAIN_ORDER = [
    "Initial context",
    "Triage vitals",
    "PMH Medication",
    "PMH Diagnosis",
    "Laboratory",
    "Imaging",
]



def nb4_normalize_clinical_category(
    value,
):

    if pd.isna(
        value
    ):

        return np.nan


    value = str(
        value
    ).strip()


    category_aliases = {
        "Triage vitals / pain": (
            "Triage vitals"
        ),
    }


    return category_aliases.get(
        value,
        value,
    )



MODEL_EVIDENCE_REQUIRED_COLUMNS = [
    "feature",
    "times_selected",
    "stability_pct",
    "importance_tag",
    "global_rank",
    "mean_abs_shap",
]


MODEL_EVIDENCE_OPTIONAL_COLUMNS = [
    "category",
    "feature_clean",
    "dominant_abs_shap",
    "abs_high_low_separation",
    "mean_shap_high_value",
    "mean_shap_low_value",
    "n_high",
    "n_low",
    "most_common_pattern_code",
    "most_common_pattern",
]


def nb4_prepare_model_evidence(
    source_df,
    dataset_prefix,
):

    missing_columns = [
        column_name
        for column_name
        in MODEL_EVIDENCE_REQUIRED_COLUMNS
        if column_name
        not in source_df.columns
    ]


    if missing_columns:

        raise RuntimeError(
            f"{dataset_prefix} SHAP stability table is "
            "missing required columns:\n"
            +
            "\n".join(
                missing_columns
            )
        )


    assert len(
        source_df
    ) == 2495


    assert source_df[
        "feature"
    ].is_unique


    assert (
        set(
            source_df[
                "feature"
            ]
        )
        ==
        set(
            LOCKED_RAW_FEATURE_ORDER
        )
    )


    keep_columns = (
        MODEL_EVIDENCE_REQUIRED_COLUMNS
        +
        [
            column_name
            for column_name
            in MODEL_EVIDENCE_OPTIONAL_COLUMNS
            if column_name
            in source_df.columns
        ]
    )


    prepared_df = (
        source_df[
            keep_columns
        ]
        .copy()
    )



    prepared_df[
        "selected"
    ] = (
        pd.to_numeric(
            prepared_df[
                "times_selected"
            ],
            errors="coerce",
        )
        .fillna(
            0
        )
        >
        0
    )


    if "importance_tag" in prepared_df.columns:

        selected_tag_error = (
            prepared_df[
                "selected"
            ]
            &
            prepared_df[
                "importance_tag"
            ]
            .astype(str)
            .eq(
                "Not selected"
            )
        )


        assert not selected_tag_error.any()


    if "category" in prepared_df.columns:

        prepared_df[
            "category"
        ] = (
            prepared_df[
                "category"
            ]
            .map(
                nb4_normalize_clinical_category
            )
        )


    rename_map = {
        column_name: (
            column_name
            if column_name
            ==
            "feature"
            else
            f"{dataset_prefix}_{column_name}"
        )
        for column_name
        in prepared_df.columns
    }


    prepared_df = (
        prepared_df
        .rename(
            columns=rename_map
        )
    )


    return prepared_df


mimic_model_evidence_df = (
    nb4_prepare_model_evidence(
        nb4_mimic_all_stability_features,
        "mimic",
    )
)


mcmed_model_evidence_df = (
    nb4_prepare_model_evidence(
        nb4_mcmed_all_stability_features,
        "mcmed",
    )
)



clinician_model_feature_validation_df = (
    clinician_feature_consensus_df
    .copy()
    .merge(
        mimic_model_evidence_df,
        on="feature",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mcmed_model_evidence_df,
        on="feature",
        how="left",
        validate="one_to_one",
    )
)


assert len(
    clinician_model_feature_validation_df
) == 2495


assert clinician_model_feature_validation_df[
    "feature"
].is_unique


assert (
    clinician_model_feature_validation_df[
        "feature"
    ]
    .tolist()
    ==
    LOCKED_RAW_FEATURE_ORDER
)


assert clinician_model_feature_validation_df[
    "mimic_selected"
].notna().all()


assert clinician_model_feature_validation_df[
    "mcmed_selected"
].notna().all()


clinician_model_feature_validation_df[
    "mimic_selected"
] = (
    clinician_model_feature_validation_df[
        "mimic_selected"
    ]
    .astype(bool)
)


clinician_model_feature_validation_df[
    "mcmed_selected"
] = (
    clinician_model_feature_validation_df[
        "mcmed_selected"
    ]
    .astype(bool)
)



for dataset_prefix in [
    "mimic",
    "mcmed",
]:

    dataset_category_column = (
        f"{dataset_prefix}_category"
    )


    if dataset_category_column in (
        clinician_model_feature_validation_df.columns
    ):

        category_mismatch = (
            clinician_model_feature_validation_df[
                dataset_category_column
            ]
            .astype(str)
            !=
            clinician_model_feature_validation_df[
                "model_category"
            ]
            .astype(str)
        )


        if category_mismatch.any():

            display(
                clinician_model_feature_validation_df.loc[
                    category_mismatch,
                    [
                        "feature",
                        "model_category",
                        dataset_category_column,
                    ],
                ]
                .head(
                    50
                )
            )


        assert not category_mismatch.any(), (
            f"{dataset_prefix} SHAP category assignments "
            "do not align with clinician-validation "
            "categories."
        )



mimic_selected_mask = (
    clinician_model_feature_validation_df[
        "mimic_selected"
    ]
)


mcmed_selected_mask = (
    clinician_model_feature_validation_df[
        "mcmed_selected"
    ]
)


selected_both_mask = (
    mimic_selected_mask
    &
    mcmed_selected_mask
)


mimic_only_mask = (
    mimic_selected_mask
    &
    ~mcmed_selected_mask
)


mcmed_only_mask = (
    ~mimic_selected_mask
    &
    mcmed_selected_mask
)


neither_mask = (
    ~mimic_selected_mask
    &
    ~mcmed_selected_mask
)


clinician_model_feature_validation_df[
    "model_replication_status"
] = np.select(
    [
        selected_both_mask,
        mimic_only_mask,
        mcmed_only_mask,
        neither_mask,
    ],
    [
        "BOTH",
        "MIMIC-IV only",
        "MC-MED only",
        "Neither",
    ],
    default="ERROR",
)


assert not (
    clinician_model_feature_validation_df[
        "model_replication_status"
    ]
    ==
    "ERROR"
).any()


clinician_model_feature_validation_df[
    "selected_in_both"
] = (
    selected_both_mask
)


clinician_model_feature_validation_df[
    "selected_in_exactly_one_dataset"
] = (
    mimic_only_mask
    |
    mcmed_only_mask
)


clinician_model_feature_validation_df[
    "selected_in_any_dataset"
] = (
    mimic_selected_mask
    |
    mcmed_selected_mask
)



clinician_model_feature_validation_df[
    "cross_dataset_min_stability_pct"
] = np.nan


clinician_model_feature_validation_df[
    "cross_dataset_mean_stability_pct"
] = np.nan


clinician_model_feature_validation_df.loc[
    selected_both_mask,
    "cross_dataset_min_stability_pct",
] = (
    clinician_model_feature_validation_df.loc[
        selected_both_mask,
        [
            "mimic_stability_pct",
            "mcmed_stability_pct",
        ],
    ]
    .min(
        axis=1
    )
)


clinician_model_feature_validation_df.loc[
    selected_both_mask,
    "cross_dataset_mean_stability_pct",
] = (
    clinician_model_feature_validation_df.loc[
        selected_both_mask,
        [
            "mimic_stability_pct",
            "mcmed_stability_pct",
        ],
    ]
    .mean(
        axis=1
    )
)


clinician_model_feature_validation_df[
    "one_dataset_stability_pct"
] = np.select(
    [
        mimic_only_mask,
        mcmed_only_mask,
    ],
    [
        clinician_model_feature_validation_df[
            "mimic_stability_pct"
        ],
        clinician_model_feature_validation_df[
            "mcmed_stability_pct"
        ],
    ],
    default=np.nan,
)



clinician_model_feature_validation_df[
    "direction_agreement"
] = (
    "Not comparable"
)


if (
    "mimic_most_common_pattern_code"
    in
    clinician_model_feature_validation_df.columns
    and
    "mcmed_most_common_pattern_code"
    in
    clinician_model_feature_validation_df.columns
):

    direction_comparable_mask = (
        selected_both_mask
        &
        clinician_model_feature_validation_df[
            "mimic_most_common_pattern_code"
        ].notna()
        &
        clinician_model_feature_validation_df[
            "mcmed_most_common_pattern_code"
        ].notna()
    )


    direction_same_mask = (
        direction_comparable_mask
        &
        (
            clinician_model_feature_validation_df[
                "mimic_most_common_pattern_code"
            ]
            .astype(str)
            ==
            clinician_model_feature_validation_df[
                "mcmed_most_common_pattern_code"
            ]
            .astype(str)
        )
    )


    direction_different_mask = (
        direction_comparable_mask
        &
        ~direction_same_mask
    )


    clinician_model_feature_validation_df.loc[
        direction_same_mask,
        "direction_agreement",
    ] = (
        "Agreement"
    )


    clinician_model_feature_validation_df.loc[
        direction_different_mask,
        "direction_agreement",
    ] = (
        "Disagreement"
    )



clinician_vote_n = (
    clinician_model_feature_validation_df[
        "clinician_vote_n"
    ]
    .astype(int)
)


clinician_unselected_mask = (
    clinician_vote_n
    ==
    0
)


clinician_underrecognized_mask = (
    clinician_vote_n
    .between(
        1,
        3,
    )
)


clinician_divided_mask = (
    clinician_vote_n
    .between(
        4,
        5,
    )
)


clinician_supported_mask = (
    clinician_vote_n
    >=
    6
)


clinician_model_feature_validation_df[
    "clinician_operational_tag"
] = np.select(
    [
        clinician_unselected_mask,
        clinician_underrecognized_mask,
        clinician_divided_mask,
        clinician_supported_mask,
    ],
    [
        "Clinician-unselected",
        "Clinician-underrecognized",
        "Clinician-divided / uncertain",
        "Clinician-supported",
    ],
    default="ERROR",
)


assert not (
    clinician_model_feature_validation_df[
        "clinician_operational_tag"
    ]
    ==
    "ERROR"
).any()



one_dataset_mask = (
    mimic_only_mask
    |
    mcmed_only_mask
)


clinician_model_feature_validation_df[
    "clinical_validation_tag"
] = np.select(
    [
        selected_both_mask
        &
        clinician_unselected_mask,

        selected_both_mask
        &
        clinician_underrecognized_mask,

        selected_both_mask
        &
        clinician_divided_mask,

        selected_both_mask
        &
        clinician_supported_mask,

        one_dataset_mask
        &
        clinician_unselected_mask,

        one_dataset_mask
        &
        clinician_underrecognized_mask,

        one_dataset_mask
        &
        clinician_divided_mask,

        one_dataset_mask
        &
        clinician_supported_mask,

        neither_mask
        &
        clinician_supported_mask,

        neither_mask
        &
        clinician_divided_mask,

        neither_mask
        &
        clinician_underrecognized_mask,

        neither_mask
        &
        clinician_unselected_mask,
    ],
    [
        "Replicated clinician-unselected model signal",

        "Replicated clinician-underrecognized model signal",

        "Replicated model signal with divided clinician opinion",

        "Replicated clinician-model concordant signal",

        "One-dataset clinician-unselected model signal",

        "One-dataset clinician-underrecognized model signal",

        "One-dataset model signal with divided clinician opinion",

        "One-dataset clinician-model concordant signal",

        "Clinician-supported feature not robustly selected in either dataset",

        "Clinician-divided feature not robustly selected in either dataset",

        "Low-support feature not robustly selected in either dataset",

        "Clinician-unselected feature not robustly selected in either dataset",
    ],
    default="ERROR",
)


assert not (
    clinician_model_feature_validation_df[
        "clinical_validation_tag"
    ]
    ==
    "ERROR"
).any()



clinician_model_feature_validation_df[
    "finding_priority"
] = np.select(
    [
        selected_both_mask
        &
        (
            clinician_unselected_mask
            |
            clinician_underrecognized_mask
        ),

        selected_both_mask
        &
        clinician_divided_mask,

        selected_both_mask
        &
        clinician_supported_mask,

        one_dataset_mask
        &
        (
            clinician_unselected_mask
            |
            clinician_underrecognized_mask
        ),

        one_dataset_mask
        &
        clinician_divided_mask,

        one_dataset_mask
        &
        clinician_supported_mask,

        neither_mask
        &
        clinician_supported_mask,
    ],
    [
        "Primary discovery finding",
        "Primary contextual finding",
        "Primary concordance finding",
        "Secondary discovery finding",
        "Secondary contextual finding",
        "Secondary concordance finding",
        "Reverse-discrepancy finding",
    ],
    default="Descriptive",
)



clinician_model_feature_validation_df[
    "primary_replicated_clinician_unselected"
] = (
    selected_both_mask
    &
    clinician_unselected_mask
)


clinician_model_feature_validation_df[
    "primary_replicated_clinician_underrecognized"
] = (
    selected_both_mask
    &
    clinician_underrecognized_mask
)


clinician_model_feature_validation_df[
    "primary_replicated_discovery_candidate"
] = (
    clinician_model_feature_validation_df[
        "primary_replicated_clinician_unselected"
    ]
    |
    clinician_model_feature_validation_df[
        "primary_replicated_clinician_underrecognized"
    ]
)


clinician_model_feature_validation_df[
    "primary_replicated_divided_opinion"
] = (
    selected_both_mask
    &
    clinician_divided_mask
)


clinician_model_feature_validation_df[
    "replicated_clinician_model_concordant"
] = (
    selected_both_mask
    &
    clinician_supported_mask
)


clinician_model_feature_validation_df[
    "secondary_one_dataset_clinician_unselected"
] = (
    one_dataset_mask
    &
    clinician_unselected_mask
)


clinician_model_feature_validation_df[
    "secondary_one_dataset_clinician_underrecognized"
] = (
    one_dataset_mask
    &
    clinician_underrecognized_mask
)


clinician_model_feature_validation_df[
    "secondary_one_dataset_discovery_candidate"
] = (
    clinician_model_feature_validation_df[
        "secondary_one_dataset_clinician_unselected"
    ]
    |
    clinician_model_feature_validation_df[
        "secondary_one_dataset_clinician_underrecognized"
    ]
)


clinician_model_feature_validation_df[
    "clinician_supported_not_robust_in_either"
] = (
    neither_mask
    &
    clinician_supported_mask
)



cross_dataset_consistency_check = (
    "NOT AVAILABLE"
)


if (
    "nb4_cross_dataset_feature_overlap"
    in globals()
    and
    {
        "feature",
        "selection_overlap",
    }
    .issubset(
        nb4_cross_dataset_feature_overlap.columns
    )
):

    previous_overlap_df = (
        nb4_cross_dataset_feature_overlap[
            [
                "feature",
                "selection_overlap",
            ]
        ]
        .copy()
    )


    current_selected_union_df = (
        clinician_model_feature_validation_df.loc[
            clinician_model_feature_validation_df[
                "model_replication_status"
            ]
            !=
            "Neither",
            [
                "feature",
                "model_replication_status",
            ],
        ]
        .copy()
    )


    previous_overlap_df = (
        previous_overlap_df
        .rename(
            columns={
                "selection_overlap": (
                    "model_replication_status"
                )
            }
        )
    )


    previous_overlap_df = (
        previous_overlap_df
        .sort_values(
            "feature"
        )
        .reset_index(
            drop=True
        )
    )


    current_selected_union_df = (
        current_selected_union_df
        .sort_values(
            "feature"
        )
        .reset_index(
            drop=True
        )
    )


    assert previous_overlap_df.equals(
        current_selected_union_df
    )


    cross_dataset_consistency_check = (
        "PASSED"
    )



primary_replicated_clinician_unselected_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "primary_replicated_clinician_unselected"
        ]
    ]
    .copy()
)


primary_replicated_clinician_underrecognized_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "primary_replicated_clinician_underrecognized"
        ]
    ]
    .copy()
)


primary_replicated_discovery_candidates_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "primary_replicated_discovery_candidate"
        ]
    ]
    .copy()
)


primary_replicated_divided_opinion_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "primary_replicated_divided_opinion"
        ]
    ]
    .copy()
)


replicated_clinician_model_concordant_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "replicated_clinician_model_concordant"
        ]
    ]
    .copy()
)


secondary_one_dataset_clinician_unselected_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "secondary_one_dataset_clinician_unselected"
        ]
    ]
    .copy()
)


secondary_one_dataset_clinician_underrecognized_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "secondary_one_dataset_clinician_underrecognized"
        ]
    ]
    .copy()
)


secondary_one_dataset_discovery_candidates_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "secondary_one_dataset_discovery_candidate"
        ]
    ]
    .copy()
)


clinician_supported_not_robust_df = (
    clinician_model_feature_validation_df.loc[
        clinician_model_feature_validation_df[
            "clinician_supported_not_robust_in_either"
        ]
    ]
    .copy()
)



for dataframe_name in [
    "primary_replicated_clinician_unselected_df",
    "primary_replicated_clinician_underrecognized_df",
    "primary_replicated_discovery_candidates_df",
    "primary_replicated_divided_opinion_df",
    "replicated_clinician_model_concordant_df",
]:

    dataframe_object = globals()[
        dataframe_name
    ]


    if not dataframe_object.empty:

        dataframe_object = (
            dataframe_object
            .sort_values(
                [
                    "model_category",
                    "cross_dataset_min_stability_pct",
                    "cross_dataset_mean_stability_pct",
                ],
                ascending=[
                    True,
                    False,
                    False,
                ],
            )
            .reset_index(
                drop=True
            )
        )


    globals()[
        dataframe_name
    ] = dataframe_object


for dataframe_name in [
    "secondary_one_dataset_clinician_unselected_df",
    "secondary_one_dataset_clinician_underrecognized_df",
    "secondary_one_dataset_discovery_candidates_df",
]:

    dataframe_object = globals()[
        dataframe_name
    ]


    if not dataframe_object.empty:

        dataframe_object = (
            dataframe_object
            .sort_values(
                [
                    "model_category",
                    "one_dataset_stability_pct",
                ],
                ascending=[
                    True,
                    False,
                ],
            )
            .reset_index(
                drop=True
            )
        )


    globals()[
        dataframe_name
    ] = dataframe_object



feature_summary_records = []


for category_name in (
    CLINICAL_VALIDATION_CATEGORY_ORDER
):

    category_df = (
        clinician_model_feature_validation_df.loc[
            clinician_model_feature_validation_df[
                "model_category"
            ]
            ==
            category_name
        ]
    )


    feature_summary_records.append(
        {
            "model_category": (
                category_name
            ),

            "n_features": int(
                len(
                    category_df
                )
            ),

            "n_model_selected_both": int(
                category_df[
                    "selected_in_both"
                ].sum()
            ),

            "n_model_selected_mimic_only": int(
                (
                    category_df[
                        "model_replication_status"
                    ]
                    ==
                    "MIMIC-IV only"
                ).sum()
            ),

            "n_model_selected_mcmed_only": int(
                (
                    category_df[
                        "model_replication_status"
                    ]
                    ==
                    "MC-MED only"
                ).sum()
            ),

            "n_model_selected_neither": int(
                (
                    category_df[
                        "model_replication_status"
                    ]
                    ==
                    "Neither"
                ).sum()
            ),

            "n_clinician_supported_6plus": int(
                (
                    category_df[
                        "clinician_vote_n"
                    ]
                    >=
                    6
                ).sum()
            ),

            "n_clinician_divided_4_5": int(
                category_df[
                    "clinician_vote_n"
                ]
                .between(
                    4,
                    5,
                )
                .sum()
            ),

            "n_clinician_underrecognized_1_3": int(
                category_df[
                    "clinician_vote_n"
                ]
                .between(
                    1,
                    3,
                )
                .sum()
            ),

            "n_clinician_unselected_0": int(
                (
                    category_df[
                        "clinician_vote_n"
                    ]
                    ==
                    0
                ).sum()
            ),

            "n_primary_replicated_unselected": int(
                category_df[
                    "primary_replicated_clinician_unselected"
                ].sum()
            ),

            "n_primary_replicated_underrecognized": int(
                category_df[
                    "primary_replicated_clinician_underrecognized"
                ].sum()
            ),

            "n_primary_replicated_discovery_total": int(
                category_df[
                    "primary_replicated_discovery_candidate"
                ].sum()
            ),

            "n_replicated_divided_opinion": int(
                category_df[
                    "primary_replicated_divided_opinion"
                ].sum()
            ),

            "n_replicated_clinician_model_concordant": int(
                category_df[
                    "replicated_clinician_model_concordant"
                ].sum()
            ),

            "n_secondary_one_dataset_unselected": int(
                category_df[
                    "secondary_one_dataset_clinician_unselected"
                ].sum()
            ),

            "n_secondary_one_dataset_underrecognized": int(
                category_df[
                    "secondary_one_dataset_clinician_underrecognized"
                ].sum()
            ),

            "n_secondary_one_dataset_discovery_total": int(
                category_df[
                    "secondary_one_dataset_discovery_candidate"
                ].sum()
            ),

            "n_clinician_supported_not_robust_either": int(
                category_df[
                    "clinician_supported_not_robust_in_either"
                ].sum()
            ),
        }
    )


clinician_model_feature_summary_by_category_df = (
    pd.DataFrame(
        feature_summary_records
    )
)



clinician_model_feature_tag_counts_df = (
    clinician_model_feature_validation_df
    .groupby(
        [
            "model_category",
            "model_replication_status",
            "clinician_consensus_band",
            "clinical_validation_tag",
            "finding_priority",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": (
                "n_features"
            )
        }
    )
)



feature_model_columns_for_group = [
    "feature",
    "mimic_selected",
    "mcmed_selected",
    "selected_in_both",
    "selected_in_exactly_one_dataset",
    "selected_in_any_dataset",
    "model_replication_status",
    "cross_dataset_min_stability_pct",
    "one_dataset_stability_pct",
]


group_membership_validation_df = (
    clinician_feature_membership_df
    .merge(
        clinician_model_feature_validation_df[
            feature_model_columns_for_group
        ],
        on="feature",
        how="left",
        validate="many_to_one",
    )
)


assert len(
    group_membership_validation_df
) == len(
    clinician_feature_membership_df
)


assert group_membership_validation_df[
    "model_replication_status"
].notna().all()


group_membership_validation_df[
    "group_feature_both"
] = (
    group_membership_validation_df[
        "model_replication_status"
    ]
    ==
    "BOTH"
)


group_membership_validation_df[
    "group_feature_mimic_only"
] = (
    group_membership_validation_df[
        "model_replication_status"
    ]
    ==
    "MIMIC-IV only"
)


group_membership_validation_df[
    "group_feature_mcmed_only"
] = (
    group_membership_validation_df[
        "model_replication_status"
    ]
    ==
    "MC-MED only"
)


group_membership_validation_df[
    "group_feature_neither"
] = (
    group_membership_validation_df[
        "model_replication_status"
    ]
    ==
    "Neither"
)


group_membership_validation_df[
    "group_feature_any"
] = (
    group_membership_validation_df[
        "model_replication_status"
    ]
    !=
    "Neither"
)



group_model_summary_df = (
    group_membership_validation_df
    .groupby(
        "clinician_group_key",
        as_index=False,
    )
    .agg(
        n_feature_memberships=(
            "feature",
            "size",
        ),

        n_unique_features=(
            "feature",
            "nunique",
        ),

        n_selected_both=(
            "group_feature_both",
            "sum",
        ),

        n_selected_mimic_only=(
            "group_feature_mimic_only",
            "sum",
        ),

        n_selected_mcmed_only=(
            "group_feature_mcmed_only",
            "sum",
        ),

        n_selected_neither=(
            "group_feature_neither",
            "sum",
        ),

        n_selected_any_dataset=(
            "group_feature_any",
            "sum",
        ),

        max_replicated_min_stability_pct=(
            "cross_dataset_min_stability_pct",
            "max",
        ),

        max_one_dataset_stability_pct=(
            "one_dataset_stability_pct",
            "max",
        ),
    )
)


clinician_model_group_validation_df = (
    clinician_group_consensus_df
    .merge(
        group_model_summary_df,
        on="clinician_group_key",
        how="left",
        validate="one_to_one",
    )
)


assert len(
    clinician_model_group_validation_df
) == len(
    clinician_group_consensus_df
)


assert len(
    clinician_model_group_validation_df
) == 75



clinician_model_group_validation_df[
    "pct_features_selected_both"
] = (
    100.0
    *
    clinician_model_group_validation_df[
        "n_selected_both"
    ]
    /
    clinician_model_group_validation_df[
        "n_unique_features"
    ]
)


clinician_model_group_validation_df[
    "pct_features_selected_any_dataset"
] = (
    100.0
    *
    clinician_model_group_validation_df[
        "n_selected_any_dataset"
    ]
    /
    clinician_model_group_validation_df[
        "n_unique_features"
    ]
)



group_contains_both_mask = (
    clinician_model_group_validation_df[
        "n_selected_both"
    ]
    >
    0
)


group_contains_one_dataset_mask = (
    ~group_contains_both_mask
    &
    (
        (
            clinician_model_group_validation_df[
                "n_selected_mimic_only"
            ]
            >
            0
        )
        |
        (
            clinician_model_group_validation_df[
                "n_selected_mcmed_only"
            ]
            >
            0
        )
    )
)


group_contains_none_mask = (
    ~group_contains_both_mask
    &
    ~group_contains_one_dataset_mask
)


clinician_model_group_validation_df[
    "group_model_evidence_status"
] = np.select(
    [
        group_contains_both_mask,
        group_contains_one_dataset_mask,
        group_contains_none_mask,
    ],
    [
        "Contains BOTH-dataset robust feature(s)",
        "Contains one-dataset robust feature(s)",
        "No robustly selected feature",
    ],
    default="ERROR",
)



group_vote_n = (
    clinician_model_group_validation_df[
        "clinician_vote_n"
    ]
    .astype(int)
)


group_unselected_mask = (
    group_vote_n
    ==
    0
)


group_underrecognized_mask = (
    group_vote_n
    .between(
        1,
        3,
    )
)


group_divided_mask = (
    group_vote_n
    .between(
        4,
        5,
    )
)


group_supported_mask = (
    group_vote_n
    >=
    6
)



clinician_model_group_validation_df[
    "group_validation_tag"
] = np.select(
    [
        group_contains_both_mask
        &
        group_unselected_mask,

        group_contains_both_mask
        &
        group_underrecognized_mask,

        group_contains_both_mask
        &
        group_divided_mask,

        group_contains_both_mask
        &
        group_supported_mask,

        group_contains_one_dataset_mask
        &
        group_unselected_mask,

        group_contains_one_dataset_mask
        &
        group_underrecognized_mask,

        group_contains_one_dataset_mask
        &
        group_divided_mask,

        group_contains_one_dataset_mask
        &
        group_supported_mask,

        group_contains_none_mask
        &
        group_supported_mask,

        group_contains_none_mask
        &
        group_divided_mask,

        group_contains_none_mask
        &
        group_underrecognized_mask,

        group_contains_none_mask
        &
        group_unselected_mask,
    ],
    [
        "Replicated clinician-unselected group",

        "Replicated clinician-underrecognized group",

        "Replicated model signal with divided clinician opinion",

        "Replicated clinician-model concordant group",

        "One-dataset clinician-unselected group",

        "One-dataset clinician-underrecognized group",

        "One-dataset group with divided clinician opinion",

        "One-dataset clinician-model concordant group",

        "Clinician-supported group not robustly captured by either dataset",

        "Clinician-divided group not robustly captured by either dataset",

        "Low-support group not robustly captured by either dataset",

        "Clinician-unselected group not robustly captured by either dataset",
    ],
    default="ERROR",
)


assert not (
    clinician_model_group_validation_df[
        "group_validation_tag"
    ]
    ==
    "ERROR"
).any()



clinician_model_group_validation_df[
    "group_finding_priority"
] = np.select(
    [
        group_contains_both_mask
        &
        (
            group_unselected_mask
            |
            group_underrecognized_mask
        ),

        group_contains_both_mask
        &
        group_divided_mask,

        group_contains_both_mask
        &
        group_supported_mask,

        group_contains_one_dataset_mask
        &
        (
            group_unselected_mask
            |
            group_underrecognized_mask
        ),

        group_contains_one_dataset_mask
        &
        group_divided_mask,

        group_contains_one_dataset_mask
        &
        group_supported_mask,

        group_contains_none_mask
        &
        group_supported_mask,
    ],
    [
        "Primary discovery finding",
        "Primary contextual finding",
        "Primary concordance finding",
        "Secondary discovery finding",
        "Secondary contextual finding",
        "Secondary concordance finding",
        "Reverse-discrepancy finding",
    ],
    default="Descriptive",
)



clinician_model_group_validation_df[
    "primary_replicated_clinician_unselected_group"
] = (
    group_contains_both_mask
    &
    group_unselected_mask
)


clinician_model_group_validation_df[
    "primary_replicated_clinician_underrecognized_group"
] = (
    group_contains_both_mask
    &
    group_underrecognized_mask
)


clinician_model_group_validation_df[
    "primary_replicated_discovery_group"
] = (
    clinician_model_group_validation_df[
        "primary_replicated_clinician_unselected_group"
    ]
    |
    clinician_model_group_validation_df[
        "primary_replicated_clinician_underrecognized_group"
    ]
)


clinician_model_group_validation_df[
    "replicated_clinician_model_concordant_group"
] = (
    group_contains_both_mask
    &
    group_supported_mask
)


clinician_model_group_validation_df[
    "secondary_one_dataset_discovery_group"
] = (
    group_contains_one_dataset_mask
    &
    (
        group_unselected_mask
        |
        group_underrecognized_mask
    )
)


clinician_model_group_validation_df[
    "clinician_supported_group_not_robust_either"
] = (
    group_contains_none_mask
    &
    group_supported_mask
)



group_summary_records = []


for domain_name in (
    CLINICAL_VALIDATION_DOMAIN_ORDER
):

    domain_df = (
        clinician_model_group_validation_df.loc[
            clinician_model_group_validation_df[
                "clinician_domain"
            ]
            ==
            domain_name
        ]
    )


    group_summary_records.append(
        {
            "clinician_domain": (
                domain_name
            ),

            "n_groups": int(
                len(
                    domain_df
                )
            ),

            "n_groups_clinician_supported_6plus": int(
                (
                    domain_df[
                        "clinician_vote_n"
                    ]
                    >=
                    6
                ).sum()
            ),

            "n_groups_divided_4_5": int(
                domain_df[
                    "clinician_vote_n"
                ]
                .between(
                    4,
                    5,
                )
                .sum()
            ),

            "n_groups_underrecognized_1_3": int(
                domain_df[
                    "clinician_vote_n"
                ]
                .between(
                    1,
                    3,
                )
                .sum()
            ),

            "n_groups_unselected_0": int(
                (
                    domain_df[
                        "clinician_vote_n"
                    ]
                    ==
                    0
                ).sum()
            ),

            "n_groups_with_both_dataset_signal": int(
                (
                    domain_df[
                        "n_selected_both"
                    ]
                    >
                    0
                ).sum()
            ),

            "n_primary_replicated_discovery_groups": int(
                domain_df[
                    "primary_replicated_discovery_group"
                ].sum()
            ),

            "n_replicated_concordant_groups": int(
                domain_df[
                    "replicated_clinician_model_concordant_group"
                ].sum()
            ),

            "n_secondary_one_dataset_discovery_groups": int(
                domain_df[
                    "secondary_one_dataset_discovery_group"
                ].sum()
            ),

            "n_clinician_supported_groups_not_robust_either": int(
                domain_df[
                    "clinician_supported_group_not_robust_either"
                ].sum()
            ),
        }
    )


clinician_model_group_summary_by_domain_df = (
    pd.DataFrame(
        group_summary_records
    )
)



CLINICIAN_MODEL_COMPARISON_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook4"
    / "clinical_validation"
    / "clinician_model_comparison"
)


FEATURE_LEVEL_DIR = (
    CLINICIAN_MODEL_COMPARISON_DIR
    / "feature_level"
)


FEATURE_CATEGORY_DIR = (
    FEATURE_LEVEL_DIR
    / "categories"
)


GROUP_LEVEL_DIR = (
    CLINICIAN_MODEL_COMPARISON_DIR
    / "group_level"
)


GROUP_DOMAIN_DIR = (
    GROUP_LEVEL_DIR
    / "domains"
)


PRIMARY_FINDINGS_DIR = (
    CLINICIAN_MODEL_COMPARISON_DIR
    / "primary_findings"
)


SECONDARY_FINDINGS_DIR = (
    CLINICIAN_MODEL_COMPARISON_DIR
    / "secondary_findings"
)


for output_directory in [
    CLINICIAN_MODEL_COMPARISON_DIR,
    FEATURE_LEVEL_DIR,
    FEATURE_CATEGORY_DIR,
    GROUP_LEVEL_DIR,
    GROUP_DOMAIN_DIR,
    PRIMARY_FINDINGS_DIR,
    SECONDARY_FINDINGS_DIR,
]:

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )



def nb4_clinical_safe_name(
    value,
):

    return (
        str(
            value
        )
        .lower()
        .replace(
            " / ",
            "_"
        )
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )



clinician_model_feature_validation_df.to_csv(
    FEATURE_LEVEL_DIR
    / "clinician_model_feature_validation_all_2495.csv",
    index=False,
)


clinician_model_feature_validation_df.to_parquet(
    FEATURE_LEVEL_DIR
    / "clinician_model_feature_validation_all_2495.parquet",
    index=False,
    engine="pyarrow",
)


clinician_model_feature_summary_by_category_df.to_csv(
    FEATURE_LEVEL_DIR
    / "clinician_model_feature_summary_by_category.csv",
    index=False,
)


clinician_model_feature_tag_counts_df.to_csv(
    FEATURE_LEVEL_DIR
    / "clinician_model_feature_tag_counts.csv",
    index=False,
)



primary_replicated_clinician_unselected_df.to_csv(
    PRIMARY_FINDINGS_DIR
    / "primary_replicated_clinician_unselected_features.csv",
    index=False,
)


primary_replicated_clinician_underrecognized_df.to_csv(
    PRIMARY_FINDINGS_DIR
    / "primary_replicated_clinician_underrecognized_features.csv",
    index=False,
)


primary_replicated_discovery_candidates_df.to_csv(
    PRIMARY_FINDINGS_DIR
    / "primary_replicated_discovery_candidates_0_to_3_of_10.csv",
    index=False,
)


primary_replicated_divided_opinion_df.to_csv(
    PRIMARY_FINDINGS_DIR
    / "replicated_divided_clinician_opinion_features.csv",
    index=False,
)


replicated_clinician_model_concordant_df.to_csv(
    PRIMARY_FINDINGS_DIR
    / "replicated_clinician_model_concordant_features.csv",
    index=False,
)


clinician_supported_not_robust_df.to_csv(
    PRIMARY_FINDINGS_DIR
    / "clinician_supported_not_robust_in_either_dataset.csv",
    index=False,
)



secondary_one_dataset_clinician_unselected_df.to_csv(
    SECONDARY_FINDINGS_DIR
    / "secondary_one_dataset_clinician_unselected_features.csv",
    index=False,
)


secondary_one_dataset_clinician_underrecognized_df.to_csv(
    SECONDARY_FINDINGS_DIR
    / "secondary_one_dataset_clinician_underrecognized_features.csv",
    index=False,
)


secondary_one_dataset_discovery_candidates_df.to_csv(
    SECONDARY_FINDINGS_DIR
    / "secondary_one_dataset_discovery_candidates_0_to_3_of_10.csv",
    index=False,
)



for category_name in (
    CLINICAL_VALIDATION_CATEGORY_ORDER
):

    safe_category = (
        nb4_clinical_safe_name(
            category_name
        )
    )


    category_df = (
        clinician_model_feature_validation_df.loc[
            clinician_model_feature_validation_df[
                "model_category"
            ]
            ==
            category_name
        ]
        .copy()
    )


    category_df.to_csv(
        FEATURE_CATEGORY_DIR
        /
        f"{safe_category}_all_features.csv",
        index=False,
    )


    category_df.loc[
        category_df[
            "primary_replicated_discovery_candidate"
        ]
    ].to_csv(
        FEATURE_CATEGORY_DIR
        /
        f"{safe_category}_primary_replicated_discovery.csv",
        index=False,
    )


    category_df.loc[
        category_df[
            "secondary_one_dataset_discovery_candidate"
        ]
    ].to_csv(
        FEATURE_CATEGORY_DIR
        /
        f"{safe_category}_secondary_one_dataset_discovery.csv",
        index=False,
    )


    category_df.loc[
        category_df[
            "replicated_clinician_model_concordant"
        ]
    ].to_csv(
        FEATURE_CATEGORY_DIR
        /
        f"{safe_category}_replicated_concordant.csv",
        index=False,
    )


    category_df.loc[
        category_df[
            "clinician_supported_not_robust_in_either"
        ]
    ].to_csv(
        FEATURE_CATEGORY_DIR
        /
        f"{safe_category}_clinician_supported_not_robust.csv",
        index=False,
    )



clinician_model_group_validation_df.to_csv(
    GROUP_LEVEL_DIR
    / "clinician_model_group_validation_all_groups.csv",
    index=False,
)


clinician_model_group_validation_df.to_parquet(
    GROUP_LEVEL_DIR
    / "clinician_model_group_validation_all_groups.parquet",
    index=False,
    engine="pyarrow",
)


clinician_model_group_summary_by_domain_df.to_csv(
    GROUP_LEVEL_DIR
    / "clinician_model_group_summary_by_domain.csv",
    index=False,
)


group_membership_validation_df.to_csv(
    GROUP_LEVEL_DIR
    / "clinician_group_feature_model_membership_validation.csv",
    index=False,
)



for domain_name in (
    CLINICAL_VALIDATION_DOMAIN_ORDER
):

    safe_domain = (
        nb4_clinical_safe_name(
            domain_name
        )
    )


    domain_df = (
        clinician_model_group_validation_df.loc[
            clinician_model_group_validation_df[
                "clinician_domain"
            ]
            ==
            domain_name
        ]
        .copy()
    )


    domain_df.to_csv(
        GROUP_DOMAIN_DIR
        /
        f"{safe_domain}_group_validation.csv",
        index=False,
    )



clinician_model_group_validation_df.loc[
    clinician_model_group_validation_df[
        "primary_replicated_discovery_group"
    ]
].to_csv(
    PRIMARY_FINDINGS_DIR
    / "primary_replicated_discovery_clinician_groups.csv",
    index=False,
)


clinician_model_group_validation_df.loc[
    clinician_model_group_validation_df[
        "replicated_clinician_model_concordant_group"
    ]
].to_csv(
    PRIMARY_FINDINGS_DIR
    / "replicated_clinician_model_concordant_groups.csv",
    index=False,
)


clinician_model_group_validation_df.loc[
    clinician_model_group_validation_df[
        "secondary_one_dataset_discovery_group"
    ]
].to_csv(
    SECONDARY_FINDINGS_DIR
    / "secondary_one_dataset_discovery_clinician_groups.csv",
    index=False,
)


clinician_model_group_validation_df.loc[
    clinician_model_group_validation_df[
        "clinician_supported_group_not_robust_either"
    ]
].to_csv(
    PRIMARY_FINDINGS_DIR
    / "clinician_supported_groups_not_robust_in_either_dataset.csv",
    index=False,
)



clinical_validation_methodology_audit_df = pd.DataFrame(
    [
        {
            "n_model_features": (
                2495
            ),

            "n_clinicians": (
                10
            ),

            "n_clinician_groups": int(
                len(
                    clinician_model_group_validation_df
                )
            ),

            "model_selected_definition": (
                "times_selected > 0 in 81-combination "
                "SHAP stability analysis"
            ),

            "below_p25_selected_features_included": (
                True
            ),

            "clinician_unselected_definition": (
                "0/10"
            ),

            "clinician_underrecognized_definition": (
                "1-3/10"
            ),

            "clinician_divided_definition": (
                "4-5/10"
            ),

            "clinician_supported_definition": (
                "6-10/10"
            ),

            "unanimous_important_definition": (
                "10/10"
            ),

            "strong_consensus_important_definition": (
                "8-9/10"
            ),

            "majority_important_definition": (
                "6-7/10"
            ),

            "primary_discovery_definition": (
                "BOTH datasets + clinician 0-3/10"
            ),

            "secondary_discovery_definition": (
                "exactly one dataset + clinician 0-3/10"
            ),

            "feature_multi_membership_rule": (
                "maximum clinician vote across valid "
                "group memberships"
            ),

            "group_level_consensus_rule": (
                "original clinician group vote"
            ),

            "formal_novelty_claim_made": (
                False
            ),

            "model_refitted": (
                False
            ),

            "recalibrated": (
                False
            ),

            "threshold_changed": (
                False
            ),

            "shap_recalculated": (
                False
            ),

            "cross_dataset_overlap_consistency": (
                cross_dataset_consistency_check
            ),
        }
    ]
)


clinical_validation_methodology_audit_df.to_csv(
    CLINICIAN_MODEL_COMPARISON_DIR
    / "clinical_validation_methodology_audit.csv",
    index=False,
)



assert (
    (
        clinician_model_feature_validation_df[
            "model_replication_status"
        ]
        .value_counts()
        .sum()
    )
    ==
    2495
)


assert (
    clinician_model_feature_validation_df[
        "primary_replicated_discovery_candidate"
    ].sum()
    ==
    (
        clinician_model_feature_validation_df[
            "primary_replicated_clinician_unselected"
        ].sum()
        +
        clinician_model_feature_validation_df[
            "primary_replicated_clinician_underrecognized"
        ].sum()
    )
)


assert (
    clinician_model_feature_validation_df[
        "secondary_one_dataset_discovery_candidate"
    ].sum()
    ==
    (
        clinician_model_feature_validation_df[
            "secondary_one_dataset_clinician_unselected"
        ].sum()
        +
        clinician_model_feature_validation_df[
            "secondary_one_dataset_clinician_underrecognized"
        ].sum()
    )
)


assert clinician_model_group_validation_df[
    "group_validation_tag"
].notna().all()



print(
    "\n=================================================="
)


print(
    "FEATURE-LEVEL CLINICAL VALIDATION SUMMARY"
)


print(
    "=================================================="
)


display(
    clinician_model_feature_summary_by_category_df
)



print(
    "\n=================================================="
)


print(
    "GROUP-LEVEL CLINICAL VALIDATION SUMMARY"
)


print(
    "=================================================="
)


display(
    clinician_model_group_summary_by_domain_df
)



print(
    "\n=================================================="
)


print(
    "PRIMARY DISCOVERY FEATURES"
)


print(
    "Stable in BOTH datasets + clinician support 0–3/10"
)


print(
    "=================================================="
)


print(
    "Clinician-unselected 0/10:",
    len(
        primary_replicated_clinician_unselected_df
    ),
)


print(
    "Clinician-underrecognized 1–3/10:",
    len(
        primary_replicated_clinician_underrecognized_df
    ),
)


print(
    "Combined primary discovery candidates:",
    len(
        primary_replicated_discovery_candidates_df
    ),
)


if not primary_replicated_discovery_candidates_df.empty:

    primary_display_columns = [
        "model_category",
        "feature",
        "clinician_vote_n",
        "clinician_consensus_band",
        "clinical_validation_tag",
        "mimic_importance_tag",
        "mimic_stability_pct",
        "mcmed_importance_tag",
        "mcmed_stability_pct",
        "cross_dataset_min_stability_pct",
        "direction_agreement",
    ]


    primary_display_columns = [
        column_name
        for column_name
        in primary_display_columns
        if column_name
        in primary_replicated_discovery_candidates_df.columns
    ]


    display(
        primary_replicated_discovery_candidates_df[
            primary_display_columns
        ]
        .round(
            3
        )
    )



print(
    "\n=================================================="
)


print(
    "SECONDARY DISCOVERY FEATURES"
)


print(
    "Stable in exactly ONE dataset + clinician 0–3/10"
)


print(
    "=================================================="
)


print(
    "Clinician-unselected 0/10:",
    len(
        secondary_one_dataset_clinician_unselected_df
    ),
)


print(
    "Clinician-underrecognized 1–3/10:",
    len(
        secondary_one_dataset_clinician_underrecognized_df
    ),
)


print(
    "Combined secondary discovery candidates:",
    len(
        secondary_one_dataset_discovery_candidates_df
    ),
)


if not secondary_one_dataset_discovery_candidates_df.empty:

    secondary_display_columns = [
        "model_category",
        "feature",
        "model_replication_status",
        "clinician_vote_n",
        "clinician_consensus_band",
        "clinical_validation_tag",
        "one_dataset_stability_pct",
    ]


    display(
        secondary_one_dataset_discovery_candidates_df[
            secondary_display_columns
        ]
        .round(
            3
        )
    )



print(
    "\n=================================================="
)


print(
    "REVERSE DISCREPANCY"
)


print(
    "Clinician support >=6/10 but model robust in neither"
)


print(
    "=================================================="
)


print(
    "Features:",
    len(
        clinician_supported_not_robust_df
    ),
)


if not clinician_supported_not_robust_df.empty:

    display(
        clinician_supported_not_robust_df[
            [
                "model_category",
                "feature",
                "clinician_vote_n",
                "clinician_consensus_band",
                "clinical_validation_tag",
            ]
        ]
    )



print(
    "\n=================================================="
)


print(
    "TEN-CLINICIAN × MODEL VALIDATION COMPLETE"
)


print(
    "=================================================="
)


print(
    "\nFeature universe:",
    len(
        clinician_model_feature_validation_df
    ),
)


print(
    "Clinician groups:",
    len(
        clinician_model_group_validation_df
    ),
)


print(
    "\nPRIMARY DISCOVERY"
)


print(
    "BOTH + 0/10:",
    len(
        primary_replicated_clinician_unselected_df
    ),
)


print(
    "BOTH + 1–3/10:",
    len(
        primary_replicated_clinician_underrecognized_df
    ),
)


print(
    "Primary discovery total:",
    len(
        primary_replicated_discovery_candidates_df
    ),
)


print(
    "\nSECONDARY DISCOVERY"
)


print(
    "Exactly one dataset + 0/10:",
    len(
        secondary_one_dataset_clinician_unselected_df
    ),
)


print(
    "Exactly one dataset + 1–3/10:",
    len(
        secondary_one_dataset_clinician_underrecognized_df
    ),
)


print(
    "Secondary discovery total:",
    len(
        secondary_one_dataset_discovery_candidates_df
    ),
)


print(
    "\nREPLICATED CLINICIAN-MODEL CONCORDANCE:",
    len(
        replicated_clinician_model_concordant_df
    ),
)


print(
    "CLINICIAN-SUPPORTED BUT ROBUST IN NEITHER:",
    len(
        clinician_supported_not_robust_df
    ),
)


print(
    "\nCross-dataset overlap consistency:",
    cross_dataset_consistency_check,
)


print(
    "\nBelow-P25 selected features included:",
    "YES",
)


print(
    "Feature-level multi-membership rule:",
    "MAXIMUM clinician vote",
)


print(
    "Group-level vote rule:",
    "ORIGINAL clinician group vote",
)


print(
    "\nFormal novelty claim:",
    "NO — literature/clinical review still required",
)


print(
    "\nModel refit:",
    "NO",
)


print(
    "Recalibration:",
    "NO",
)


print(
    "Threshold change:",
    "NO",
)


print(
    "SHAP recalculation:",
    "NO",
)


print(
    "\nOutputs saved to:"
)


print(
    CLINICIAN_MODEL_COMPARISON_DIR
)


print(
    "\nREADY FOR CATEGORY-WISE INTERPRETATION AND "
    "NOVELTY/LITERATURE REVIEW:",
    "YES",
)

### Detailed interpretation of replicated clinician-unselected model signals

The 28 primary discovery features were robustly selected in both MIMIC-IV and MC-MED while receiving 0/10 clinician votes for importance. These features are treated as replicated clinician-unselected model signals and remain candidates for subsequent literature-based novelty assessment rather than being labelled novel at this stage.

Because selection in both datasets does not imply equal robustness, the 28 features are descriptively prioritised using a conservative cross-dataset ranking. The minimum of the MIMIC-IV and MC-MED stability percentages is used as the principal ranking criterion. Mean cross-dataset stability is used as the first tie-breaker, followed by mean absolute SHAP across the two datasets. This ranking does not alter feature selection or introduce an additional filtering threshold.

Both dataset-specific global SHAP ranks and stability tiers are retained alongside the cross-dataset priority rank. A separate within-category rank is also reported.

Interactive SHAP distributions are generated from the previously frozen grouped SHAP matrices without recalculating SHAP values. All available MIMIC-IV internal-test and MC-MED external-validation rows are used.

All 28 primary discovery features are binary indicators. Interactive SHAP colours therefore represent feature presence:

- red = feature present / high value (1)
- blue = feature absent / low value (0)

The horizontal SHAP axis remains on the native XGBoost margin scale. Positive SHAP values push the prediction toward ADMITTED, while negative SHAP values push the prediction toward HOME.

One interactive figure contains all 28 features, ranked by conservative cross-dataset stability. Additional interactive figures are generated separately for each represented clinical category. Each figure provides controls to switch between MIMIC-IV and MC-MED while retaining the same feature order and horizontal SHAP scale to facilitate direct cross-dataset comparison.

These visualisations are interpretive analyses only. No model fitting, recalibration, threshold modification, SHAP recalculation or predictive feature reselection is performed.

In [ ]:



PRIMARY_FINAL_N = 28


required_primary28_objects = [
    "PROJECT_ROOT",
    "LOCKED_RAW_FEATURE_ORDER",
    "RAW_FEATURE_POSITION",
    "mimic_shap_values",
    "mimic_feature_values",
    "mcmed_shap_values",
    "mcmed_feature_values",
    "primary_replicated_clinician_unselected_df",
    "CLINICIAN_MODEL_COMPARISON_DIR",
]


missing_primary28_objects = [
    object_name
    for object_name
    in required_primary28_objects
    if object_name not in globals()
]


if missing_primary28_objects:

    raise RuntimeError(
        "Run the completed post-review adjudication cell "
        "first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_primary28_objects
        )
    )


if "go" not in globals():

    raise RuntimeError(
        "Plotly graph_objects ('go') is not loaded. "
        "Run the centralized Notebook 4 imports cell."
    )


MIMIC_N_ROWS = int(
    np.asarray(
        mimic_shap_values
    ).shape[
        0
    ]
)


MCMED_N_ROWS = int(
    np.asarray(
        mcmed_shap_values
    ).shape[
        0
    ]
)


assert np.asarray(
    mimic_shap_values
).shape[
    1
] == 2495


assert np.asarray(
    mcmed_shap_values
).shape[
    1
] == 2495


assert np.asarray(
    mimic_feature_values
).shape == np.asarray(
    mimic_shap_values
).shape


assert np.asarray(
    mcmed_feature_values
).shape == np.asarray(
    mcmed_shap_values
).shape



primary28_ranked_df = (
    primary_replicated_clinician_unselected_df
    .copy()
)


assert len(
    primary28_ranked_df
) == PRIMARY_FINAL_N, (
    "Expected exactly 28 post-review replicated "
    "clinician-unselected clinical candidates."
)


assert (
    primary28_ranked_df[
        "clinician_vote_n"
    ]
    .astype(int)
    ==
    0
).all()


assert (
    primary28_ranked_df[
        "model_replication_status"
    ]
    ==
    "BOTH"
).all()


assert (
    primary28_ranked_df[
        "selected_in_both"
    ]
).all()


assert primary28_ranked_df[
    "feature"
].is_unique


assert set(
    primary28_ranked_df[
        "feature"
    ]
).issubset(
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
)



assert not (
    primary28_ranked_df[
        "model_category"
    ]
    ==
    "Imaging / radiology"
).any(), (
    "Imaging features are still present in the final 28. "
    "Run the post-review adjudication cell first."
)


PRIMARY28_CATEGORY_ORDER = [
    "PMH ICD title",
    "PMH medication class",
    "PMH medication name",
]


observed_primary28_categories = (
    primary28_ranked_df[
        "model_category"
    ]
    .drop_duplicates()
    .tolist()
)


unexpected_primary28_categories = sorted(
    set(
        observed_primary28_categories
    )
    -
    set(
        PRIMARY28_CATEGORY_ORDER
    )
)


assert not unexpected_primary28_categories, (
    "Unexpected category/categories in final 28:\n"
    +
    "\n".join(
        unexpected_primary28_categories
    )
)



def primary28_fallback_clean_label(
    feature_name,
):

    clean_name = str(
        feature_name
    )


    known_prefixes = [
        "pmh_icd_title__",
        "pmh_med_class__",
        "pmh_med_name__",
    ]


    for prefix in known_prefixes:

        if clean_name.startswith(
            prefix
        ):

            clean_name = clean_name[
                len(
                    prefix
                ):
            ]

            break


    clean_name = (
        clean_name
        .replace(
            "__",
            " "
        )
        .replace(
            "_",
            " "
        )
    )


    clean_name = " ".join(
        clean_name.split()
    )


    clean_name = clean_name.title()


    replacements = {
        "Copd": "COPD",
        "Ace ": "ACE ",
        "Arb": "ARB",
        "Ibd": "IBD",
        "Hmg Coa": "HMG-CoA",
    }


    for old_text, new_text in (
        replacements.items()
    ):

        clean_name = clean_name.replace(
            old_text,
            new_text,
        )


    return clean_name


def primary28_get_display_label(
    row,
):

    for clean_column in [
        "mimic_feature_clean",
        "mcmed_feature_clean",
    ]:

        if (
            clean_column
            in
            primary28_ranked_df.columns
        ):

            clean_value = row.get(
                clean_column
            )


            if (
                pd.notna(
                    clean_value
                )
                and
                str(
                    clean_value
                ).strip()
                !=
                ""
            ):

                return str(
                    clean_value
                ).strip()


    return primary28_fallback_clean_label(
        row[
            "feature"
        ]
    )


primary28_ranked_df[
    "feature_label"
] = (
    primary28_ranked_df
    .apply(
        primary28_get_display_label,
        axis=1,
    )
)



primary28_ranked_df[
    "cross_dataset_mean_stability_pct"
] = (
    primary28_ranked_df[
        [
            "mimic_stability_pct",
            "mcmed_stability_pct",
        ]
    ]
    .mean(
        axis=1
    )
)


primary28_ranked_df[
    "cross_dataset_min_stability_pct"
] = (
    primary28_ranked_df[
        [
            "mimic_stability_pct",
            "mcmed_stability_pct",
        ]
    ]
    .min(
        axis=1
    )
)



if {
    "mimic_mean_abs_shap",
    "mcmed_mean_abs_shap",
}.issubset(
    primary28_ranked_df.columns
):

    primary28_ranked_df[
        "cross_dataset_mean_abs_shap"
    ] = (
        primary28_ranked_df[
            [
                "mimic_mean_abs_shap",
                "mcmed_mean_abs_shap",
            ]
        ]
        .mean(
            axis=1
        )
    )


else:

    primary28_ranked_df[
        "cross_dataset_mean_abs_shap"
    ] = np.nan



primary28_ranked_df[
    "mimic_stability_rank_within_28"
] = (
    primary28_ranked_df[
        "mimic_stability_pct"
    ]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)


primary28_ranked_df[
    "mcmed_stability_rank_within_28"
] = (
    primary28_ranked_df[
        "mcmed_stability_pct"
    ]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)



primary28_ranked_df[
    "cross_dataset_stability_rank"
] = (
    primary28_ranked_df[
        "cross_dataset_min_stability_pct"
    ]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)



primary28_ranked_df[
    "_mean_abs_sort"
] = (
    primary28_ranked_df[
        "cross_dataset_mean_abs_shap"
    ]
    .fillna(
        -np.inf
    )
)


primary28_ranked_df = (
    primary28_ranked_df
    .sort_values(
        [
            "cross_dataset_min_stability_pct",
            "cross_dataset_mean_stability_pct",
            "_mean_abs_sort",
            "feature",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


primary28_ranked_df[
    "cross_dataset_priority_rank"
] = np.arange(
    1,
    PRIMARY_FINAL_N
    +
    1,
    dtype=int,
)


primary28_ranked_df = (
    primary28_ranked_df
    .drop(
        columns=[
            "_mean_abs_sort",
        ]
    )
)



primary28_ranked_df[
    "category_priority_rank"
] = (
    primary28_ranked_df
    .groupby(
        "model_category",
        sort=False,
    )
    .cumcount()
    +
    1
)



primary28_category_summary_df = (
    primary28_ranked_df
    .groupby(
        "model_category",
        as_index=False,
        sort=False,
    )
    .agg(
        n_primary_features=(
            "feature",
            "size",
        ),

        mimic_stability_min=(
            "mimic_stability_pct",
            "min",
        ),

        mimic_stability_median=(
            "mimic_stability_pct",
            "median",
        ),

        mimic_stability_max=(
            "mimic_stability_pct",
            "max",
        ),

        mcmed_stability_min=(
            "mcmed_stability_pct",
            "min",
        ),

        mcmed_stability_median=(
            "mcmed_stability_pct",
            "median",
        ),

        mcmed_stability_max=(
            "mcmed_stability_pct",
            "max",
        ),

        replicated_min_stability_min=(
            "cross_dataset_min_stability_pct",
            "min",
        ),

        replicated_min_stability_median=(
            "cross_dataset_min_stability_pct",
            "median",
        ),

        replicated_min_stability_max=(
            "cross_dataset_min_stability_pct",
            "max",
        ),

        n_direction_agreement=(
            "direction_agreement",
            lambda values: int(
                (
                    values
                    ==
                    "Agreement"
                ).sum()
            ),
        ),
    )
)


primary28_category_summary_df[
    "pct_of_primary_28"
] = (
    100.0
    *
    primary28_category_summary_df[
        "n_primary_features"
    ]
    /
    PRIMARY_FINAL_N
)


primary28_category_order = [
    category_name
    for category_name
    in PRIMARY28_CATEGORY_ORDER
    if category_name
    in set(
        primary28_ranked_df[
            "model_category"
        ]
    )
]


primary28_category_order_lookup = {
    category_name: category_position
    for category_position, category_name
    in enumerate(
        primary28_category_order
    )
}


primary28_category_summary_df[
    "_category_order"
] = (
    primary28_category_summary_df[
        "model_category"
    ]
    .map(
        primary28_category_order_lookup
    )
)


primary28_category_summary_df = (
    primary28_category_summary_df
    .sort_values(
        "_category_order"
    )
    .drop(
        columns=[
            "_category_order",
        ]
    )
    .reset_index(
        drop=True
    )
)



primary28_summary_columns = [
    "cross_dataset_priority_rank",
    "cross_dataset_stability_rank",
    "category_priority_rank",
    "model_category",
    "feature_label",
    "feature",
    "clinician_vote_n",
    "mimic_importance_tag",
    "mimic_stability_pct",
    "mimic_stability_rank_within_28",
    "mcmed_importance_tag",
    "mcmed_stability_pct",
    "mcmed_stability_rank_within_28",
    "cross_dataset_min_stability_pct",
    "cross_dataset_mean_stability_pct",
    "mimic_global_rank",
    "mcmed_global_rank",
    "mimic_mean_abs_shap",
    "mcmed_mean_abs_shap",
    "cross_dataset_mean_abs_shap",
    "direction_agreement",
    "mimic_most_common_pattern_code",
    "mcmed_most_common_pattern_code",
]


primary28_summary_columns = [
    column_name
    for column_name
    in primary28_summary_columns
    if column_name
    in primary28_ranked_df.columns
]


primary28_ranked_summary_df = (
    primary28_ranked_df[
        primary28_summary_columns
    ]
    .copy()
)



PRIMARY28_ANALYSIS_DIR = (
    CLINICIAN_MODEL_COMPARISON_DIR
    / "primary_28_replicated_clinician_unselected"
)


PRIMARY28_TABLE_DIR = (
    PRIMARY28_ANALYSIS_DIR
    / "tables"
)


PRIMARY28_INTERACTIVE_DIR = (
    PRIMARY28_ANALYSIS_DIR
    / "interactive_shap"
)


PRIMARY28_CATEGORY_INTERACTIVE_DIR = (
    PRIMARY28_INTERACTIVE_DIR
    / "categories"
)


PRIMARY28_PLOT_DATA_DIR = (
    PRIMARY28_ANALYSIS_DIR
    / "plot_data"
)


for output_directory in [
    PRIMARY28_ANALYSIS_DIR,
    PRIMARY28_TABLE_DIR,
    PRIMARY28_INTERACTIVE_DIR,
    PRIMARY28_CATEGORY_INTERACTIVE_DIR,
    PRIMARY28_PLOT_DATA_DIR,
]:

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )



primary28_ranked_summary_df.to_csv(
    PRIMARY28_TABLE_DIR
    / "primary_28_ranked_cross_dataset_summary.csv",
    index=False,
)


primary28_ranked_summary_df.to_parquet(
    PRIMARY28_TABLE_DIR
    / "primary_28_ranked_cross_dataset_summary.parquet",
    index=False,
    engine="pyarrow",
)


primary28_category_summary_df.to_csv(
    PRIMARY28_TABLE_DIR
    / "primary_28_category_summary.csv",
    index=False,
)



primary28_ranked_tables_by_category = {}


def primary28_safe_filename(
    value,
):

    return (
        str(
            value
        )
        .lower()
        .replace(
            " / ",
            "_"
        )
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )


for category_name in (
    primary28_category_order
):

    category_table = (
        primary28_ranked_summary_df.loc[
            primary28_ranked_summary_df[
                "model_category"
            ]
            ==
            category_name
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    primary28_ranked_tables_by_category[
        category_name
    ] = category_table


    safe_category = (
        primary28_safe_filename(
            category_name
        )
    )


    category_table.to_csv(
        PRIMARY28_TABLE_DIR
        /
        f"primary_28_{safe_category}_ranked.csv",
        index=False,
    )



print(
    "\n=================================================="
)


print(
    "28 FINAL REPLICATED CLINICIAN-UNSELECTED "
    "CLINICAL CANDIDATES"
)


print(
    "Ranked by conservative cross-dataset stability"
)


print(
    "=================================================="
)


primary28_display_columns = [
    "cross_dataset_priority_rank",
    "cross_dataset_stability_rank",
    "model_category",
    "feature_label",
    "feature",
    "mimic_importance_tag",
    "mimic_stability_pct",
    "mcmed_importance_tag",
    "mcmed_stability_pct",
    "cross_dataset_min_stability_pct",
    "cross_dataset_mean_stability_pct",
    "direction_agreement",
]


display(
    primary28_ranked_df[
        primary28_display_columns
    ]
    .round(
        3
    )
)


print(
    "\n=================================================="
)


print(
    "PRIMARY 28 — CATEGORY SUMMARY"
)


print(
    "=================================================="
)


display(
    primary28_category_summary_df
    .round(
        3
    )
)



for category_name in (
    primary28_category_order
):

    category_table = (
        primary28_ranked_tables_by_category[
            category_name
        ]
    )


    print(
        "\n",
        category_name,
        "—",
        len(
            category_table
        ),
        "features",
    )


    category_display_columns = [
        "cross_dataset_priority_rank",
        "category_priority_rank",
        "feature_label",
        "feature",
        "mimic_stability_pct",
        "mcmed_stability_pct",
        "cross_dataset_min_stability_pct",
        "mimic_importance_tag",
        "mcmed_importance_tag",
        "direction_agreement",
    ]


    category_display_columns = [
        column_name
        for column_name
        in category_display_columns
        if column_name
        in category_table.columns
    ]


    display(
        category_table[
            category_display_columns
        ]
        .round(
            3
        )
    )



for feature_name in (
    primary28_ranked_df[
        "feature"
    ]
):

    feature_position = (
        RAW_FEATURE_POSITION[
            feature_name
        ]
    )


    for dataset_name, feature_matrix in [
        (
            "MIMIC-IV",
            mimic_feature_values,
        ),
        (
            "MC-MED",
            mcmed_feature_values,
        ),
    ]:

        values = np.asarray(
            feature_matrix[
                :,
                feature_position
            ],
            dtype=float,
        )


        finite_values = values[
            np.isfinite(
                values
            )
        ]


        binary_mask = (
            np.isclose(
                finite_values,
                0.0,
                atol=1e-8,
                rtol=0.0,
            )
            |
            np.isclose(
                finite_values,
                1.0,
                atol=1e-8,
                rtol=0.0,
            )
        )


        assert binary_mask.all(), (
            "A final primary candidate is unexpectedly "
            "non-binary.\n"
            f"Dataset: {dataset_name}\n"
            f"Feature: {feature_name}\n"
            f"Unique values: "
            f"{np.unique(finite_values)[:20]}"
        )



primary28_plot_chunks = []


primary28_rng = np.random.default_rng(
    42
)


dataset_specs = [
    (
        "MIMIC-IV",
        np.asarray(
            mimic_shap_values
        ),
        np.asarray(
            mimic_feature_values
        ),
    ),

    (
        "MC-MED",
        np.asarray(
            mcmed_shap_values
        ),
        np.asarray(
            mcmed_feature_values
        ),
    ),
]


for (
    dataset_name,
    shap_matrix,
    feature_matrix,
) in dataset_specs:

    for feature_name in (
        primary28_ranked_df[
            "feature"
        ]
    ):

        feature_position = (
            RAW_FEATURE_POSITION[
                feature_name
            ]
        )


        shap_values_for_feature = np.asarray(
            shap_matrix[
                :,
                feature_position
            ],
            dtype=float,
        )


        feature_values_for_feature = np.asarray(
            feature_matrix[
                :,
                feature_position
            ],
            dtype=float,
        )


        valid_mask = (
            np.isfinite(
                shap_values_for_feature
            )
            &
            np.isfinite(
                feature_values_for_feature
            )
        )


        valid_row_indices = np.flatnonzero(
            valid_mask
        )


        shap_values_valid = (
            shap_values_for_feature[
                valid_mask
            ]
        )


        feature_values_valid = (
            feature_values_for_feature[
                valid_mask
            ]
        )


        presence_labels = np.where(
            feature_values_valid
            >
            0.5,
            "Present / high (1)",
            "Absent / low (0)",
        )


        y_jitter = (
            primary28_rng.uniform(
                low=-0.19,
                high=0.19,
                size=len(
                    shap_values_valid
                ),
            )
        )


        primary28_plot_chunks.append(
            pd.DataFrame(
                {
                    "dataset": (
                        dataset_name
                    ),

                    "row_index": (
                        valid_row_indices
                    ),

                    "feature": (
                        feature_name
                    ),

                    "feature_value": (
                        feature_values_valid
                    ),

                    "presence_label": (
                        presence_labels
                    ),

                    "shap_value": (
                        shap_values_valid
                    ),

                    "y_jitter": (
                        y_jitter
                    ),
                }
            )
        )


primary28_interactive_plot_data_df = (
    pd.concat(
        primary28_plot_chunks,
        ignore_index=True,
    )
)


EXPECTED_PRIMARY28_PLOT_ROWS = (
    PRIMARY_FINAL_N
    *
    (
        MIMIC_N_ROWS
        +
        MCMED_N_ROWS
    )
)


assert len(
    primary28_interactive_plot_data_df
) == EXPECTED_PRIMARY28_PLOT_ROWS



primary28_interactive_plot_data_df.to_parquet(
    PRIMARY28_PLOT_DATA_DIR
    / "primary_28_interactive_shap_plot_data.parquet",
    index=False,
    engine="pyarrow",
)



primary28_hover_metadata_columns = [
    "feature",
    "feature_label",
    "model_category",
    "cross_dataset_priority_rank",
    "cross_dataset_stability_rank",
    "mimic_stability_pct",
    "mcmed_stability_pct",
    "cross_dataset_min_stability_pct",
    "cross_dataset_mean_stability_pct",
    "mimic_importance_tag",
    "mcmed_importance_tag",
    "direction_agreement",
]


primary28_hover_metadata_df = (
    primary28_ranked_df[
        primary28_hover_metadata_columns
    ]
    .copy()
)


primary28_interactive_hover_df = (
    primary28_interactive_plot_data_df
    .merge(
        primary28_hover_metadata_df,
        on="feature",
        how="left",
        validate="many_to_one",
    )
)


assert primary28_interactive_hover_df[
    "feature_label"
].notna().all()



PRIMARY28_SHAP_BLUE = (
    "#008BFB"
)


PRIMARY28_SHAP_RED = (
    "#FF0051"
)


PRIMARY28_ZERO_LINE = (
    "#444444"
)


PRIMARY28_ROW_HEIGHT_PX = 32



def primary28_make_interactive_shap(
    feature_subset,
    figure_title,
):

    feature_subset = list(
        feature_subset
    )


    subset_rank_df = (
        primary28_ranked_df.loc[
            primary28_ranked_df[
                "feature"
            ]
            .isin(
                feature_subset
            )
        ]
        .sort_values(
            "cross_dataset_priority_rank"
        )
        .copy()
    )


    if subset_rank_df.empty:

        raise ValueError(
            "No features available for this interactive "
            "SHAP figure."
        )


    feature_order = (
        subset_rank_df[
            "feature"
        ]
        .tolist()
    )


    feature_to_y = {
        feature_name: feature_position
        for feature_position, feature_name
        in enumerate(
            feature_order
        )
    }


    tick_values = list(
        range(
            len(
                feature_order
            )
        )
    )


    tick_text = [
        (
            f"#{int(row.cross_dataset_priority_rank):02d}  "
            f"{row.feature_label}"
        )
        for row
        in subset_rank_df.itertuples()
    ]


    figure_data = (
        primary28_interactive_hover_df.loc[
            primary28_interactive_hover_df[
                "feature"
            ]
            .isin(
                feature_order
            )
        ]
        .copy()
    )


    figure_data[
        "y_position"
    ] = (
        figure_data[
            "feature"
        ]
        .map(
            feature_to_y
        )
        .astype(float)
        +
        figure_data[
            "y_jitter"
        ]
    )


    max_abs_shap = float(
        np.nanmax(
            np.abs(
                figure_data[
                    "shap_value"
                ]
                .to_numpy(
                    dtype=float
                )
            )
        )
    )


    x_axis_limit = max(
        0.05,
        max_abs_shap
        *
        1.05,
    )


    figure = go.Figure()


    for dataset_name in [
        "MIMIC-IV",
        "MC-MED",
    ]:

        for (
            presence_label,
            marker_colour,
        ) in [
            (
                "Absent / low (0)",
                PRIMARY28_SHAP_BLUE,
            ),

            (
                "Present / high (1)",
                PRIMARY28_SHAP_RED,
            ),
        ]:

            trace_df = (
                figure_data.loc[
                    (
                        figure_data[
                            "dataset"
                        ]
                        ==
                        dataset_name
                    )
                    &
                    (
                        figure_data[
                            "presence_label"
                        ]
                        ==
                        presence_label
                    )
                ]
            )


            custom_data = np.column_stack(
                [
                    trace_df[
                        "feature_label"
                    ].astype(str),

                    trace_df[
                        "feature"
                    ].astype(str),

                    trace_df[
                        "model_category"
                    ].astype(str),

                    trace_df[
                        "feature_value"
                    ].to_numpy(),

                    trace_df[
                        "cross_dataset_priority_rank"
                    ].to_numpy(),

                    trace_df[
                        "mimic_stability_pct"
                    ].to_numpy(),

                    trace_df[
                        "mcmed_stability_pct"
                    ].to_numpy(),

                    trace_df[
                        "cross_dataset_min_stability_pct"
                    ].to_numpy(),

                    trace_df[
                        "cross_dataset_mean_stability_pct"
                    ].to_numpy(),

                    trace_df[
                        "mimic_importance_tag"
                    ].astype(str),

                    trace_df[
                        "mcmed_importance_tag"
                    ].astype(str),

                    trace_df[
                        "direction_agreement"
                    ].astype(str),

                    np.repeat(
                        dataset_name,
                        len(
                            trace_df
                        ),
                    ),
                ]
            )


            figure.add_trace(
                go.Scattergl(
                    x=(
                        trace_df[
                            "shap_value"
                        ]
                    ),

                    y=(
                        trace_df[
                            "y_position"
                        ]
                    ),

                    mode="markers",

                    name=(
                        presence_label
                    ),

                    visible=(
                        dataset_name
                        ==
                        "MIMIC-IV"
                    ),

                    marker=dict(
                        color=(
                            marker_colour
                        ),

                        size=5,

                        opacity=0.55,
                    ),

                    customdata=(
                        custom_data
                    ),

                    hovertemplate=(
                        "<b>%{customdata[0]}</b>"
                        "<br>"
                        "Dataset: %{customdata[12]}"
                        "<br>"
                        "Category: %{customdata[2]}"
                        "<br>"
                        "Technical feature: %{customdata[1]}"
                        "<br>"
                        "Feature value: %{customdata[3]}"
                        "<br>"
                        "<b>SHAP: %{x:.5f}</b>"
                        "<br>"
                        "Cross-dataset priority rank: "
                        "%{customdata[4]}"
                        "<br>"
                        "MIMIC stability: "
                        "%{customdata[5]:.3f}%"
                        "<br>"
                        "MC-MED stability: "
                        "%{customdata[6]:.3f}%"
                        "<br>"
                        "Conservative min stability: "
                        "%{customdata[7]:.3f}%"
                        "<br>"
                        "Mean stability: "
                        "%{customdata[8]:.3f}%"
                        "<br>"
                        "MIMIC tier: %{customdata[9]}"
                        "<br>"
                        "MC-MED tier: %{customdata[10]}"
                        "<br>"
                        "Direction: %{customdata[11]}"
                        "<extra></extra>"
                    ),
                )
            )


    dataset_buttons = [
        dict(
            label="MIMIC-IV",
            method="update",
            args=[
                {
                    "visible": [
                        True,
                        True,
                        False,
                        False,
                    ]
                },

                {
                    "title": {
                        "text": (
                            f"MIMIC-IV — {figure_title}"
                        )
                    }
                },
            ],
        ),

        dict(
            label="MC-MED",
            method="update",
            args=[
                {
                    "visible": [
                        False,
                        False,
                        True,
                        True,
                    ]
                },

                {
                    "title": {
                        "text": (
                            f"MC-MED — {figure_title}"
                        )
                    }
                },
            ],
        ),
    ]


    figure.update_layout(
        title=dict(
            text=(
                f"MIMIC-IV — {figure_title}"
            ),

            x=0.01,
        ),

        width=1450,

        height=max(
            650,
            (
                PRIMARY28_ROW_HEIGHT_PX
                *
                len(
                    feature_order
                )
            )
            +
            220,
        ),

        margin=dict(
            l=420,
            r=80,
            t=150,
            b=90,
        ),

        hovermode="closest",

        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1.0,
        ),

        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                buttons=(
                    dataset_buttons
                ),
                x=0.0,
                y=1.10,
                xanchor="left",
                yanchor="bottom",
                showactive=True,
            )
        ],

        xaxis=dict(
            title=(
                "SHAP value on raw XGBoost margin "
                "(negative → HOME, positive → ADMITTED)"
            ),

            range=[
                -x_axis_limit,
                x_axis_limit,
            ],

            zeroline=False,

            showgrid=True,

            gridcolor=(
                "rgba(128,128,128,0.18)"
            ),
        ),

        yaxis=dict(
            title="",

            tickmode="array",

            tickvals=(
                tick_values
            ),

            ticktext=(
                tick_text
            ),

            range=[
                len(
                    feature_order
                )
                -
                0.5,
                -0.5,
            ],

            showgrid=True,

            gridcolor=(
                "rgba(128,128,128,0.12)"
            ),
        ),
    )


    figure.add_vline(
        x=0.0,
        line_width=1.2,
        line_dash="solid",
        line_color=(
            PRIMARY28_ZERO_LINE
        ),
    )


    figure.add_annotation(
        x=0.5,
        y=-0.11,
        xref="paper",
        yref="paper",
        showarrow=False,
        text=(
            "BLUE = feature absent / low (0)   |   "
            "RED = feature present / high (1)"
        ),
        font=dict(
            size=12
        ),
    )


    return figure



primary28_plotly_config = {
    "displaylogo": False,
    "responsive": True,
    "scrollZoom": True,
}



primary28_all_features_figure = (
    primary28_make_interactive_shap(
        feature_subset=(
            primary28_ranked_df[
                "feature"
            ]
            .tolist()
        ),

        figure_title=(
            "28 final replicated clinician-unselected "
            "clinical candidates"
        ),
    )
)


PRIMARY28_ALL_INTERACTIVE_HTML = (
    PRIMARY28_INTERACTIVE_DIR
    /
    "01_primary_28_all_categories_interactive_shap.html"
)


primary28_all_features_figure.write_html(
    PRIMARY28_ALL_INTERACTIVE_HTML,
    include_plotlyjs="directory",
    full_html=True,
    config=(
        primary28_plotly_config
    ),
)


print(
    "\n=================================================="
)


print(
    "INTERACTIVE SHAP — ALL 28 FINAL CANDIDATES"
)


print(
    "Use buttons above figure to switch "
    "MIMIC-IV / MC-MED"
)


print(
    "=================================================="
)


primary28_all_features_figure.show(
    config=(
        primary28_plotly_config
    )
)



primary28_category_figures = {}


primary28_interactive_manifest_records = [
    {
        "figure_scope": (
            "All 28 final candidates"
        ),

        "n_features": (
            PRIMARY_FINAL_N
        ),

        "contains_mimic": (
            True
        ),

        "contains_mcmed": (
            True
        ),

        "html_path": str(
            PRIMARY28_ALL_INTERACTIVE_HTML
        ),
    }
]


for category_index, category_name in enumerate(
    primary28_category_order,
    start=2,
):

    category_feature_df = (
        primary28_ranked_df.loc[
            primary28_ranked_df[
                "model_category"
            ]
            ==
            category_name
        ]
        .sort_values(
            "cross_dataset_priority_rank"
        )
    )


    category_features = (
        category_feature_df[
            "feature"
        ]
        .tolist()
    )


    if not category_features:

        continue


    category_figure = (
        primary28_make_interactive_shap(
            feature_subset=(
                category_features
            ),

            figure_title=(
                f"{category_name} — final replicated "
                "clinician-unselected clinical candidates"
            ),
        )
    )


    primary28_category_figures[
        category_name
    ] = category_figure


    safe_category = (
        primary28_safe_filename(
            category_name
        )
    )


    category_html_path = (
        PRIMARY28_CATEGORY_INTERACTIVE_DIR
        /
        (
            f"{category_index:02d}_"
            f"{safe_category}_"
            "interactive_shap.html"
        )
    )


    category_figure.write_html(
        category_html_path,
        include_plotlyjs="directory",
        full_html=True,
        config=(
            primary28_plotly_config
        ),
    )


    primary28_interactive_manifest_records.append(
        {
            "figure_scope": (
                category_name
            ),

            "n_features": int(
                len(
                    category_features
                )
            ),

            "contains_mimic": (
                True
            ),

            "contains_mcmed": (
                True
            ),

            "html_path": str(
                category_html_path
            ),
        }
    )


    print(
        "\n=================================================="
    )


    print(
        "INTERACTIVE SHAP CATEGORY:",
        category_name,
    )


    print(
        "Features:",
        len(
            category_features
        ),
    )


    print(
        "=================================================="
    )


    category_figure.show(
        config=(
            primary28_plotly_config
        )
    )



primary28_interactive_manifest_df = (
    pd.DataFrame(
        primary28_interactive_manifest_records
    )
)


primary28_interactive_manifest_df.to_csv(
    PRIMARY28_ANALYSIS_DIR
    / "primary_28_interactive_shap_manifest.csv",
    index=False,
)



primary28_plot_data_audit_df = (
    primary28_interactive_plot_data_df
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        n_plot_rows=(
            "shap_value",
            "size",
        ),

        n_features=(
            "feature",
            "nunique",
        ),

        n_source_rows=(
            "row_index",
            "nunique",
        ),
    )
)


primary28_plot_data_audit_df.to_csv(
    PRIMARY28_ANALYSIS_DIR
    / "primary_28_interactive_shap_plot_data_audit.csv",
    index=False,
)



primary28_ranking_methodology_audit_df = pd.DataFrame(
    [
        {
            "n_primary_features": (
                PRIMARY_FINAL_N
            ),

            "clinician_vote_definition": (
                "0/10 clinicians"
            ),

            "model_replication_definition": (
                "selected in BOTH MIMIC-IV and MC-MED"
            ),

            "primary_ranking_variable": (
                "minimum of MIMIC-IV and MC-MED "
                "stability percentage"
            ),

            "first_tie_break": (
                "mean MIMIC-IV/MC-MED stability percentage"
            ),

            "second_tie_break": (
                "mean absolute SHAP across datasets"
            ),

            "ranking_changes_feature_selection": (
                False
            ),

            "imaging_in_final_candidate_set": (
                False
            ),

            "imaging_shap_generated": (
                False
            ),

            "binary_feature_colour_blue": (
                "feature absent / low (0)"
            ),

            "binary_feature_colour_red": (
                "feature present / high (1)"
            ),

            "shap_positive_direction": (
                "toward ADMITTED"
            ),

            "shap_negative_direction": (
                "toward HOME"
            ),

            "mimic_rows_used": (
                MIMIC_N_ROWS
            ),

            "mcmed_rows_used": (
                MCMED_N_ROWS
            ),

            "shap_recalculated": (
                False
            ),

            "sampling_used": (
                False
            ),
        }
    ]
)


primary28_ranking_methodology_audit_df.to_csv(
    PRIMARY28_ANALYSIS_DIR
    / "primary_28_ranking_methodology_audit.csv",
    index=False,
)



assert len(
    primary28_ranked_df
) == PRIMARY_FINAL_N


assert (
    primary28_ranked_df[
        "cross_dataset_priority_rank"
    ]
    .tolist()
    ==
    list(
        range(
            1,
            PRIMARY_FINAL_N
            +
            1,
        )
    )
)


assert (
    primary28_category_summary_df[
        "n_primary_features"
    ].sum()
    ==
    PRIMARY_FINAL_N
)


assert not (
    primary28_ranked_df[
        "model_category"
    ]
    ==
    "Imaging / radiology"
).any()


assert (
    primary28_plot_data_audit_df.loc[
        primary28_plot_data_audit_df[
            "dataset"
        ]
        ==
        "MIMIC-IV",
        "n_source_rows",
    ].iloc[
        0
    ]
    ==
    MIMIC_N_ROWS
)


assert (
    primary28_plot_data_audit_df.loc[
        primary28_plot_data_audit_df[
            "dataset"
        ]
        ==
        "MC-MED",
        "n_source_rows",
    ].iloc[
        0
    ]
    ==
    MCMED_N_ROWS
)


assert (
    primary28_plot_data_audit_df[
        "n_features"
    ]
    ==
    PRIMARY_FINAL_N
).all()



n_direction_agreement = int(
    (
        primary28_ranked_df[
            "direction_agreement"
        ]
        ==
        "Agreement"
    ).sum()
)


print(
    "\n=================================================="
)


print(
    "PRIMARY 28 DETAILED INTERPRETATION COMPLETE"
)


print(
    "=================================================="
)


print(
    "\nFinal replicated clinician-unselected "
    "clinical candidates:",
    len(
        primary28_ranked_df
    ),
)


print(
    "Direction agreement across datasets:",
    f"{n_direction_agreement}/{PRIMARY_FINAL_N}",
)


print(
    "\nCategories represented:"
)


for category_name in (
    primary28_category_order
):

    category_count = int(
        (
            primary28_ranked_df[
                "model_category"
            ]
            ==
            category_name
        ).sum()
    )


    print(
        f"  {category_name}: "
        f"{category_count}"
    )


print(
    "\nImaging category present:",
    "NO",
)


print(
    "Imaging SHAP generated:",
    "NO",
)


print(
    "\nStrongest conservative stability:"
)


print(
    round(
        float(
            primary28_ranked_df[
                "cross_dataset_min_stability_pct"
            ].max()
        ),
        3,
    ),
    "%",
)


print(
    "Lowest conservative stability:"
)


print(
    round(
        float(
            primary28_ranked_df[
                "cross_dataset_min_stability_pct"
            ].min()
        ),
        3,
    ),
    "%",
)


print(
    "\nRanking rule:"
)


print(
    "1. Cross-dataset MIN stability"
)


print(
    "2. Cross-dataset MEAN stability"
)


print(
    "3. Cross-dataset mean |SHAP|"
)


print(
    "Ranking changes selection:",
    "NO",
)


print(
    "\nInteractive SHAP colour:"
)


print(
    "BLUE = absent / low (0)"
)


print(
    "RED  = present / high (1)"
)


print(
    "\nSHAP direction:"
)


print(
    "Negative = pushes HOME"
)


print(
    "Positive = pushes ADMITTED"
)


print(
    "\nAll-patient interactive SHAP:",
    "YES",
)


print(
    "Sampling:",
    "NO",
)


print(
    "SHAP recalculation:",
    "NO",
)


print(
    "\nRanked summary saved to:"
)


print(
    PRIMARY28_TABLE_DIR
    / "primary_28_ranked_cross_dataset_summary.csv"
)


print(
    "\nOverall interactive SHAP saved to:"
)


print(
    PRIMARY28_ALL_INTERACTIVE_HTML
)


print(
    "\nCategory interactive SHAP directory:"
)


print(
    PRIMARY28_CATEGORY_INTERACTIVE_DIR
)


print(
    "\nREADY FOR LITERATURE-BASED NOVELTY REVIEW:",
    "YES",
)